# DAMPS-MMHCL Training for Amazon Clothing Dataset (Local)

This notebook trains the **DAMPS-MMHCL** (Spectral Domain Representation Calibration upgrade for the Multi-Modal Hypergraph Contrastive Learning framework) on the Amazon Clothing dataset (**HF MMHCL LATTICE 39k**: **39,387 users / 23,033 items** from [Xu-SII-BNU/MMHCL](https://huggingface.co/datasets/Xu-SII-BNU/MMHCL/tree/main/Clothing)) on a local machine with NVIDIA RTX 5090 (32 GB VRAM).

It is a drop-in replacement for the original MMHCL notebook and follows the *Revision 11 â€” Phase 1 (Quick Win) Execution Plan with Quantitative Stop-Gate* architecture spec (see `DAMPS_to_MMHCL_architecture_revision44.tex` / `.pdf`), which builds on the *Revision 9 â€” 100% Compliance Check & Final Lock* baseline (`DAMPS_to_MMHCL_architecture_revision42.tex` / `.pdf`).

**Phase 1 â€” Quick Win:** the notebook sweeps **four** rev44 configurations Ã— **5 seeds** (**20** training runs) for the *Ï„-saturation / embedding-collapse* story (rev44 Â§3). **`Phase1_Day1_Final_Verdict.tex`** records the Day-1 sprint: the **APC-off** single-seed probe hit **R@20 â‰ˆ 0.09197** (MMHCL paper **0.0881**), while **`combined`** with APC on the **hash-fallback** (missing `meta_categories.npy`) sat near **~0.0851** â€” about **âˆ’0.0069** absolute recall vs APC-off. **H3 (top-K)** is rejected there; **H10** is deprioritised.

| Variant | Flags | Hypothesis |
| ------- | ----- | ---------- |
| (a) anchor   | `--temperature 0.1 --learnable_tau 1 --damps_avrf 1` | rev42 baseline; Ï„ saturates at ~0.0909 |
| (b) tau03    | `--temperature 0.3 --learnable_tau 0 --damps_avrf 1` | static Ï„ removes saturation |
| (c) avrf_off | `--temperature 0.1 --learnable_tau 1 --damps_avrf 0` | AVRF off recovers sparse coverage |
| (d) combined | `--temperature 0.3 --learnable_tau 0 --damps_avrf 0` | **RECOMMENDED** combined fix (valid APC needs real `meta_categories.npy`) |
| **(e) apc_off_combined** (Round-2) | same as (d) **+** `--damps_apc 0` | **5-seed validation** after Day-1 â€” code cell **immediately after** the four-variant sweep; feeds Â§4 / Â§4.5 |

**Stop-gate (rev44 Â§5.2):** mean **test** `Recall@20 â‰¥ 0.0870` **and** `NDCG@20 â‰¥ 0.0390` over the **5 seeds** (paired *t*-test vs anchor where applicable); **`Recall@20 â‰¥ 0.0900`** is the stricter â€œpaper validatedâ€ bar vs **0.0881**. **If (e) still fails:** run **`build_meta_categories.py`**, place **`meta_categories.npy`**, keep **`--damps_apc 1`** on **`combined`**, and **re-run** Phase 1. (Raise `n_runs` for a 10-seed protocol.)

## DAMPS pipeline (per forward pass)
1. **Spectral decomposition** â€” 1-D rFFT on the projected modality features (d = 64).
2. **Metadata-Aware APC** â€” von-Mises MLE phase calibration grouped by static metadata categories (no k-means).
3. **AVRF** â€” logit-clipped Wiener gate with strict `[-2, 2]` initialisation prior (data-driven from raw modality SNR) and per-epoch EMA MAD aggregation.
4. **Residual IMCF** â€” ASC consensus coefficient applied in residual form to avoid multiplicative attenuation.
5. **Inverse FFT** â€” back to spatial domain â†’ fed into HGCN via **Soft Residual-Routing** (eq. 3 of the spec).

Auxiliary upgrades wired into the trainer:
- **Pattern B' (Scheduled Rebuild)** â€” reconstruct the K-NN multi-modal hypergraph every `R` epochs from the **Slim Momentum** buffers (98 % VRAM saving vs MoCo).
- **Dual-path K-NN** â€” chunked PyTorch (default) with FAISS HNSW fallback when N â‰¥ 60 000.
- **`bfloat16` mixed precision** (-30 / -40 % wall-clock, -40 % VRAM on Ada Lovelace).
- **Static InfoNCE temperature Ï„ = 0.3** (rev44 Phase 1 default; toggle back to learnable nn.Parameter via `--learnable_tau 1`).
- **cuFFT plan-cache lockdown** (`cufft_plan_cache.max_size = 1`, Section 3.3).
- **Diagnostic logs** â€” NNZ / avg_deg per rebuild, tanh-saturation rate of the AVRF gates.

## Configuration
- **Dataset**: HF MMHCL Clothing LATTICE 39k (`data/Clothing/5-core`, 39387 users / 23033 items); features at `data/Clothing/{image,text}_feat.npy`
- **GPU**: NVIDIA RTX 5090 (32 GB VRAM)
- **Working dir**: `MMHCL_DAMPS_Project/`
- **Entry point**: `train.py` (replaces the original `codes/main.py`)
- **Batch Size**: 1024
- **Max Epochs**: 250
- **Rebuild cadence**: every 5 epochs after a 10-epoch warm-up
- **W&B Project**: `damps-mmhcl-clothing`

## Steps
1. Verify environment and GPU
2. Install dependencies for the DAMPS-MMHCL package
3. Set up W&B logging
4. (Optional) GPU memory monitor
5. Run a quick smoke test of the DAMPS package
6. **§2.7 Dataset integrity** — assert HF LATTICE 39k (39387 users / 23033 items) + purge stale `5-core/*.pth` caches
7. **rev44 Phase 1 Item #1** — 7-point eval-protocol audit (must PASS before training)
8. **Root-cause roadmap Â§5.0** â€” Day 1 diagnostic sprint (`MMHCL_DAMPS_Project/scripts/day1_diagnostic_sprint.py`; notebook Â§3.0.5)
9. Multi-seed training â€” **4 variants Ã— 5 seeds** (20 runs)
10. **Round-2 (Day-1 verdict)** â€” **`apc_off_combined`** on the **same 5 seeds** (code cell directly under Â§3 training; see `Phase1_Day1_Final_Verdict.tex`). **PASS Phase 1** if mean **test R@20 â‰¥ 0.0870** (and NDCG per Â§5.2).
11. **Â§4** â€” aggregate results (writes `phase1_*_summary.json`, including `phase1_apc_off_combined_summary.json` after step 9)
12. **Â§4.5** â€” Bonferroni paired *t*-tests + stop-gate (**N = 4** non-anchor tests once variant **(e)** exists)
13. If still **FAIL** â€” **`build_meta_categories.py`** â†’ real **`meta_categories.npy`** â†’ rerun Phase 1 **`combined`** with **`--damps_apc 1`**
14. Export results to Excel


## 1. Environment Setup & GPU Verification

In [1]:
from __future__ import annotations

import os
import sys
import torch

# ---------------------------------------------------------------------------
#  Resolve project root.  The notebook may be opened from either the
#  workspace root or one of the package sub-directories.
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(('codes', 'MMHCL_DAMPS_Project')):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

# DAMPS-MMHCL implementation lives under MMHCL_DAMPS_Project/
DAMPS_DIR: str = os.path.join(PROJECT_ROOT, 'MMHCL_DAMPS_Project')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')
# Legacy CODES_DIR is still around for the original MMHCL baseline ablation
CODES_DIR: str = os.path.join(PROJECT_ROOT, 'codes')

print(f"Project Root        : {PROJECT_ROOT}")
print(f"DAMPS-MMHCL Directory: {DAMPS_DIR}")
print(f"Data Directory       : {DATA_DIR}")

assert os.path.exists(DAMPS_DIR), (
    f"MMHCL_DAMPS_Project not found at {DAMPS_DIR}. "
    f"Make sure you cloned the full DAMPS-MMHCL repository."
)
assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
print("\n[OK] All directories verified")

# Verify GPU & bfloat16 capability (DAMPS uses bf16 AMP by default)
print("\n" + "=" * 60)
print("GPU Information")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    total_gb: float = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU Memory  : {total_gb:.1f} GB")

    # bfloat16 is a hard requirement for the DAMPS bf16 AMP path. Detect
    # support so the user gets a clear error early instead of a cryptic
    # autocast failure mid-training.
    bf16_ok: bool = torch.cuda.is_bf16_supported()
    print(f"bfloat16 support: {bf16_ok}")
    if not bf16_ok:
        print(
            "[WARNING] bfloat16 not supported on this GPU. Pass --use_amp 0 "
            "to fall back to fp32, or run on Ampere/Ada Lovelace+."
        )
else:
    print("[WARNING] No GPU detected. DAMPS training will be very slow on CPU.")

Project Root        : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing
DAMPS-MMHCL Directory: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
Data Directory       : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data

[OK] All directories verified

GPU Information
CUDA available: True
GPU         : NVIDIA GeForce RTX 5090
CUDA version: 12.8
GPU Memory  : 31.8 GB
bfloat16 support: True


## 1.5 Install Dependencies (Run Once)

In [2]:
# Install required dependencies for the DAMPS-MMHCL package (run once).
#
# The DAMPS-MMHCL implementation pins its requirements in
# `MMHCL_DAMPS_Project/requirements.txt`. We install from there first,
# then add notebook-only extras (wandb, pandas, openpyxl).
from __future__ import annotations

import subprocess
import sys
import os

print("=" * 60)
print("Installing DAMPS-MMHCL dependencies")
print("=" * 60)

# UV-managed Python sometimes refuses installs without --break-system-packages
pip_extra_args: list[str] = []
test_result: subprocess.CompletedProcess[str] = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--dry-run', 'pip'],
    capture_output=True, text=True,
)
if 'externally-managed' in test_result.stderr:
    pip_extra_args = ['--break-system-packages']
    print("\n[INFO] UV-managed Python detected, using --break-system-packages")

# Step 1 - install DAMPS-MMHCL package requirements
requirements_path: str = os.path.join(DAMPS_DIR, 'requirements.txt')
if not os.path.exists(requirements_path):
    requirements_path = os.path.join(PROJECT_ROOT, 'requirements.txt')

if os.path.exists(requirements_path):
    print(f"\n1. Installing packages from: {requirements_path}")
    print("   This may take several minutes for PyTorch (~2.6 GB)...")
    cmd: list[str] = [sys.executable, '-m', 'pip', 'install', '-r', requirements_path] + pip_extra_args
    result: subprocess.CompletedProcess[str] = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"   [ERROR] Installation failed! Exit code: {result.returncode}")
        print("\n--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise Exception("Failed to install requirements")
    print("   [OK] DAMPS-MMHCL package requirements installed")
else:
    print(f"   [WARNING] No requirements.txt found at {requirements_path}")

# Step 2 - install notebook-only extras
# nbformat is required by wandb to save notebook history inside Jupyter/Colab.
additional_packages: list[str] = ['wandb', 'openpyxl', 'pandas', 'nbformat']
print(f"\n2. Installing notebook extras: {additional_packages}")
for package in additional_packages:
    print(f"   Installing {package}...")
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', package] + pip_extra_args
    subprocess.run(cmd, capture_output=True)
print("   [OK] Notebook extras installed")

# Step 3 - sanity-check imports of all DAMPS-MMHCL modules
print("\n3. Verifying DAMPS-MMHCL package imports...")
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)
try:
    import damps  # noqa: F401
    from damps import (  # noqa: F401
        DAMPS,
        SlimMomentumEncoder,
        DualPathKNN,
        compute_avrf_prior,
        compute_avrf_logit,
    )
    print("   [OK] `damps` core package imports cleanly")
except Exception as exc:  # pragma: no cover
    print(f"   [WARNING] Could not import `damps` package yet: {exc}")
    print("   This is expected on the very first run; restart the kernel and try again.")

print("\n" + "=" * 60)
print("[OK] All packages installed successfully!")
print("=" * 60)
print("\n[IMPORTANT] If this is the first install, RESTART THE KERNEL before running the next cells.")

Installing DAMPS-MMHCL dependencies

1. Installing packages from: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\requirements.txt
   This may take several minutes for PyTorch (~2.6 GB)...
   [OK] DAMPS-MMHCL package requirements installed

2. Installing notebook extras: ['wandb', 'openpyxl', 'pandas', 'nbformat']
   Installing wandb...
   Installing openpyxl...
   Installing pandas...
   Installing nbformat...
   [OK] Notebook extras installed

3. Verifying DAMPS-MMHCL package imports...
   [OK] `damps` core package imports cleanly

[OK] All packages installed successfully!

[IMPORTANT] If this is the first install, RESTART THE KERNEL before running the next cells.


## 2. Weights & Biases Setup

In [3]:
from __future__ import annotations

import wandb
from IPython.display import display, Markdown

# ========== Login to Weights & Biases ==========
# A dedicated project so DAMPS runs do not get mixed in with the original
# MMHCL baseline runs. Change `wandb_entity` if you push to your own account.
wandb_project: str = 'damps-mmhcl-clothing'
wandb_entity: str = 'baitapck51cc-uet'

# Your W&B API key (replace with your own if you fork this notebook)
WANDB_API_KEY: str = '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'

wandb.login(key=WANDB_API_KEY)

print("[OK] W&B login successful")
print(f"     Entity : {wandb_entity}")
print(f"     Project: {wandb_project}")

wandb_url: str = f"https://wandb.ai/{wandb_entity}/{wandb_project}"
print()
display(Markdown(f"## [Open Weights & Biases Dashboard]({wandb_url})"))
print(f"Direct Link: {wandb_url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Anh Khoi\_netrc
wandb: Currently logged in as: baitapck51cc (baitapck51cc-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[OK] W&B login successful
     Entity : baitapck51cc-uet
     Project: damps-mmhcl-clothing



## [Open Weights & Biases Dashboard](https://wandb.ai/baitapck51cc-uet/damps-mmhcl-clothing)

Direct Link: https://wandb.ai/baitapck51cc-uet/damps-mmhcl-clothing


## 2.5 GPU Memory Monitor (Optional)

Start a lightweight background monitor to print GPU VRAM usage while training.
Run the stop function after training completes.

In [4]:
from __future__ import annotations

import threading
import time
import torch
import csv
import os
from typing import Optional

# GPU monitor controls
GPU_MONITOR_RUNNING: bool = False
GPU_MONITOR_THREAD: Optional[threading.Thread] = None


def _resolve_monitor_root() -> str:
    current_dir: str = os.getcwd()
    if current_dir.endswith('codes'):
        return os.path.dirname(current_dir)
    return current_dir


def _gpu_monitor(interval_seconds: int, csv_path: str, print_live: bool) -> None:
    global GPU_MONITOR_RUNNING
    if not torch.cuda.is_available():
        print("[WARNING] CUDA not available. GPU monitor will not run.")
        GPU_MONITOR_RUNNING = False
        return

    total_mem: int = torch.cuda.get_device_properties(0).total_memory
    start_time: float = time.time()

    file_exists: bool = os.path.exists(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as csv_file:
        writer: csv.writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow([
                'relative_time_s',
                'allocated_gb',
                'reserved_gb',
                'allocated_pct',
                'reserved_pct'
            ])

        print(f"[INFO] GPU monitor started. Logging to: {csv_path}")
        while GPU_MONITOR_RUNNING:
            allocated: int = torch.cuda.memory_allocated(0)
            reserved: int = torch.cuda.memory_reserved(0)
            allocated_pct: float = (allocated / total_mem) * 100
            reserved_pct: float = (reserved / total_mem) * 100
            rel_time: float = time.time() - start_time

            writer.writerow([
                f"{rel_time:.2f}",
                f"{allocated / 1024**3:.4f}",
                f"{reserved / 1024**3:.4f}",
                f"{allocated_pct:.2f}",
                f"{reserved_pct:.2f}",
            ])
            csv_file.flush()

            if print_live:
                print(
                    f"[GPU] allocated={allocated/1024**3:.2f} GB ({allocated_pct:.1f}%), "
                    f"reserved={reserved/1024**3:.2f} GB ({reserved_pct:.1f}%)"
                )

            time.sleep(interval_seconds)

    print("[INFO] GPU monitor stopped.")


def start_gpu_monitor(interval_seconds: int = 10, csv_filename: str = 'gpu_memory_monitor.csv', print_live: bool = False) -> None:
    """Start background GPU monitor. Logs to CSV every N seconds."""
    global GPU_MONITOR_RUNNING, GPU_MONITOR_THREAD
    if GPU_MONITOR_RUNNING:
        print("[INFO] GPU monitor already running.")
        return

    monitor_root: str = _resolve_monitor_root()
    csv_path: str = os.path.join(monitor_root, csv_filename)

    GPU_MONITOR_RUNNING = True
    GPU_MONITOR_THREAD = threading.Thread(
        target=_gpu_monitor,
        args=(interval_seconds, csv_path, print_live),
        daemon=True
    )
    GPU_MONITOR_THREAD.start()


def stop_gpu_monitor() -> None:
    """Stop background GPU monitor."""
    global GPU_MONITOR_RUNNING
    GPU_MONITOR_RUNNING = False


# Start monitor with 15s interval (adjust as needed)
# Set print_live=True if you want console output too
start_gpu_monitor(interval_seconds=15, csv_filename='gpu_memory_monitor.csv', print_live=False)

# Call stop_gpu_monitor() after training completes.

[INFO] GPU monitor started. Logging to: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\gpu_memory_monitor.csv


## 2.6 DAMPS Smoke Test (Optional)

Run a small end-to-end sanity check of the DAMPS-MMHCL package before launching the full multi-seed training. This invokes `MMHCL_DAMPS_Project/tests/smoke_test.py`, which exercises:

- AVRF prior derivation + logit clipping
- DAMPS core forward pass (FFT â†’ APC â†’ AVRF â†’ Residual IMCF â†’ iFFT)
- Slim Momentum Encoder update step
- Dual-Path KNN graph construction
- Full `DAMPS_MMHCL` model forward + backward

Skip this cell if you have already validated the package.

In [5]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

# ---------------------------------------------------------------------------
# Make this cell self-sufficient: section 2.6 is documented as "Optional",
# so users often run it without first executing the Section 2.1 env-setup
# cell that defines DAMPS_DIR. If DAMPS_DIR is not yet in the namespace,
# auto-discover it from the notebook's working directory using three
# resolution strategies, then publish it back so the rest of the cell (and
# any downstream cells executed in this same kernel) can rely on it.
# ---------------------------------------------------------------------------
try:
    _damps_dir: str = DAMPS_DIR  # populated by the env-setup cell
except NameError:
    _cwd = Path.cwd().resolve()
    if _cwd.name == 'MMHCL_DAMPS_Project':
        _damps_dir = str(_cwd)
    elif (_cwd / 'MMHCL_DAMPS_Project').is_dir():
        _damps_dir = str((_cwd / 'MMHCL_DAMPS_Project').resolve())
    else:
        _ancestor = next(
            (p for p in [_cwd, *_cwd.parents]
             if (p / 'MMHCL_DAMPS_Project').is_dir()),
            None,
        )
        if _ancestor is None:
            raise RuntimeError(
                "Could not locate the MMHCL_DAMPS_Project directory.\n"
                "Either run the environment-setup cell (Section 2.1) first, "
                "or open this notebook from the repository root."
            )
        _damps_dir = str((_ancestor / 'MMHCL_DAMPS_Project').resolve())
    DAMPS_DIR = _damps_dir
    print(f"[smoke-test] DAMPS_DIR auto-resolved to: {_damps_dir}")

smoke_path: str = os.path.join(DAMPS_DIR, 'tests', 'smoke_test.py')

if not os.path.exists(smoke_path):
    print(f"[WARN] Smoke test not found at {smoke_path} - skipping.")
else:
    print("=" * 70)
    print("Running DAMPS-MMHCL smoke test...")
    print(f"  Script: {smoke_path}")
    print("=" * 70)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    env['PYTHONUNBUFFERED'] = '1'

    result = subprocess.run(
        [sys.executable, smoke_path],
        cwd=DAMPS_DIR,
        capture_output=True,
        text=True,
        env=env,
        encoding='utf-8',
        errors='replace',
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("\n[FAIL] Smoke test exited with code", result.returncode)
        if result.stderr:
            print("\n--- STDERR ---")
            print(result.stderr)
    else:
        print("\n[OK] DAMPS-MMHCL smoke test passed.")

Running DAMPS-MMHCL smoke test...
  Script: c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\tests\smoke_test.py
== DAMPS core ==
  [OK] forward + backward, 133 params
  [OK] tanh_sat: {'img': 0.0, 'txt': 0.0, 'aud': 0.0}
  [OK] update_epoch_mad runs
  [OK] IMCF schedule: forward-pass counter +5, epoch held at 3 then advanced to 7 via set_epoch
== Slim Momentum Encoder ==
  [OK] EMA buffers update + initialised flag
== Dual-path KNN ==
  [OK] single-modality K-NN, NNZ=256
  [OK] multi-modal hypergraph, NNZ=1594, avg_deg=24.91
== Data-driven prior ==
  [OK] prior range=[0.401,0.693]  logit clipped
== DAMPS_MMHCL full model ==
  [OK] static tau initialised at 0.3000 (rev44 Phase 1 anchor=0.3, registered as buffer)
  [OK] forward pass shapes OK
  [OK] backward OK; loss=0.8206
  [OK] diag: {'damps_params': 133, 'tau': 0.30000001192092896, 'tau_clamped': 0.30000001192092896, 'tau_mode': 'static', 'alpha_img': 0.1000000

## 3. Multi-Seed Training (DAMPS-MMHCL) â€” rev44 Phase 1 four-variant sweep + Round-2

Train the **DAMPS-MMHCL** model across the **four rev44 / Revision 11 Phase 1 variants Ã— 5 random seeds** = **20** training runs. The next code cell calls `MMHCL_DAMPS_Project/train.py` directly â€” the same entry-point used by `scripts/run_phase1_ablation.py`. (To match a 10-seed paper protocol, set `n_runs = 10` in that cell for 40 runs total.)

**After** that sweep completes, run the **following code cell (Round-2 / variant `apc_off_combined`)** so the Day-1 finding in `Phase1_Day1_Final_Verdict.tex` is checked on the **same 5 seeds** â€” then run **Â§4** and **Â§4.5** below to refresh aggregates and Bonferroni tests.

After the eval-protocol audit (Â§3.0), you can run **Â§3.0.5** *before* this sweep: D1 (CPU) plus optional D2/D3/APC GPU probes from `scripts/day1_diagnostic_sprint.py` (see `Phase1_RootCause_Analysis_and_Remediation_Roadmap.tex` and `MMHCL_DAMPS_Project/README.md` Â§5.0).

### Backbone hyperparameters (parity with original MMHCL paper)
- **Batch Size**: 1024 (RTX 5090, 32 GB VRAM)
- **Max Epochs**: 250
- **Learning Rate**: 0.0001
- **Embedding Dim**: 64 / **Top-k**: 5
- **Ï„**: variant-dependent (anchor 0.1 learnable, Phase 1 0.3 static)
- **`User_layers` / `Item_layers`**: 3 / 2
- **`user_loss_ratio` / `item_loss_ratio`**: 0.03 / 0.07
- **Early Stopping**: patience = 30, min_epochs = 75, monitor = `val_recall@20`
- **ReduceLROnPlateau**: factor = 0.5, patience = 3

### Per-variant overrides (rev44 Â§4)

| Variant | `--temperature` | `--learnable_tau` | `--damps_avrf` |
| ------- | --------------- | ----------------- | -------------- |
| `anchor`   (a) â€” rev42 baseline | 0.1 | 1 | 1 |
| `tau03`    (b) â€” static Ï„ only  | 0.3 | 0 | 1 |
| `avrf_off` (c) â€” AVRF off only  | 0.1 | 1 | 0 |
| `combined` (d) â€” recommended    | 0.3 | 0 | 0 |

### Other DAMPS-specific flags (rev44, kept constant across variants)
- `--damps_apc 1` â€” Metadata-Aware Adaptive Phase Calibration
- `--damps_imcf 1` â€” Residual Inter-Modal Coherence Filter
- `--damps_soft_routing 1` â€” Soft Residual-Routing into HGCN
- `--damps_momentum 1` â€” Slim Momentum Encoder
- `--damps_data_driven_prior 1` â€” SNR-derived AVRF priors
- `--damps_num_categories 10` â€” # static metadata clusters for APC
- `--damps_warmup_epochs 10` â€” adaptive EMA warm-up
- `--rebuild_R 5` â€” Pattern B' rebuild cadence
- `--knn_chunk_size 4096` / `--faiss_threshold 60000` â€” Dual-path K-NN
- `--use_amp 1` â€” bfloat16 mixed precision
- `--clip_grad_norm 1.0` â€” gradient clipping

### Subset to a single variant (optional)
To run **only** the recommended Phase 1 config (variant `combined`), set `variants_to_run = {'combined': phase1_variants['combined']}` near the top of the training code cell. The default `variants_to_run = phase1_variants` runs all four (**20 runs** with `n_runs = 5`). Round-2 **`apc_off_combined`** is always launched from the **separate** cell immediately after training (5 additional runs when you execute it).

### 2.7 Dataset integrity — HF MMHCL Clothing (LATTICE 39k)

This notebook expects the **official MMHCL Clothing release** from
[Xu-SII-BNU/MMHCL on Hugging Face](https://huggingface.co/datasets/Xu-SII-BNU/MMHCL/tree/main/Clothing)
(the LATTICE **39k** variant), **not** the older sparse / remapped Clothing dump.

| Asset | Path | Expected |
| ----- | ---- | -------- |
| Splits | `data/Clothing/5-core/{train,val,test}.json` | **39,387** users, **23,033** items (`max_id + 1`) |
| Image feats | `data/Clothing/image_feat.npy` | shape `(23033, 4096)` |
| Text feats | `data/Clothing/text_feat.npy` | shape `(23033, 1024)` |
| Graph caches | `data/Clothing/5-core/*.pth` | **rebuilt** after a dataset swap |

The next cell:
1. Asserts the LATTICE 39k user/item counts on the JSON splits.
2. Asserts feature matrix shapes.
3. **Purges** stale graph caches (`UI_mat_sym.pth`, `User_mat_rw.pth`,
   `hypergraph_mat_mul_sym_topk_10.pth`, `hypergraph_mat_origin_topk_10.pth`)
   so the first training run rebuilds graphs for the new catalogue.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np

# ---------------------------------------------------------------------------
#  HF MMHCL Clothing — LATTICE 39k pre-flight (must pass before training)
# ---------------------------------------------------------------------------
try:
    _ROOT = PROJECT_ROOT
except NameError:
    _cwd = os.getcwd()
    if _cwd.endswith(("codes", "MMHCL_DAMPS_Project")):
        _ROOT = os.path.dirname(_cwd)
    else:
        _ROOT = _cwd

EXPECTED_N_USERS = 39387
EXPECTED_N_ITEMS = 23033
DATASET = "Clothing"
CORE = "5-core"

clothing_dir = Path(_ROOT) / "data" / DATASET
core_dir = clothing_dir / CORE

print("=" * 72)
print("DATASET INTEGRITY — HF MMHCL Clothing (LATTICE 39k)")
print("=" * 72)
print(f"  clothing_dir : {clothing_dir}")
print(f"  core_dir     : {core_dir}")

if not core_dir.is_dir():
    raise FileNotFoundError(
        f"Missing {core_dir}. Download Clothing from "
        "https://huggingface.co/datasets/Xu-SII-BNU/MMHCL and place "
        "5-core/{train,val,test}.json under data/Clothing/."
    )

max_uid = -1
max_iid = -1
n_keys: dict[str, int] = {}
n_inter: dict[str, int] = {}
for split in ("train", "val", "test"):
    path = core_dir / f"{split}.json"
    if not path.is_file():
        raise FileNotFoundError(f"Missing split file: {path}")
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    n_keys[split] = len(data)
    n_inter[split] = sum(len(v) for v in data.values())
    for uid_str, items in data.items():
        max_uid = max(max_uid, int(uid_str))
        if items:
            max_iid = max(max_iid, max(int(i) for i in items))
    print(
        f"  {split:5s}: keys={n_keys[split]:,}  "
        f"interactions={n_inter[split]:,}  "
        f"bytes={path.stat().st_size:,}"
    )

n_users = max_uid + 1
n_items = max_iid + 1
print(f"\n  inferred n_users (max_uid+1) = {n_users:,}")
print(f"  inferred n_items (max_iid+1) = {n_items:,}")

if n_users != EXPECTED_N_USERS or n_items != EXPECTED_N_ITEMS:
    raise RuntimeError(
        "Clothing splits are NOT the HF MMHCL LATTICE 39k release.\n"
        f"  got      n_users={n_users}, n_items={n_items}\n"
        f"  expected n_users={EXPECTED_N_USERS}, n_items={EXPECTED_N_ITEMS}\n"
        "  Re-download from "
        "https://huggingface.co/datasets/Xu-SII-BNU/MMHCL/tree/main/Clothing"
    )
print(
    f"  [OK] LATTICE 39k split counts match "
    f"({EXPECTED_N_USERS} users / {EXPECTED_N_ITEMS} items)"
)

# Feature matrices live next to 5-core/, not inside it.
image_path = clothing_dir / "image_feat.npy"
text_path = clothing_dir / "text_feat.npy"
for feat_path, expect_shape in (
    (image_path, (EXPECTED_N_ITEMS, 4096)),
    (text_path, (EXPECTED_N_ITEMS, 1024)),
):
    if not feat_path.is_file():
        raise FileNotFoundError(
            f"Missing {feat_path}. Download Clothing/image_feat.npy and "
            "Clothing/text_feat.npy from the HF MMHCL dataset."
        )
    arr = np.load(feat_path, mmap_mode="r")
    print(f"  {feat_path.name}: shape={tuple(arr.shape)} dtype={arr.dtype}")
    if tuple(arr.shape) != expect_shape:
        raise RuntimeError(
            f"{feat_path.name} shape {tuple(arr.shape)} != expected {expect_shape}. "
            "Wrong Clothing feature dump — re-download from HF MMHCL."
        )
print("  [OK] image_feat / text_feat shapes match LATTICE 39k")

meta_path = clothing_dir / "meta_categories.npy"
if meta_path.is_file():
    meta = np.load(meta_path)
    print(f"  meta_categories.npy: shape={meta.shape}")
    if int(meta.shape[0]) != EXPECTED_N_ITEMS:
        print(
            "  [WARN] meta_categories length != n_items — APC will hash-fallback "
            "until you rebuild meta with build_meta_categories.py"
        )
else:
    print(
        "  [WARN] meta_categories.npy missing — APC uses deterministic hash "
        "fallback (see Day-1 H2 notes)"
    )

# Purge stale graph caches so the new catalogue is used on the next run.
CACHE_FILES = (
    "UI_mat_sym.pth",
    "User_mat_rw.pth",
    "hypergraph_mat_mul_sym_topk_10.pth",
    "hypergraph_mat_origin_topk_10.pth",
)
print("\n  Pre-run cache purge (forces graph rebuild):")
for name in CACHE_FILES:
    fp = core_dir / name
    if fp.is_file():
        fp.unlink()
        print(f"    DELETED {fp.name}")
    else:
        print(f"    absent  {fp.name}")

print("\n[OK] Dataset pre-flight passed — safe to run audit / training cells.")


### 3.0 rev44 Phase 1 Item #1 â€” 7-point Eval Protocol Audit (mandatory pre-flight)

Per `DAMPS_to_MMHCL_architecture_revision44.tex` Section 4 / `Phase1_Fixed_Implementation_Audit_rev44.tex` Finding 3, this audit **must PASS** before launching the four-variant Phase 1 sweep below. It runs `MMHCL_DAMPS_Project/scripts/audit_eval_protocol.py`, which verifies all seven checklist items against the on-disk Clothing dataset:

| # | Check | Source of truth |
| - | ----- | --------------- |
| (i)   | 5-core filtering threshold              | `args.core` |
| (ii)  | Train/val/test split + reproducibility  | counts in the JSON splits |
| (iii) | All-ranking vs sampled evaluation       | static analysis of `utility/batch_test.py::test_one_user` |
| (iv)  | NDCG `log2(i+2)` discount convention    | live probe of `metrics.dcg_at_k` |
| (v)   | User/item ID remapping consistency      | bounds check on every uid/iid in the splits |
| (vi)  | Popularity filtering at test time       | live simulation of one user's candidate pool |
| (vii) | @K cutoff sorting stability on ties     | `heapq.nlargest` determinism probe |

The cell raises `RuntimeError` on any **strict** failure, blocking the rest of the notebook. The previous Phase 1 audit specifically called this out as missing (Finding 3 in `Phase1_Fixed_Implementation_Audit_rev44.tex`).

In [4]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

# ---------------------------------------------------------------------------
#  rev44 Phase 1 Item #1 -- 7-point eval-protocol audit.
#
#  Self-bootstraps PROJECT_ROOT / DAMPS_DIR so this cell can run before the
#  multi-seed training cell defines those names. Re-running the audit cell
#  after the training cell has run is safe -- the values are identical.
# ---------------------------------------------------------------------------
_cwd = os.getcwd()
if _cwd.endswith(('codes', 'MMHCL_DAMPS_Project')):
    _PROJECT_ROOT = os.path.dirname(_cwd)
else:
    _PROJECT_ROOT = _cwd
_DAMPS_DIR = os.path.join(_PROJECT_ROOT, 'MMHCL_DAMPS_Project')
_DATA_DIR = os.path.join(_PROJECT_ROOT, 'data')
_PYTHON_EXE = sys.executable

# Keep the dataset name in sync with the multi-seed training cell.
_dataset = 'Clothing'

audit_script = os.path.join(_DAMPS_DIR, 'scripts', 'audit_eval_protocol.py')
if not os.path.isfile(audit_script):
    raise FileNotFoundError(
        f"Audit script not found at {audit_script}. Make sure "
        f"MMHCL_DAMPS_Project/scripts/audit_eval_protocol.py is present "
        f"(it ships with rev44 / Revision 11)."
    )

audit_cmd = [
    _PYTHON_EXE,
    audit_script,
    '--dataset', _dataset,
    '--data_path', _DATA_DIR + os.sep,
    '--core', '5',
]
print('Running rev44 Phase 1 7-point eval-protocol audit:')
print('  ' + ' '.join(audit_cmd))
print()

audit_res = subprocess.run(
    audit_cmd,
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace',
    cwd=_DAMPS_DIR,
)
print(audit_res.stdout)
if audit_res.returncode != 0:
    if audit_res.stderr:
        print('[stderr]')
        print(audit_res.stderr)
    raise RuntimeError(
        f"rev44 Phase 1 eval-protocol audit FAILED (exit={audit_res.returncode}). "
        f"At least one of the 7 checklist items did not pass; the four-variant "
        f"Phase 1 sweep is BLOCKED until the failure is resolved."
    )

print('[OK] rev44 Phase 1 eval-protocol audit PASSED -- proceeding to training.')

Running rev44 Phase 1 7-point eval-protocol audit:
  c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\scripts\audit_eval_protocol.py --dataset Clothing --data_path c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data\ --core 5

Auditing DAMPS-MMHCL evaluation protocol against rev44 Section 4
  cwd     = c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
  dataset = Clothing
  data_path = c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data\
  core    = 5

== (i) 5-core filtering threshold ==
  [OK]   args.core = 5 (matches MMHCL paper convention)

== (ii) Train/val/test split ratio + reproducibility ==
  [OK]   |train| = 90362 (81.0%) | |val| = 10234 (9.2%) | |test| = 10926 (9.8%)  -> to

### 3.0.5 Day 1 Diagnostic Sprint â€” D1 / D2 / D3

This block runs the **recall-focused, leakage-safe** probes from *Phase 1 Root-Cause Analysis and Remediation Roadmap* **Â§5.0** (Day 1 sprint), implemented as `MMHCL_DAMPS_Project/scripts/day1_diagnostic_sprint.py` (also summarized in `MMHCL_DAMPS_Project/README.md` Â§5.0).

| ID | What it checks | Hardware | When invoked |
|----|----------------|----------|--------------|
| **D1** | Metadata / APC integrity (H2): `meta_categories.npy` vs inferred `n_items`, label diversity | **CPU** | Always unless you add `--skip-d1` in a custom command |
| **D2** | Hypergraph coverage: top-*K* sweep on the Phase-1 **combined** recipe | **GPU** | `--run-d2` |
| **D3** | **Pure MMHCL** baseline (all DAMPS modules off) for the H10 sampled-eval scale check | **GPU** | `--run-d3` |
| **APC probe** | Single train with `--damps_apc 0` on the combined recipe | **GPU** | `--run-d1-apc-probe` |

**Exit code:** D1 alone may return **1** when the roadmapâ€™s *H2 suspected* banner fires (missing meta, shape mismatch, or collapsed label diversity). Treat that as **diagnostic**, not necessarily a blocker before you run GPU probes.

**Operational gate (not a bug):** The next cell defaults `DAY1_RUN_GPU = False` so **only D1** runs (CPU metadata audit). That is **intentional** â€” D1 is cheap and **fail-fast** on metadata integrity before GPU time. To run **D2 + D3 + APC probe** on the GPU, set `DAY1_RUN_GPU = True`.

**D2 sweep includes K = 5:** With GPU mode on, the notebook passes `--topk-list 5 10 15 20` so the first point matches the **paper default** top-*K*; the sprint script then prints **Î”Recall@20 vs K = 5** inline for K = 10, 15, 20 (see `day1_diagnostic_sprint.py`).

**APC probe:** Uses `topk_default = 5` on the **combined** recipe with only APC toggled off â€” matches the roadmap; no extra notebook flag.

**Log-path parsing:** `expected_log_path` mirrors `train.py`â€™s filename convention so parsed logs recover `BEST_Test_Recall@20` / `BEST_Test_NDCG@20`, Î”Recall, and the H10 verdict â€” handled in the script, not duplicated here.

**Pre-flight:** Confirm `data/Clothing/meta_categories.npy` exists and that the flattened length matches **n_items** (â‰ˆ **23 033** for 5-core Clothing, as D1 prints). If D1 exits **1**, restore the original metadata from the dataset release or **explicitly accept** hash-fallback APC (distorts APC signals).

**After GPU diagnostics:** Run `paired_ttest.py` (5 seeds Ã— variant) to refresh the Phase-1 stop-gate comparison table.

**Toggles (next cell):** Lower `DAY1_DIAG_EPOCH` for smoke tests when `DAY1_RUN_GPU = True`.

**Full Phase-1 diagnostic command** (from `MMHCL_DAMPS_Project/`, copy verbatim):

```text
python scripts/day1_diagnostic_sprint.py \
    --dataset Clothing --data_path ../data/ \
    --seed 737791071 --epoch 250 \
    --topk-list 5 10 15 20 \
    --run-d2 --run-d3 --run-d1-apc-probe
```

In [11]:
from __future__ import annotations

import os
import subprocess
import sys

# ---------------------------------------------------------------------------
#  Path resolution â€” mirror the multi-seed training cell (cwd = DAMPS_DIR).
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(("codes", "MMHCL_DAMPS_Project")):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

DAMPS_DIR: str = os.path.join(PROJECT_ROOT, "MMHCL_DAMPS_Project")
if not os.path.isdir(DAMPS_DIR):
    raise FileNotFoundError(f"Expected MMHCL_DAMPS_Project under {PROJECT_ROOT!r}")

if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
    os.chdir(DAMPS_DIR)
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)

PYTHON_EXE: str = sys.executable
DATA_PATH: str = os.path.join(PROJECT_ROOT, "data") + os.sep

dataset: str = "Clothing"

# --- toggles (see Â§3.0.5 markdown) ---
# DAY1_RUN_GPU=False is intentional: D1 (CPU) runs first; set True for D2+D3+APC probe.
DAY1_RUN_GPU: bool = True
DAY1_DIAG_EPOCH: int = 250
DAY1_SEED: int = 737791071
# D2 top-K list when GPU mode is on (include 5 = paper default for Î”Recall vs K=5).
DAY1_D2_TOPK_LIST: tuple[int, ...] = (5, 10, 15, 20)
DAY1_USE_WANDB: int = 0
DAY1_WANDB_PROJECT: str = "damps-mmhcl-clothing"
DAY1_WANDB_ENTITY: str = "baitapck51cc-uet"

# --- pre-flight: meta file vs n_items (5-core Clothing â‰ˆ 23_033 items) ---
_meta = os.path.join(PROJECT_ROOT, "data", dataset, "meta_categories.npy")
if os.path.isfile(_meta):
    import numpy as np

    _arr = np.load(_meta).astype(np.int64).reshape(-1)
    print(f"[pre-flight] meta_categories.npy: path={_meta!s}  len={len(_arr):,}")
    print("             Compare len to D1 'inferred n_items' above; they should match.")
else:
    print(f"[pre-flight] MISSING {_meta} â€” D1 may exit 1 (H2); restore meta or accept hash-fallback APC.")

cmd: list[str] = [
    PYTHON_EXE,
    os.path.join("scripts", "day1_diagnostic_sprint.py"),
    "--dataset",
    dataset,
    "--data_path",
    DATA_PATH,
    "--epoch",
    str(DAY1_DIAG_EPOCH),
    "--seed",
    str(DAY1_SEED),
    "--python_exe",
    PYTHON_EXE,
]
if DAY1_USE_WANDB:
    cmd += ["--use_wandb", "1", "--wandb_project", DAY1_WANDB_PROJECT]
    if DAY1_WANDB_ENTITY:
        cmd += ["--wandb_entity", DAY1_WANDB_ENTITY]

if DAY1_RUN_GPU:
    cmd += [
        "--topk-list",
        *[str(k) for k in DAY1_D2_TOPK_LIST],
        "--run-d2",
        "--run-d3",
        "--run-d1-apc-probe",
    ]

print("Day-1 diagnostic sprint")
print("  command:", subprocess.list2cmdline(cmd))
print("  cwd    :", DAMPS_DIR)

proc = subprocess.run(cmd, cwd=DAMPS_DIR, check=False)
print("\nreturn code:", proc.returncode)
if proc.returncode == 1 and not DAY1_RUN_GPU:
    print(
        "[NOTE] Exit code 1 on D1-only often means roadmap H2 suspected â€” "
        "see the banner above; you may still enable DAY1_RUN_GPU for D2/D3."
    )
elif proc.returncode != 0:
    print("[WARN] Non-zero exit code â€” review logs above.")

[pre-flight] MISSING c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data\Clothing\meta_categories.npy â€” D1 may exit 1 (H2); restore meta or accept hash-fallback APC.
Day-1 diagnostic sprint
  command: c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe scripts\day1_diagnostic_sprint.py --dataset Clothing --data_path "c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\data\\" --epoch 250 --seed 737791071 --python_exe c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe --topk-list 5 10 15 20 --run-d2 --run-d3 --run-d1-apc-probe
  cwd    : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project

return code: 1
[WARN] Non-zero exit code â€” review logs above.


In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
import json
import re
import numpy as np
import time
import random
from pathlib import Path
from typing import Any, Optional

# ---------------------------------------------------------------------------
#  Resolve project directories. The DAMPS-MMHCL package lives under
#  MMHCL_DAMPS_Project/. ``train.py`` writes its logs relative to its own
#  cwd as ``../<dataset>/<path_name>/<path_name>.txt``, so we must launch
#  the subprocess with cwd = MMHCL_DAMPS_Project.
# ---------------------------------------------------------------------------
current_dir: str = os.getcwd()
if current_dir.endswith(('codes', 'MMHCL_DAMPS_Project')):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

DAMPS_DIR: str = os.path.join(PROJECT_ROOT, 'MMHCL_DAMPS_Project')
DATA_DIR: str = os.path.join(PROJECT_ROOT, 'data')

# Switch into the DAMPS package directory so relative paths in train.py
# (``../{dataset}/...``) resolve to the workspace root.
if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
    os.chdir(DAMPS_DIR)
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)

PYTHON_EXE: str = sys.executable

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Working directory : {os.getcwd()}")
print(f"Python executable : {PYTHON_EXE}")

print("\n" + "=" * 80)
print("MULTI-SEED TRAINING: DAMPS-MMHCL â€” rev44 Phase 1 four-variant sweep")
print("=" * 80)

# Generate reproducible-but-fresh seeds based on current time
base_seed: int = int(time.time_ns() % (2 ** 31))
random.seed(base_seed)

# Phase 1 notebook default: 5 seeds per variant (4 Ã— 5 = 20 runs). For a
# stricter 10-seed protocol (40 runs), set ``n_runs = 10``.
# Statistical reporting still uses paired t-tests on whatever n_runs is.
n_runs: int = 5
seeds: list[int] = [random.randint(1, 2 ** 31 - 1) for _ in range(n_runs)]

print(f"\nGenerated {n_runs} random seeds based on current time:")
print(f"Base timestamp seed : {base_seed}")
print(f"Seeds for experiments: {seeds}")
print(f"(These seeds are saved in the summary file for reproducibility)")

# W&B configuration (already created at login time, kept in sync here)
wandb_project: str = 'damps-mmhcl-clothing'
wandb_entity: str = 'baitapck51cc-uet'

# ---------------------------------------------------------------------------
#  Backbone hyper-parameters (parity with the original MMHCL paper)
# ---------------------------------------------------------------------------
dataset: str = 'Clothing'
gpu_id: int = 0
epoch: int = 250
verbose: int = 5

batch_size: int = 1024
lr: float = 0.0001
regs: float = 1e-3
embed_size: int = 64
topk: int = 5
core: int = 5
User_layers: int = 3
Item_layers: int = 2
user_loss_ratio: float = 0.03
item_loss_ratio: float = 0.07
temperature: float = 0.3       # rev44 Phase 1 anchor of the static-tau sweep ({0.2, 0.3, 0.5})
learnable_tau: int = 0         # 0 = static buffer (rev44 Phase 1 default); 1 = nn.Parameter (rev42 anchor)
clip_grad_norm: float = 1.0

# Early stopping
early_stopping_patience: int = 30
early_stopping_min_epochs: int = 75
early_stopping_min_delta: float = 0.0001
early_stopping_mode: str = 'max'
early_stopping_restore_best: int = 1

# ReduceLROnPlateau
use_reduce_lr: int = 1
reduce_lr_factor: float = 0.5
reduce_lr_patience: int = 3
reduce_lr_min: float = 1e-6

# ---------------------------------------------------------------------------
#  DAMPS-specific configuration (Revision 9, full feature set)
# ---------------------------------------------------------------------------
damps_apc: int = 1                    # Metadata-Aware Adaptive Phase Calibration
damps_avrf: int = 0                   # rev44 Phase 1 default: AVRF OFF (over-attenuates sparse Clothing); 1 = rev42 anchor
damps_imcf: int = 1                   # Residual Inter-Modal Coherence Filter
damps_permutation_fft: int = 0        # Falsifiability ablation (off in main runs)
damps_soft_routing: int = 1           # h_input = h_raw + alpha * LN(h_cal)
damps_momentum: int = 1               # Slim Momentum Encoder
damps_data_driven_prior: int = 1      # SNR-derived AVRF priors
damps_num_categories: int = 10        # Static metadata clusters for APC
damps_warmup_epochs: int = 10         # Adaptive EMA warm-up
rebuild_R: int = 5                    # Pattern B' cadence
faiss_threshold: int = 60_000         # FAISS HNSW fallback threshold
faiss_use_gpu: int = 1                # GPU FAISS resources when N >= threshold
knn_chunk_size: int = 4096            # Chunked PyTorch K-NN tile size
knn_efsearch: int = 64                # HNSW efSearch (recall/speed trade-off)
use_amp: int = 1                      # bfloat16 mixed precision

# --- Speedup Guide toggles (Speedup_Guide_complete sections 3.1 / 3.2) ---
use_torch_compile: int = 1            # Wrap DAMPS submodule in torch.compile
torch_compile_mode: str = "reduce-overhead"  # JIT mode (best for medium models)
torch_compile_dynamic: int = 1        # dynamic=True -> tolerates batch_size drift

# ---------------------------------------------------------------------------
#  rev44 / Revision 11 -- Phase 1 four-variant sweep (Section 4 of the spec)
#  Each variant overrides three knobs vs the rev42 anchor:
#     (a) anchor   : --temperature 0.1 --learnable_tau 1 --damps_avrf 1
#     (b) tau03    : --temperature 0.3 --learnable_tau 0 --damps_avrf 1
#     (c) avrf_off : --temperature 0.1 --learnable_tau 1 --damps_avrf 0
#     (d) combined : --temperature 0.3 --learnable_tau 0 --damps_avrf 0  <- recommended
# ---------------------------------------------------------------------------
phase1_variants: dict[str, dict[str, Any]] = {
    'anchor':   {'temperature': 0.1, 'learnable_tau': 1, 'damps_avrf': 1,
                 'label': '(a) rev42 anchor'},
    'tau03':    {'temperature': 0.3, 'learnable_tau': 0, 'damps_avrf': 1,
                 'label': '(b) static tau=0.3 only'},
    'avrf_off': {'temperature': 0.1, 'learnable_tau': 1, 'damps_avrf': 0,
                 'label': '(c) AVRF off only'},
    'combined': {'temperature': 0.3, 'learnable_tau': 0, 'damps_avrf': 0,
                 'label': '(d) static tau + AVRF off  (RECOMMENDED Phase 1)'},
}

# Subset to run. Default = ALL FOUR variants (5 seeds Ã— 4 variants = 20
# training runs with the default ``n_runs``). To run ONLY the recommended
# Phase 1 config, set this to {'combined': phase1_variants['combined']}.
variants_to_run: dict[str, dict[str, Any]] = phase1_variants

# Per-variant storage: {variant_name: [per-seed result dict, ...]}
all_variant_results: dict[str, list[dict[str, Any]]] = {v: [] for v in variants_to_run}

print(f"\n{'=' * 80}")
print("rev44 PHASE 1 -- Four-Variant Training Configuration")
print(f"{'=' * 80}")
print(f"  Dataset            : {dataset}")
print(f"  GPU ID             : {gpu_id}")
print(f"  Max Epochs         : {epoch}")
print(f"  Batch Size         : {batch_size}")
print(f"  Learning Rate      : {lr}")
print(f"  Early Stopping     : patience={early_stopping_patience}, min_epochs={early_stopping_min_epochs}")
print(f"  Mixed Precision    : bfloat16  (use_amp={use_amp})")
print(f"  Rebuild Cadence    : every {rebuild_R} epochs (Pattern B')")
print(f"  DAMPS Switches     : APC={damps_apc} | IMCF={damps_imcf} | "
      f"Soft-Routing={damps_soft_routing} | Momentum={damps_momentum}")
print(f"  Variants           : {list(variants_to_run.keys())}")
print(f"  Seeds              : {seeds}")
print(f"  Total runs         : {len(variants_to_run)} variants x {n_runs} seeds = "
      f"{len(variants_to_run) * n_runs}")
print(f"  W&B Project        : {wandb_project}")


def _build_path_name(
    seed: int,
    temp_val: float,
    learn_tau: int,
    avrf_val: int,
) -> str:
    """Mirror the directory naming scheme of ``train.py::_experiment_paths``.

    Includes the ``taulearn={0,1}`` token introduced in rev44 so the four
    Phase 1 variants land in distinct log folders.
    """
    return (
        f"damps_uu_ii={User_layers}_{Item_layers}"
        f"_{user_loss_ratio}_{item_loss_ratio}"
        f"_topk={topk}_t={temp_val}_taulearn={learn_tau}_R={rebuild_R}"
        f"_apc={damps_apc}_avrf={avrf_val}_imcf={damps_imcf}"
        f"_regs={regs}_dim={embed_size}_seed={seed}_"
    )


# ---------------------------------------------------------------------------
#  Outer loop: 4 variants -> Inner loop: n_runs seeds (default 5)
# ---------------------------------------------------------------------------
total_runs: int = len(variants_to_run) * n_runs
global_run_idx: int = 0

for variant_name, ov in variants_to_run.items():
    v_temp: float = float(ov['temperature'])
    v_learn_tau: int = int(ov['learnable_tau'])
    v_avrf: int = int(ov['damps_avrf'])
    v_label: str = str(ov['label'])

    print(f"\n{'#' * 80}")
    print(f"# VARIANT '{variant_name}' -- {v_label}")
    print(f"#   --temperature {v_temp}  --learnable_tau {v_learn_tau}  --damps_avrf {v_avrf}")
    print(f"{'#' * 80}")

    for run_idx, seed in enumerate(seeds, 1):
        global_run_idx += 1
        print(f"\n{'=' * 80}")
        print(
            f"RUN {global_run_idx}/{total_runs}  (variant '{variant_name}' "
            f"seed {run_idx}/{n_runs}={seed}): Training DAMPS-MMHCL"
        )
        print(f"{'=' * 80}")

        cmd: list[str] = [
            PYTHON_EXE, 'train.py',
            # Data / schedule
            '--dataset', dataset,
            '--gpu_id', str(gpu_id),
            '--seed', str(seed),
            '--epoch', str(epoch),
            '--verbose', str(verbose),
            '--batch_size', str(batch_size),
            '--lr', str(lr),
            '--regs', str(regs),
            '--clip_grad_norm', str(clip_grad_norm),
            # Model architecture
            '--embed_size', str(embed_size),
            '--topk', str(topk),
            '--core', str(core),
            '--User_layers', str(User_layers),
            '--Item_layers', str(Item_layers),
            '--user_loss_ratio', str(user_loss_ratio),
            '--item_loss_ratio', str(item_loss_ratio),
            # ----- rev44 Phase 1 variant overrides -----
            '--temperature', str(v_temp),
            '--learnable_tau', str(v_learn_tau),
            # Early stopping
            '--early_stopping_patience', str(early_stopping_patience),
            '--early_stopping_min_epochs', str(early_stopping_min_epochs),
            '--early_stopping_min_delta', str(early_stopping_min_delta),
            '--early_stopping_mode', str(early_stopping_mode),
            '--early_stopping_restore_best', str(early_stopping_restore_best),
            # ReduceLROnPlateau
            '--use_reduce_lr', str(use_reduce_lr),
            '--reduce_lr_factor', str(reduce_lr_factor),
            '--reduce_lr_patience', str(reduce_lr_patience),
            '--reduce_lr_min', str(reduce_lr_min),
            # DAMPS-specific
            '--damps_apc', str(damps_apc),
            '--damps_avrf', str(v_avrf),                 # rev44 Phase 1 variant override
            '--damps_imcf', str(damps_imcf),
            '--damps_permutation_fft', str(damps_permutation_fft),
            '--damps_soft_routing', str(damps_soft_routing),
            '--damps_momentum', str(damps_momentum),
            '--damps_data_driven_prior', str(damps_data_driven_prior),
            '--damps_num_categories', str(damps_num_categories),
            '--damps_warmup_epochs', str(damps_warmup_epochs),
            # Pattern B' rebuild
            '--rebuild_R', str(rebuild_R),
            '--faiss_threshold', str(faiss_threshold),
            '--faiss_use_gpu', str(faiss_use_gpu),
            '--knn_chunk_size', str(knn_chunk_size),
            '--knn_efsearch', str(knn_efsearch),
            # Mixed precision
            '--use_amp', str(use_amp),
            # Speedup Guide: torch.compile on the DAMPS submodule (+25-35%)
            '--use_torch_compile', str(use_torch_compile),
            '--torch_compile_mode', str(torch_compile_mode),
            '--torch_compile_dynamic', str(torch_compile_dynamic),
            # W&B -- include the variant tag so the runs are filterable by group
            '--use_wandb', '1',
            '--wandb_project', wandb_project,
            '--wandb_entity', wandb_entity,
            '--wandb_run_name', f'phase1_{variant_name}_seed_{seed}',
            '--ablation_target', f'phase1_{variant_name}',
        ]

        print(f"Command: {' '.join(cmd)}")
        print(f"Current directory: {os.getcwd()}\n")

        env: dict[str, str] = os.environ.copy()
        env['PYTHONIOENCODING'] = 'utf-8'
        env['PYTHONUNBUFFERED'] = '1'

        result: subprocess.CompletedProcess[str] = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            env=env,
            encoding='utf-8',
            errors='replace',
        )

        if result.stdout:
            print(result.stdout)

        if result.returncode != 0:
            print(
                f"\n[WARNING] variant '{variant_name}' seed={seed} exited "
                f"with code {result.returncode}"
            )
            if result.stderr:
                print("\n[ERROR OUTPUT]:")
                print(result.stderr)
        else:
            print(
                f"\n[OK] variant '{variant_name}' seed={seed} completed successfully"
            )

        # ----- Extract metrics from the per-run log file -----------------
        path_name: str = _build_path_name(seed, v_temp, v_learn_tau, v_avrf)
        # ``ablation_target`` is appended to ``path_name`` by train.py.
        path_name = path_name + f"phase1_{variant_name}"
        log_file: str = f"../{dataset}/{path_name}/{path_name}.txt"

        if os.path.exists(log_file):
            with open(log_file, 'r', encoding='utf-8') as f:
                log_content: str = f.read()

            sep: str = r"[:=]\s*"
            # The order matters: ``BEST_Test_*`` is the canonical
            # validation-selected test result. ``BEST_Val_*`` is logged
            # alongside so the notebook can surface BOTH numbers --
            # avoiding the classic confusion where the user reads the
            # WandB ``val/recall@20`` peak (validation) and compares it
            # to the program's BEST_Test_Recall@20 (test). The two are
            # measured on different splits and should NOT be expected
            # to match.
            recall_test_match: Optional[re.Match[str]] = (
                re.search(rf'BEST_Test_Recall@20{sep}([\d.]+)', log_content)
                or re.search(rf'Test_Recall@20{sep}([\d.]+)', log_content)
            )
            recall_val_match: Optional[re.Match[str]] = re.search(
                rf'BEST_Val_Recall@20{sep}([\d.]+)', log_content
            )
            recall_val_epoch_match: Optional[re.Match[str]] = re.search(
                rf'BEST_Val_Recall_Peak_Epoch{sep}(-?\d+)', log_content
            )
            precision_match: Optional[re.Match[str]] = (
                re.search(rf'BEST_Test_Precision@20{sep}([\d.]+)', log_content)
                or re.search(rf'Test_Precision@20{sep}([\d.]+)', log_content)
            )
            ndcg_test_match: Optional[re.Match[str]] = (
                re.search(rf'BEST_Test_NDCG@20{sep}([\d.]+)', log_content)
                or re.search(rf'Test_NDCG@20{sep}([\d.]+)', log_content)
            )
            ndcg_val_match: Optional[re.Match[str]] = re.search(
                rf'BEST_Val_NDCG@20{sep}([\d.]+)', log_content
            )

            tau_match: Optional[re.Match[str]] = re.search(r'tau\s*=\s*([\d.]+)', log_content)
            sat_match: Optional[re.Match[str]] = re.search(r'tanh_sat\s*=\s*([\d.]+)', log_content)
            nnz_match: Optional[re.Match[str]] = re.search(r'nnz\s*=\s*([\d_]+)', log_content)
            avg_deg_match: Optional[re.Match[str]] = re.search(r'avg_deg\s*=\s*([\d.]+)', log_content)

            if recall_test_match and ndcg_test_match:
                precision_value: Optional[float] = (
                    float(precision_match.group(1)) if precision_match is not None else None
                )
                val_recall_value: Optional[float] = (
                    float(recall_val_match.group(1)) if recall_val_match else None
                )
                val_ndcg_value: Optional[float] = (
                    float(ndcg_val_match.group(1)) if ndcg_val_match else None
                )
                val_recall_peak_epoch: Optional[int] = (
                    int(recall_val_epoch_match.group(1))
                    if recall_val_epoch_match else None
                )
                run_result: dict[str, Any] = {
                    'variant': variant_name,
                    'variant_label': v_label,
                    'seed': seed,
                    # NOTE: keys ``recall@20`` and ``ndcg@20`` are kept for
                    # backward compatibility with the aggregator + paired-
                    # t-test cell; they map to BEST_Test_*  (i.e. the test
                    # result at the val-recall / val-ndcg peak epoch).
                    'recall@20': float(recall_test_match.group(1)),
                    'precision@20': precision_value,
                    'ndcg@20': float(ndcg_test_match.group(1)),
                    # New: surface the validation peak too so a reviewer
                    # can sanity-check the gap with the WandB val charts.
                    'val_recall@20': val_recall_value,
                    'val_ndcg@20': val_ndcg_value,
                    'val_recall_peak_epoch': val_recall_peak_epoch,
                    'tau_final': float(tau_match.group(1)) if tau_match else None,
                    'tanh_sat_final': float(sat_match.group(1)) if sat_match else None,
                    'last_nnz': int(nnz_match.group(1).replace('_', '')) if nnz_match else None,
                    'last_avg_deg': float(avg_deg_match.group(1)) if avg_deg_match else None,
                    'log_file': log_file,
                }
                all_variant_results[variant_name].append(run_result)
                precision_str: str = (
                    f"{precision_value:.6f}" if precision_value is not None else "n/a"
                )
                # Disambiguated print: explicitly say "TEST (at val-recall peak)"
                # for the headline metric and surface VAL alongside so the
                # reviewer never has to guess which split the number is on.
                print(
                    f"  Extracted TEST metrics  (snapshotted at the epoch where val_recall@20 peaked): "
                    f"Recall@20={run_result['recall@20']:.6f}, "
                    f"Precision@20={precision_str}, "
                    f"NDCG@20={run_result['ndcg@20']:.6f}"
                )
                if val_recall_value is not None or val_ndcg_value is not None:
                    epoch_str = (
                        str(val_recall_peak_epoch)
                        if val_recall_peak_epoch is not None else "?"
                    )
                    val_recall_str = (
                        f"{val_recall_value:.6f}" if val_recall_value is not None else "n/a"
                    )
                    val_ndcg_str = (
                        f"{val_ndcg_value:.6f}" if val_ndcg_value is not None else "n/a"
                    )
                    print(
                        f"  Reference VAL peaks (used for early stopping ONLY -- not the headline): "
                        f"val_Recall@20={val_recall_str} (epoch {epoch_str}), "
                        f"val_NDCG@20={val_ndcg_str}"
                    )
                if run_result['tau_final'] is not None:
                    print(f"  DAMPS diagnostics : tau={run_result['tau_final']:.4f}, "
                          f"tanh_sat={run_result['tanh_sat_final']}, "
                          f"last_nnz={run_result['last_nnz']}, "
                          f"last_avg_deg={run_result['last_avg_deg']}")
            else:
                print(f"  [WARN] Could not extract metrics from log file: {log_file}")
        else:
            print(f"  [WARN] Log file not found: {log_file}")

# ---------------------------------------------------------------------------
# Backward-compatibility alias: legacy cells (and the W&B sweep notebook)
# expect ``all_results`` to be a flat list of every per-seed run across every
# variant. We expose it here in addition to the per-variant dictionary.
# ---------------------------------------------------------------------------
all_results: list[dict[str, Any]] = [
    r for v_results in all_variant_results.values() for r in v_results
]

print(f"\n{'=' * 80}")
print("ALL DAMPS-MMHCL Phase 1 TRAINING RUNS COMPLETED")
print(f"{'=' * 80}")
print(f"Total runs collected   : {len(all_results)} / {total_runs}")
for v in variants_to_run:
    print(f"  variant '{v:<10s}' : {len(all_variant_results[v])} successful runs")

Project Root      : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing
Working directory : c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project
Python executable : c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe

MULTI-SEED TRAINING: DAMPS-MMHCL â€” rev44 Phase 1 four-variant sweep

Generated 5 random seeds based on current time:
Base timestamp seed : 85462164
Seeds for experiments: [1404632241, 2005933260, 1280068671, 762951128, 569184024]
(These seeds are saved in the summary file for reproducibility)

rev44 PHASE 1 -- Four-Variant Training Configuration
  Dataset            : Clothing
  GPU ID             : 0
  Max Epochs         : 250
  Batch Size         : 1024
  Learning Rate      : 0.0001
  Early Stopping     : patience=30, min_epochs=75
  Mixed Precision    : bfloat16  (use_amp=1)
  Rebuild Cadence    : every 5 epochs (Pattern B')
  DAMPS Switches     : AP

In [ ]:
"""
rev44 Phase 1 â€” ROUND 2:  apc_off_combined (5-seed validation)
==============================================================

Runs immediately after the **four-variant** training cell. See
`Phase1_Day1_Final_Verdict.tex`: Day-1 APC-off probe **R@20 â‰ˆ 0.09197** (paper
**0.0881**); **combined** on **hash-fallback** ~**0.0851** (~**âˆ’0.0069** vs
APC-off). **H3** rejected; **H10** deprioritised.

This cell trains **(e) apc_off_combined** (`--damps_apc 0`, same Ï„/AVRF as (d))
on the **`seeds`** list from the sweep when that cell has already run; **otherwise**
it **generates** ``n_runs`` fresh random seeds (default **5**, same scheme as Â§3).
**PASS Phase 1** if mean **test R@20 â‰¥ 0.0870** (rev44 Â§5.2; full NDCG verdict in Â§4.5). If **FAIL**, run
**`build_meta_categories.py`**, restore **`meta_categories.npy`**, keep APC on
for **combined**, and re-run Phase 1.
"""

from __future__ import annotations

import os
import random
import re
import subprocess
import sys
import time
from typing import Any

# ---------------------------------------------------------------------------
# Standalone bootstrap (no Â§3 cell required).
#
# IPython stores notebook variables in ``user_ns``.  Writing only into a
# nested function's ``globals()`` view can leave ``seeds`` invisible to the
# rest of this cell on some builds â€” so we update **both** ``user_ns`` and
# ``globals()`` here.
# ---------------------------------------------------------------------------
try:
    _USER_NS: dict[str, Any] = get_ipython().user_ns  # type: ignore[name-defined]
except Exception:
    _USER_NS = globals()

_boot: dict[str, Any] = {}

if "PROJECT_ROOT" not in _USER_NS or "DAMPS_DIR" not in _USER_NS:
    _cd = os.getcwd()
    if _cd.endswith(("codes", "MMHCL_DAMPS_Project")):
        _boot["PROJECT_ROOT"] = os.path.dirname(_cd)
    else:
        _boot["PROJECT_ROOT"] = _cd
    _boot["DAMPS_DIR"] = os.path.join(_boot["PROJECT_ROOT"], "MMHCL_DAMPS_Project")

if "seeds" not in _USER_NS or not _USER_NS.get("seeds"):
    _n = int(_USER_NS.get("n_runs", 5))
    _bs = int(time.time_ns() % (2**31))
    random.seed(_bs)
    _boot["n_runs"] = _n
    _boot["base_seed"] = _bs
    _boot["seeds"] = [random.randint(1, 2**31 - 1) for _ in range(_n)]
    print(
        f"[apc_off_combined] standalone: generated {_n} random seeds "
        f"(base_seed={_bs})"
    )
    print(f"  seeds = {_boot['seeds']}")

_defaults: dict[str, Any] = {
    "dataset": "Clothing",
    "PYTHON_EXE": sys.executable,
    "wandb_project": "damps-mmhcl-clothing",
    "wandb_entity": "baitapck51cc-uet",
    "gpu_id": 0,
    "epoch": 250,
    "verbose": 5,
    "batch_size": 1024,
    "lr": 0.0001,
    "regs": 1e-3,
    "embed_size": 64,
    "topk": 5,
    "core": 5,
    "User_layers": 3,
    "Item_layers": 2,
    "user_loss_ratio": 0.03,
    "item_loss_ratio": 0.07,
    "clip_grad_norm": 1.0,
    "early_stopping_patience": 30,
    "early_stopping_min_epochs": 75,
    "early_stopping_min_delta": 0.0001,
    "early_stopping_mode": "max",
    "early_stopping_restore_best": 1,
    "use_reduce_lr": 1,
    "reduce_lr_factor": 0.5,
    "reduce_lr_patience": 3,
    "reduce_lr_min": 1e-6,
    "damps_imcf": 1,
    "damps_permutation_fft": 0,
    "damps_soft_routing": 1,
    "damps_momentum": 1,
    "damps_data_driven_prior": 1,
    "damps_num_categories": 10,
    "damps_warmup_epochs": 10,
    "rebuild_R": 5,
    "faiss_threshold": 60_000,
    "faiss_use_gpu": 1,
    "knn_chunk_size": 4096,
    "knn_efsearch": 64,
    "use_amp": 1,
    "use_torch_compile": 1,
    "torch_compile_mode": "reduce-overhead",
    "torch_compile_dynamic": 1,
}
for _k, _v in _defaults.items():
    if _k not in _USER_NS and _k not in _boot:
        _boot[_k] = _v

if "all_variant_results" not in _USER_NS:
    _boot["all_variant_results"] = {}
if "phase1_variants" not in _USER_NS:
    _boot["phase1_variants"] = {}

_USER_NS.update(_boot)
globals().update(_boot)

PROJECT_ROOT = _USER_NS["PROJECT_ROOT"]
DAMPS_DIR = _USER_NS["DAMPS_DIR"]
PYTHON_EXE = _USER_NS["PYTHON_EXE"]
dataset = _USER_NS["dataset"]
seeds = _USER_NS["seeds"]

if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
    os.chdir(DAMPS_DIR)
if DAMPS_DIR not in sys.path:
    sys.path.insert(0, DAMPS_DIR)

# ---------------------------------------------------------------------------
# rev44 Phase 1 -- 5th variant config
#   Equal to (d) combined  PLUS  --damps_apc 0
# ---------------------------------------------------------------------------
variant_name: str = "apc_off_combined"
variant_label: str = "(e) static tau + AVRF off + APC OFF  (ROUND-2 Phase 1 candidate)"

v_temp:      float = 0.3   # static tau anchor
v_learn_tau: int   = 0     # static (not learnable)
v_avrf:      int   = 0     # AVRF off
v_apc:       int   = 0     # NEW: APC off  (overrides cell-16 `damps_apc = 1`)

print("\n" + "#" * 80)
print(f"# VARIANT '{variant_name}' -- {variant_label}")
print(f"#   --temperature {v_temp} --learnable_tau {v_learn_tau} "
      f"--damps_avrf {v_avrf} --damps_apc {v_apc}")
print("#" * 80)
print(f"\n  seeds          : {seeds}")
print(f"  runs           : {len(seeds)}  (single-variant 5-seed validation)")
print(f"  expected wall  : ~3 hours on RTX 5090 (bfloat16 + torch.compile)")
print(f"  stop-gate      : mean(R@20) >= 0.0870 to PASS Phase 1")

# Storage for this variant only.  Will be MERGED into the global
# `all_variant_results` from cell 16 so the Â§4 / Â§4.5 cells pick it up.
_apc_off_results: list[dict[str, Any]] = []

# Regex pre-compiled to extract metrics from the per-run log
_sep: str = r"[:=]\s*"
_re_test_r20  = re.compile(rf"BEST_Test_Recall@20{_sep}([\d.]+)")
_re_test_n20  = re.compile(rf"BEST_Test_NDCG@20{_sep}([\d.]+)")
_re_test_p20  = re.compile(rf"BEST_Test_Precision@20{_sep}([\d.]+)")
_re_val_r20   = re.compile(rf"BEST_Val_Recall@20{_sep}([\d.]+)")
_re_val_n20   = re.compile(rf"BEST_Val_NDCG@20{_sep}([\d.]+)")
_re_val_rpeak = re.compile(rf"BEST_Val_Recall_Peak_Epoch{_sep}(\d+)")
_re_val_npeak = re.compile(rf"BEST_Val_NDCG_Peak_Epoch{_sep}(\d+)")

for run_idx, seed in enumerate(seeds, 1):
    print(f"\n{'=' * 80}")
    print(f"RUN {run_idx}/{len(seeds)}  (variant '{variant_name}' seed={seed})")
    print(f"{'=' * 80}")

    cmd: list[str] = [
        PYTHON_EXE, "train.py",
        # Data / schedule
        "--dataset", dataset,
        "--gpu_id", str(gpu_id),
        "--seed", str(seed),
        "--epoch", str(epoch),
        "--verbose", str(verbose),
        "--batch_size", str(batch_size),
        "--lr", str(lr),
        "--regs", str(regs),
        "--clip_grad_norm", str(clip_grad_norm),
        # Model
        "--embed_size", str(embed_size),
        "--topk", str(topk),
        "--core", str(core),
        "--User_layers", str(User_layers),
        "--Item_layers", str(Item_layers),
        "--user_loss_ratio", str(user_loss_ratio),
        "--item_loss_ratio", str(item_loss_ratio),
        # rev44 Round-2 overrides
        "--temperature", str(v_temp),
        "--learnable_tau", str(v_learn_tau),
        # Early stopping
        "--early_stopping_patience",   str(early_stopping_patience),
        "--early_stopping_min_epochs", str(early_stopping_min_epochs),
        "--early_stopping_min_delta",  str(early_stopping_min_delta),
        "--early_stopping_mode",       str(early_stopping_mode),
        "--early_stopping_restore_best", str(early_stopping_restore_best),
        # ReduceLROnPlateau
        "--use_reduce_lr",      str(use_reduce_lr),
        "--reduce_lr_factor",   str(reduce_lr_factor),
        "--reduce_lr_patience", str(reduce_lr_patience),
        "--reduce_lr_min",      str(reduce_lr_min),
        # DAMPS  --  THIS is where the variant differs from (d)
        "--damps_apc",            str(v_apc),                 # <-- NEW: 0
        "--damps_avrf",           str(v_avrf),                # 0
        "--damps_imcf",           str(damps_imcf),            # 1
        "--damps_permutation_fft", str(damps_permutation_fft),
        "--damps_soft_routing",   str(damps_soft_routing),
        "--damps_momentum",       str(damps_momentum),
        "--damps_data_driven_prior", str(damps_data_driven_prior),
        "--damps_num_categories", str(damps_num_categories),
        "--damps_warmup_epochs",  str(damps_warmup_epochs),
        # Pattern B' rebuild
        "--rebuild_R",      str(rebuild_R),
        "--faiss_threshold", str(faiss_threshold),
        "--faiss_use_gpu",   str(faiss_use_gpu),
        "--knn_chunk_size",  str(knn_chunk_size),
        "--knn_efsearch",    str(knn_efsearch),
        # Mixed precision + compile
        "--use_amp",                str(use_amp),
        "--use_torch_compile",      str(use_torch_compile),
        "--torch_compile_mode",     str(torch_compile_mode),
        "--torch_compile_dynamic",  str(torch_compile_dynamic),
        # W&B
        "--use_wandb",      "1",
        "--wandb_project",  wandb_project,
        "--wandb_entity",   wandb_entity,
        "--wandb_run_name", f"phase1_{variant_name}_seed_{seed}",
        "--ablation_target", f"phase1_{variant_name}",
    ]

    print(f"Command: {' '.join(cmd)}")
    print(f"Current directory: {os.getcwd()}\n")

    env: dict[str, str] = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"]  = "1"

    t0: float = time.time()
    result: subprocess.CompletedProcess[str] = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=env,
        encoding="utf-8",
        errors="replace",
    )
    wall: float = time.time() - t0

    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"\n[WARNING] variant '{variant_name}' seed={seed} exited with "
              f"code {result.returncode}")
        if result.stderr:
            print("\n[ERROR OUTPUT]:")
            print(result.stderr)
    else:
        print(f"\n[OK] variant '{variant_name}' seed={seed} completed in "
              f"{wall/60:.1f} min")

    # -----------------------------------------------------------------
    # Parse the per-run log (mirrors cell-16 logic; note apc=0 in name).
    # -----------------------------------------------------------------
    path_name: str = (
        f"damps_uu_ii={User_layers}_{Item_layers}"
        f"_{user_loss_ratio}_{item_loss_ratio}"
        f"_topk={topk}_t={v_temp}_taulearn={v_learn_tau}_R={rebuild_R}"
        f"_apc={v_apc}_avrf={v_avrf}_imcf={damps_imcf}"
        f"_regs={regs}_dim={embed_size}_seed={seed}_"
        f"phase1_{variant_name}"
    )
    log_file: str = f"../{dataset}/{path_name}/{path_name}.txt"

    if not os.path.exists(log_file):
        print(f"[ERROR] log file missing: {log_file}")
        continue

    with open(log_file, "r", encoding="utf-8") as f:
        log_content: str = f.read()

    def _grab(rx: re.Pattern[str], cast=float) -> Any:
        m = rx.search(log_content)
        return cast(m.group(1)) if m else None

    rec: dict[str, Any] = {
        "seed":              seed,
        "variant":           variant_name,
        "log_file":          log_file,
        "test_recall@20":    _grab(_re_test_r20),
        "test_ndcg@20":      _grab(_re_test_n20),
        "test_precision@20": _grab(_re_test_p20),
        "val_recall@20":     _grab(_re_val_r20),
        "val_ndcg@20":       _grab(_re_val_n20),
        "val_recall_peak_epoch": _grab(_re_val_rpeak, int),
        "val_ndcg_peak_epoch":   _grab(_re_val_npeak, int),
        "wall_minutes":      round(wall / 60.0, 2),
    }
    _apc_off_results.append(rec)
    print(f"  parsed: R@20={rec['test_recall@20']:.6f}  "
          f"NDCG@20={rec['test_ndcg@20']:.6f}  "
          f"(val peak epochs: R={rec['val_recall_peak_epoch']}, "
          f"N={rec['val_ndcg_peak_epoch']})")

# ---------------------------------------------------------------------------
# Merge into the global variant store for the Â§4 summary + Â§4.5 t-tests.
# ---------------------------------------------------------------------------
if "all_variant_results" not in globals():
    all_variant_results = {}
all_variant_results[variant_name] = _apc_off_results

# Also extend the variant catalogue so the Â§4 pretty-printer has the label.
phase1_variants[variant_name] = {
    "temperature":     v_temp,
    "learnable_tau":   v_learn_tau,
    "damps_avrf":      v_avrf,
    "damps_apc":       v_apc,
    "label":           variant_label,
}

# ---------------------------------------------------------------------------
# In-cell quick verdict (full stop-gate decision lives in Â§4.5)
# ---------------------------------------------------------------------------
import numpy as np  # imported lazily; cell 16 already imports it globally
_r = np.array([r["test_recall@20"] for r in _apc_off_results
               if r["test_recall@20"] is not None], dtype=float)
_n = np.array([r["test_ndcg@20"]  for r in _apc_off_results
               if r["test_ndcg@20"]  is not None], dtype=float)

print("\n" + "=" * 80)
print(f"VARIANT '{variant_name}' -- 5-seed summary (preliminary)")
print("=" * 80)
print(f"  test_Recall@20 : mean={_r.mean():.6f}  std={_r.std(ddof=1):.6f}  "
      f"min={_r.min():.6f}  max={_r.max():.6f}  values={_r.round(4).tolist()}")
print(f"  test_NDCG@20   : mean={_n.mean():.6f}  std={_n.std(ddof=1):.6f}  "
      f"min={_n.min():.6f}  max={_n.max():.6f}  values={_n.round(4).tolist()}")

_paper_r20: float = 0.0881
_gate_r20:  float = 0.0870
print(f"\n  vs MMHCL paper (R@20={_paper_r20}) : "
      f"delta = {_r.mean() - _paper_r20:+.4f}  "
      f"({(_r.mean()/_paper_r20 - 1) * 100:+.1f}%)")
print(f"  rev44 stop-gate (R@20 >= {_gate_r20}) : "
      f"{'PASS' if _r.mean() >= _gate_r20 else 'FAIL'}")

print("\nNext step:")
print("  Run the **Â§ 4 Results Summary** code cell (next section), then")
print("  the **Â§ 4.5 Bonferroni** code cell â€” `apc_off_combined` is now in")
print("  `all_variant_results` / `phase1_variants` (5 variants Ã— 5 seeds).")


[apc_off_combined] standalone: generated 5 random seeds (base_seed=2117451092)
  seeds = [507362491, 1569402867, 1818287788, 26293619, 41344138]

################################################################################
# VARIANT 'apc_off_combined' -- (e) static tau + AVRF off + APC OFF  (ROUND-2 Phase 1 candidate)
#   --temperature 0.3 --learnable_tau 0 --damps_avrf 0 --damps_apc 0
################################################################################

  seeds          : [507362491, 1569402867, 1818287788, 26293619, 41344138]
  runs           : 5  (single-variant 5-seed validation)
  expected wall  : ~3 hours on RTX 5090 (bfloat16 + torch.compile)
  stop-gate      : mean(R@20) >= 0.0870 to PASS Phase 1

RUN 1/5  (variant 'apc_off_combined' seed=507362491)
Command: c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe train.py --dataset Clothing --gpu_id 0 --seed 507362491 --epoch 250 --verbose 5 --batch_size 1024 --lr 0.0001 --regs 0.001 --clip_grad_norm 1.0 --embed_siz

: 

## 4. Results Summary & Statistics

> **Reading this section: TEST vs VAL.** The headline numbers (`recall@20`, `precision@20`, `ndcg@20`) are computed on the **test split** and snapshotted at the epoch where `val_recall@20` (resp. `val_ndcg@20`) peaked. They match `BEST_Test_*` in the per-run text logs and `best_test_*` in the WandB run summary.
>
> The `val_recall@20` / `val_ndcg@20` columns are the **validation peaks** used only for early stopping. They match `BEST_Val_*` in the text logs and `best_val_*` in WandB, AND are exactly the maxima of the WandB `val/recall@20` / `val/ndcg@20` charts.
>
> **Validation recall/NDCG are typically higher than the test values on Amazon Clothing.** That is normal. The two are computed on different user splits â€” do *not* read off a chart maximum and compare it to the headline number.
>
> If you saw a peak `0.086098` in `recall@20.csv` from WandB and the program reported `0.077679`, then (a) the chart you looked at was `val/recall@20` (validation), (b) the program output was `BEST_Test_Recall@20` (test), and very likely (c) the two come from different *variants* of the four-variant sweep â€” every seed runs four times. To match a chart point with a headline number, check `wandb.summary.best_val_recall_peak_epoch` (added in rev44 Phase 1) and the `val_recall@20` column below.

### Run order (Round-2 / Day-1 verdict)

Execute **after** the **Round-2 `apc_off_combined`** code cell (five runs on the **same seeds** as Â§3) completes. This cell writes `../Clothing/phase1_<variant>_summary.json` for each populated variant, including **`phase1_apc_off_combined_summary.json`**. Then run **Â§4.5**. For variant **(e)**, **PASS Phase 1** if mean **test R@20 â‰¥ 0.0870** (and NDCG per Â§5.2 â€” see `Phase1_Day1_Final_Verdict.tex`). **If metrics still fail**, run **`build_meta_categories.py`**, install **`meta_categories.npy`**, keep **`--damps_apc 1`** on **`combined`**, and **repeat** the four-variant sweep.

In [6]:
from __future__ import annotations

from typing import Any
import numpy as np
import json
import os

# ---------------------------------------------------------------------------
#  Aggregate DAMPS-MMHCL multi-seed results, per Phase 1 variant.
#
#  ``all_variant_results`` is populated by the previous cell as a dict
#  keyed by variant_name (anchor / tau03 / avrf_off / combined). We emit
#  per-variant JSON summaries (consumed by the next cell's paired t-test)
#  AND a global combined summary file.
#
#  IMPORTANT -- which split each metric is computed on:
#    * ``recall@20``, ``precision@20``, ``ndcg@20``  ARE THE TEST metrics
#      (snapshotted at the epoch where val_recall@20 / val_ndcg@20 peaked).
#      They match BEST_Test_* in the per-run text logs and ``best_test_*``
#      in the WandB run summary.
#    * ``val_recall@20``, ``val_ndcg@20`` are the VALIDATION peaks (used
#      for early-stopping). They match BEST_Val_* in the text logs and
#      ``best_val_*`` in the WandB summary, AND the maxima of the
#      ``val/recall@20`` / ``val/ndcg@20`` charts in WandB.
#    * Validation recall/NDCG are typically HIGHER than the test values
#      on Amazon Clothing -- so do NOT confuse a chart maximum with the
#      headline number; the headline reports test, not val.
# ---------------------------------------------------------------------------
metrics: list[str] = ['recall@20', 'precision@20', 'ndcg@20']
val_metrics: list[str] = ['val_recall@20', 'val_ndcg@20']
diag_keys: list[str] = ['tau_final', 'tanh_sat_final', 'last_nnz', 'last_avg_deg']

# Paper anchor values (rev44 footers Section 5.2).
paper_values: dict[str, float] = {
    'recall@20': 0.0881,
    'precision@20': 0.0045,
    'ndcg@20': 0.0394,
}

# Per-variant aggregated stats {variant_name: {metric: {mean, std, values}}}
variant_stats: dict[str, dict[str, dict[str, Any]]] = {}
variant_diag_summary: dict[str, dict[str, dict[str, Any]]] = {}
variant_summary_paths: dict[str, str] = {}

if not all_variant_results or all(len(v) == 0 for v in all_variant_results.values()):
    print("\n[WARNING] No results were extracted from any DAMPS-MMHCL training run!")
    print("Please check the log files under MMHCL_DAMPS_Project/../<dataset>/ manually.")
else:
    for variant_name, run_list in all_variant_results.items():
        if not run_list:
            print(f"\n[skip] variant '{variant_name}' has no successful runs.")
            continue

        v_label: str = phase1_variants[variant_name]['label']
        print(f"\n{'=' * 80}")
        print(
            f"DAMPS-MMHCL RESULTS -- variant '{variant_name}'  {v_label}  "
            f"(n={len(run_list)})"
        )
        print(f"{'=' * 80}")
        print(f"  TEST metrics (headline -- snapshotted at val-recall / val-ndcg peak):")

        stats: dict[str, dict[str, Any]] = {}
        for metric in metrics:
            raw_values: list[Any] = [r.get(metric) for r in run_list]
            values: list[float] = [float(v) for v in raw_values if v is not None]
            if not values:
                stats[metric] = {'mean': float('nan'), 'std': float('nan'), 'values': []}
                print(f"    {metric.upper()}: no values available")
                continue
            mean_val = float(np.mean(values))
            std_val = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            stats[metric] = {'mean': mean_val, 'std': std_val, 'values': values}
            print(
                f"    test_{metric:<14s}: mean={mean_val:.6f}  std={std_val:.6f}  "
                f"values=[{', '.join(f'{v:.4f}' for v in values)}]"
            )
        variant_stats[variant_name] = stats

        # Also surface the validation peaks used for early stopping, so
        # the reviewer can sanity-check the gap with the WandB val/* charts.
        print(f"  VAL peaks (early-stop signal -- NOT the headline):")
        for vmetric in val_metrics:
            raw_v: list[Any] = [r.get(vmetric) for r in run_list]
            v_values: list[float] = [float(v) for v in raw_v if v is not None]
            if not v_values:
                print(f"    {vmetric.upper():18s}: no values available")
                continue
            v_mean = float(np.mean(v_values))
            v_std = float(np.std(v_values, ddof=1)) if len(v_values) > 1 else 0.0
            stats[vmetric] = {'mean': v_mean, 'std': v_std, 'values': v_values}
            print(
                f"    {vmetric:<18s}: mean={v_mean:.6f}  std={v_std:.6f}  "
                f"values=[{', '.join(f'{v:.4f}' for v in v_values)}]"
            )

        # Per-variant DAMPS diagnostics
        diag_summary: dict[str, dict[str, Any]] = {}
        for k in diag_keys:
            raw_d: list[Any] = [r.get(k) for r in run_list]
            clean: list[float] = [float(v) for v in raw_d if v is not None]
            if clean:
                diag_summary[k] = {
                    'mean': float(np.mean(clean)),
                    'std':  float(np.std(clean, ddof=1)) if len(clean) > 1 else 0.0,
                    'values': clean,
                }
        variant_diag_summary[variant_name] = diag_summary
        if diag_summary:
            for k, vd in diag_summary.items():
                print(
                    f"  diag {k:18s}: mean={vd['mean']:.4f}  std={vd['std']:.4f}"
                )

        # Comparison with paper anchor
        print(f"  ----- vs MMHCL paper (Recall@20=0.0881, NDCG@20=0.0394) -----")
        for metric in metrics:
            if not stats[metric]['values']:
                continue
            ours = stats[metric]['mean']
            pv = paper_values[metric]
            pct = (ours - pv) / pv * 100.0
            status = "[OK]" if abs(pct) < 10 else "[CHECK]"
            print(f"    {metric.upper():12s}: ours={ours:.4f}  paper={pv:.4f}  ({pct:+.1f}%) {status}")

        # ----- Persist per-variant JSON summary (consumed by t-test cell) ----
        ov = phase1_variants[variant_name]
        v_summary_path: str = f"../{dataset}/phase1_{variant_name}_summary.json"
        v_summary_data: dict[str, Any] = {
            'variant': variant_name,
            'label': v_label,
            'overrides': {
                'temperature': float(ov['temperature']),
                'learnable_tau': int(ov['learnable_tau']),
                'damps_avrf': int(ov['damps_avrf']),
            },
            'n_runs': len(run_list),
            'model': 'DAMPS-MMHCL (rev44 / Revision 11 -- Phase 1)',
            'seeds': [r['seed'] for r in run_list],
            'statistics': {
                m: {
                    'mean': float(stats[m]['mean']),
                    'std':  float(stats[m]['std']),
                    'values': list(stats[m]['values']),
                } for m in metrics
            },
            'damps_diagnostics': diag_summary,
            'individual_results': run_list,
        }
        with open(v_summary_path, 'w', encoding='utf-8') as f:
            json.dump(v_summary_data, f, indent=2)
        variant_summary_paths[variant_name] = v_summary_path
        print(f"  [OK] per-variant JSON saved -> {v_summary_path}")

    # ---- Global combined summary (legacy file name kept for back-compat) ---
    print(f"\n{'=' * 80}")
    print("rev44 PHASE 1 -- ALL VARIANTS (mean +/- std over seeds per variant)")
    print(f"{'=' * 80}")
    print(
        "  Showing test_*  (headline) AND val_*  (early-stop signal). "
        "These are different splits;"
    )
    print(
        "  do NOT compare a WandB val/recall@20 chart maximum to the headline test_recall@20."
    )
    print()
    print(
        f"  {'Variant':<40s} "
        f"{'test_Recall@20':>16s} {'test_NDCG@20':>14s}  ||  "
        f"{'val_Recall@20':>14s} {'val_NDCG@20':>14s}"
    )
    for variant_name in variants_to_run:
        if variant_name not in variant_stats:
            continue
        s = variant_stats[variant_name]
        rl = phase1_variants[variant_name]['label']
        t_rec = f"{s['recall@20']['mean']:.4f}+-{s['recall@20']['std']:.4f}"
        t_ndc = f"{s['ndcg@20']['mean']:.4f}+-{s['ndcg@20']['std']:.4f}"
        v_rec_s = s.get('val_recall@20')
        v_ndc_s = s.get('val_ndcg@20')
        v_rec = (
            f"{v_rec_s['mean']:.4f}+-{v_rec_s['std']:.4f}"
            if v_rec_s and v_rec_s['values'] else " n/a"
        )
        v_ndc = (
            f"{v_ndc_s['mean']:.4f}+-{v_ndc_s['std']:.4f}"
            if v_ndc_s and v_ndc_s['values'] else " n/a"
        )
        print(
            f"  {rl:<40s} {t_rec:>16s} {t_ndc:>14s}  ||  {v_rec:>14s} {v_ndc:>14s}"
        )

    summary_file: str = f"../{dataset}/multi_seed_summary.json"
    combined_data: dict[str, Any] = {
        'model': 'DAMPS-MMHCL (rev44 / Revision 11 -- Phase 1)',
        'config': {
            'batch_size': batch_size,
            'max_epochs': epoch,
            'lr': lr,
            'embed_size': embed_size,
            'topk': topk,
            'rebuild_R': rebuild_R,
            'use_amp': bool(use_amp),
            'damps_apc': bool(damps_apc),
            'damps_imcf': bool(damps_imcf),
            'damps_soft_routing': bool(damps_soft_routing),
            'damps_momentum': bool(damps_momentum),
            'damps_data_driven_prior': bool(damps_data_driven_prior),
            'damps_num_categories': damps_num_categories,
            'damps_warmup_epochs': damps_warmup_epochs,
            'early_stopping_patience': early_stopping_patience,
            'early_stopping_min_epochs': early_stopping_min_epochs,
            'wandb_project': wandb_project,
        },
        'phase1_variants': phase1_variants,
        'variant_summary_paths': variant_summary_paths,
        'variant_statistics': {
            v: {
                m: {
                    'mean': float(variant_stats[v][m]['mean']),
                    'std':  float(variant_stats[v][m]['std']),
                } for m in metrics
            } for v in variant_stats
        },
        'variant_damps_diagnostics': variant_diag_summary,
        'all_individual_results': all_results,
    }
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, indent=2)
    print(f"\n[OK] combined JSON summary saved -> {summary_file}")

    # ---- Human-readable summary (kept for legacy consumers) --------------
    summary_txt_file: str = f"../{dataset}/multi_seed_summary.txt"
    with open(summary_txt_file, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write(
            f"DAMPS-MMHCL rev44 Phase 1 -- {len(phase1_variants)} variants x {n_runs} seeds "
            f"= {sum(len(r) for r in all_variant_results.values())} runs\n"
        )
        f.write(
            f"Configuration: batch_size={batch_size}, max_epochs={epoch}, "
            f"lr={lr}, rebuild_R={rebuild_R}, use_amp={bool(use_amp)}\n"
        )
        f.write("=" * 80 + "\n\n")
        for variant_name in variants_to_run:
            if variant_name not in variant_stats:
                f.write(f"variant '{variant_name}': no successful runs\n\n")
                continue
            s = variant_stats[variant_name]
            f.write(
                f"variant '{variant_name}'  ({phase1_variants[variant_name]['label']}):\n"
            )
            for m in metrics:
                if not s[m]['values']:
                    continue
                f.write(
                    f"  {m.upper():14s}: mean={s[m]['mean']:.6f}  "
                    f"std={s[m]['std']:.6f}  "
                    f"values={[f'{v:.6f}' for v in s[m]['values']]}\n"
                )
            f.write("\n")
    print(f"[OK] human-readable summary saved -> {summary_txt_file}")

    # Legacy variable expected by the Excel-export cell. We expose the
    # COMBINED stats (mean/std across the recommended variant 'combined'
    # if available, else across all runs) under the old name.
    if 'combined' in variant_stats:
        stats: dict[str, dict[str, Any]] = variant_stats['combined']
        diag_summary = variant_diag_summary.get('combined', {})
    else:
        # Fall back to flat aggregation when only one variant ran.
        stats = {}
        for m in metrics:
            v_all: list[float] = []
            for vstats in variant_stats.values():
                v_all.extend(vstats[m]['values'])
            if v_all:
                stats[m] = {
                    'mean': float(np.mean(v_all)),
                    'std':  float(np.std(v_all, ddof=1)) if len(v_all) > 1 else 0.0,
                    'values': v_all,
                }
            else:
                stats[m] = {'mean': float('nan'), 'std': float('nan'), 'values': []}
        diag_summary = {}


DAMPS-MMHCL RESULTS -- variant 'anchor'  (a) rev42 anchor  (n=5)
  TEST metrics (headline -- snapshotted at val-recall / val-ndcg peak):
    test_recall@20     : mean=0.079493  std=0.002483  values=[0.0827, 0.0814, 0.0766, 0.0782, 0.0786]
    test_precision@20  : mean=0.004427  std=0.000129  values=[0.0046, 0.0045, 0.0043, 0.0044, 0.0044]
    test_ndcg@20       : mean=0.037981  std=0.000897  values=[0.0393, 0.0381, 0.0380, 0.0369, 0.0376]
  VAL peaks (early-stop signal -- NOT the headline):
    val_recall@20     : mean=0.080485  std=0.001898  values=[0.0805, 0.0785, 0.0834, 0.0809, 0.0792]
    val_ndcg@20       : mean=0.037141  std=0.000626  values=[0.0372, 0.0364, 0.0380, 0.0375, 0.0367]
  diag tau_final         : mean=1.0000  std=0.0000
  diag last_avg_deg      : mean=69.2960  std=0.7635
  ----- vs MMHCL paper (Recall@20=0.0881, NDCG@20=0.0394) -----
    RECALL@20   : ours=0.0795  paper=0.0881  (-9.8%) [OK]
    PRECISION@20: ours=0.0044  paper=0.0045  (-1.6%) [OK]
    NDCG@20     : 

## 4.5 rev44 Phase 1 â€” Bonferroni-corrected paired *t*-test + stop-gate

This cell consumes the per-variant JSON summaries written by the previous cell (`../{dataset}/phase1_<variant>_summary.json`) and:

1. Runs **paired *t*-tests** (matched seeds) for each non-anchor variant â€” `tau03`, `avrf_off`, `combined`, and (**after Round-2**) `apc_off_combined` â€” against the rev42 anchor `anchor`, on both `Recall@20` and `NDCG@20`.
2. Applies a **Bonferroni correction** with `N = #non-anchor variants` (**3** before Round-2; **4** once **`apc_off_combined`** summaries exist) so the family-wise Î± stays at 0.05.
3. Reports the **rev44 Â§5.2 quantitative stop-gate verdict** per variant:
   * `Recall@20 â‰¥ 0.0900` â†’ **PAPER VALIDATED** (strictly beats MMHCL paper's 0.0881).
   * `Recall@20 â‰¥ 0.0870 âˆ§ NDCG@20 â‰¥ 0.0390` â†’ **PHASE 1 PASS** (closes coverage gap; advance to Phase 2).
   * Else â†’ **PHASE 1 FAIL** â€” if APC was on **hash-fallback**, run **`build_meta_categories.py`**, restore **`meta_categories.npy`**, rerun **`combined`** with **`--damps_apc 1`**; otherwise re-audit the eval protocol (checklist (iii) sampled vs all-ranking).

The cell is a thin wrapper around `MMHCL_DAMPS_Project/scripts/paired_ttest.py` with `--bonferroni N`. Skip it without affecting downstream cells if you only ran a single variant.

In [7]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np

# ---------------------------------------------------------------------------
#  rev44 / Revision 11 -- Phase 1 Bonferroni-corrected paired t-test cell.
#
#  Pulls per-variant aggregated JSON summaries written by the previous cell,
#  runs paired t-tests vs the anchor with --bonferroni N (where N = # of
#  non-anchor variants present), and prints the rev44 Section 5.2 stop-gate
#  verdict per variant.
# ---------------------------------------------------------------------------

# Make MMHCL_DAMPS_Project/scripts/ importable so we can call the helper
# directly without spawning a subprocess (faster + cleaner traceback).
_DAMPS_DIR = Path(DAMPS_DIR) if 'DAMPS_DIR' in dir() else Path(os.getcwd())
if str(_DAMPS_DIR) not in sys.path:
    sys.path.insert(0, str(_DAMPS_DIR))

try:
    from scripts.paired_ttest import paired_ttest_report
except Exception as exc:                                            # pragma: no cover
    print(f"[ERROR] failed to import scripts.paired_ttest: {exc}")
    print("        Is MMHCL_DAMPS_Project/scripts/paired_ttest.py present?")
    raise

# Stop-gate thresholds (rev44 Â§5.2).
ACCEPTANCE_RECALL: float = 0.0870
ACCEPTANCE_NDCG: float = 0.0390
PAPER_VALIDATION_RECALL: float = 0.0900

ANCHOR_NAME: str = 'anchor'

# Resolve where cell 14 wrote its per-variant summaries
ds_dir: Path = Path(f"../{dataset}").resolve()


def _load_variant(name: str) -> dict[str, Any] | None:
    p = ds_dir / f"phase1_{name}_summary.json"
    if not p.is_file():
        return None
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)


anchor_data: dict[str, Any] | None = _load_variant(ANCHOR_NAME)
if anchor_data is None or not anchor_data['statistics']['recall@20']['values']:
    print(
        f"\n[SKIP] Anchor variant '{ANCHOR_NAME}' has no aggregated "
        f"summary at {ds_dir / 'phase1_anchor_summary.json'}; cannot run "
        f"paired t-tests. (Did you run the training cell with the anchor "
        f"variant included?)"
    )
else:
    other_variants: list[str] = [
        v for v in variants_to_run
        if v != ANCHOR_NAME and v in all_variant_results
        and len(all_variant_results[v]) > 0
    ]
    bonferroni_N: int = max(1, len(other_variants))

    print(f"\n{'=' * 80}")
    print(
        f"rev44 Phase 1 -- Paired t-tests vs '{ANCHOR_NAME}' "
        f"(Bonferroni correction = {bonferroni_N})"
    )
    print(f"{'=' * 80}")

    ttest_report: dict[str, dict[str, dict[str, Any]]] = {}
    for v_name in other_variants:
        v_data = _load_variant(v_name)
        if v_data is None:
            print(f"[skip] missing summary for variant '{v_name}'")
            continue
        ttest_report[v_name] = {}
        for metric in ('recall@20', 'ndcg@20'):
            a_vals: list[float] = list(anchor_data['statistics'][metric]['values'])
            v_vals: list[float] = list(v_data['statistics'][metric]['values'])
            n = min(len(a_vals), len(v_vals))
            if n < 2:
                print(
                    f"[skip] variant '{v_name}' vs anchor on {metric}: "
                    f"only {n} paired observations"
                )
                continue
            print(f"\n----- variant '{v_name}' vs '{ANCHOR_NAME}' on {metric} (n={n}) -----")
            r = paired_ttest_report(
                damps_scores=v_vals[:n],
                baseline_scores=a_vals[:n],
                alpha=0.05,
                bonferroni=bonferroni_N,
                label_a=v_data.get('label', v_name),
                label_b=anchor_data.get('label', ANCHOR_NAME),
            )
            ttest_report[v_name][metric] = r

    # ----- rev44 Section 5.2 stop-gate verdict -----
    print(f"\n{'=' * 80}")
    print("rev44 Section 5.2 stop-gate verdict (per variant)")
    print(f"{'=' * 80}")

    final_verdicts: dict[str, dict[str, Any]] = {}
    for v_name in variants_to_run:
        v_data = _load_variant(v_name)
        if v_data is None or not v_data['statistics']['recall@20']['values']:
            print(f"  variant '{v_name:<10s}' -- no data")
            continue
        rmean = float(v_data['statistics']['recall@20']['mean'])
        nmean = float(v_data['statistics']['ndcg@20']['mean'])
        beat_paper = rmean >= PAPER_VALIDATION_RECALL
        beat_acceptance = (
            rmean >= ACCEPTANCE_RECALL and nmean >= ACCEPTANCE_NDCG
        )
        if beat_paper:
            verdict = "PAPER VALIDATED  (Recall@20 >= 0.0900 -- strictly beats MMHCL paper)"
        elif beat_acceptance:
            verdict = "PHASE 1 PASS     (Recall@20 >= 0.0870 AND NDCG@20 >= 0.0390)"
        else:
            verdict = (
                "PHASE 1 FAIL     (re-audit eval protocol; checklist (iii) "
                "all-ranking vs sampled is the highest-suspicion mismatch)"
            )
        final_verdicts[v_name] = {
            'recall@20_mean': rmean,
            'ndcg@20_mean': nmean,
            'beat_acceptance': bool(beat_acceptance),
            'beat_paper': bool(beat_paper),
            'verdict': verdict,
        }
        print(
            f"  variant '{v_name:<10s}'  Recall@20={rmean:.4f}  "
            f"NDCG@20={nmean:.4f} -> {verdict}"
        )

    # Save the consolidated stop-gate + t-test report so reviewers can
    # inspect it without re-running the notebook.
    final_report_path: Path = ds_dir / "phase1_stopgate_report.json"
    with final_report_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                'stopgate_thresholds': {
                    'acceptance_recall@20': ACCEPTANCE_RECALL,
                    'acceptance_ndcg@20':   ACCEPTANCE_NDCG,
                    'paper_validation_recall@20': PAPER_VALIDATION_RECALL,
                },
                'bonferroni_N': bonferroni_N,
                'paired_ttest': ttest_report,
                'verdicts': final_verdicts,
            },
            f,
            indent=2,
        )
    print(f"\n[OK] consolidated stop-gate report -> {final_report_path}")


rev44 Phase 1 -- Paired t-tests vs 'anchor' (Bonferroni correction = 3)

----- variant 'tau03' vs 'anchor' on recall@20 (n=5) -----
|  Method            |   Mean   |    Std   |   N  |
|--------------------|----------|----------|------|
|  (b) static tau=0.3 only|  0.0829  |  0.0007  |  5   |
|  (a) rev42 anchor  |  0.0795  |  0.0025  |  5   |

Paired t-test : t=2.674, p=0.05557 [n.s.]
Bonferroni(3) -> corrected alpha = 0.05/3 = 0.0166667
98% CI of mean diff = [-0.0016, 0.0085]  (Bonferroni-widened)
alpha = 0.05, alpha_eff = 0.0166667, n = 5

----- variant 'tau03' vs 'anchor' on ndcg@20 (n=5) -----
|  Method            |   Mean   |    Std   |   N  |
|--------------------|----------|----------|------|
|  (b) static tau=0.3 only|  0.0397  |  0.0005  |  5   |
|  (a) rev42 anchor  |  0.0380  |  0.0009  |  5   |

Paired t-test : t=4.306, p=0.01258 [significant]
Bonferroni(3) -> corrected alpha = 0.05/3 = 0.0166667
98% CI of mean diff = [0.0001, 0.0033]  (Bonferroni-widened)
alpha = 0.05, al

## 5. Export Results to Excel

In [8]:
from __future__ import annotations

from typing import Any

import pandas as pd
from openpyxl.styles import Font, PatternFill

if len(all_results) > 0:
    print("=" * 80)
    print("EXPORTING DAMPS-MMHCL RESULTS TO EXCEL")
    print("=" * 80)

    excel_file: str = f"../{dataset}/DAMPS_MMHCL_{dataset}_Results_Local.xlsx"

    # ---- Individual Results (incl. DAMPS diagnostics) -----------------------
    individual_data: list[dict[str, Any]] = []
    for i, r in enumerate(all_results, 1):
        row: dict[str, Any] = {
            'Run': i,
            'Seed': r['seed'],
            'Recall@20': r['recall@20'],
            'Precision@20': r.get('precision@20'),
            'NDCG@20': r['ndcg@20'],
            'tau_final': r.get('tau_final'),
            'tanh_sat_final': r.get('tanh_sat_final'),
            'last_nnz': r.get('last_nnz'),
            'last_avg_deg': r.get('last_avg_deg'),
        }
        individual_data.append(row)

    df_individual: pd.DataFrame = pd.DataFrame(individual_data)

    # ---- Summary statistics --------------------------------------------------
    summary_data_list: list[dict[str, Any]] = []
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        summary_data_list.append({
            'Metric': metric.upper(),
            'Mean': stats[metric]['mean'],
            'Std': stats[metric]['std'],
            'Mean +/- Std': f"{stats[metric]['mean']:.6f} +/- {stats[metric]['std']:.6f}",
        })
    df_summary: pd.DataFrame = pd.DataFrame(summary_data_list)

    # ---- Paper comparison ----------------------------------------------------
    comparison_data: list[dict[str, Any]] = []
    for metric in metrics:
        if not stats[metric]['values']:
            continue
        our_mean: float = stats[metric]['mean']
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100.0
        comparison_data.append({
            'Metric': metric.upper(),
            'DAMPS-MMHCL (Ours)': our_mean,
            'MMHCL Paper': paper_val,
            'Difference (%)': diff_pct,
        })
    df_comparison: pd.DataFrame = pd.DataFrame(comparison_data)

    # ---- DAMPS configuration / diagnostics ----------------------------------
    config_rows: list[dict[str, Any]] = [
        {'Setting': 'Model', 'Value': 'DAMPS-MMHCL (Revision 9)'},
        {'Setting': 'Dataset', 'Value': dataset},
        {'Setting': 'Batch size', 'Value': batch_size},
        {'Setting': 'Max epochs', 'Value': epoch},
        {'Setting': 'Learning rate', 'Value': lr},
        {'Setting': 'Embedding dim', 'Value': embed_size},
        {'Setting': 'Top-k (KNN)', 'Value': topk},
        {'Setting': 'Temperature init', 'Value': temperature},
        {'Setting': 'Rebuild R (Pattern B\')', 'Value': rebuild_R},
        {'Setting': 'use_amp (bf16)', 'Value': bool(use_amp)},
        {'Setting': 'damps_apc', 'Value': bool(damps_apc)},
        {'Setting': 'damps_avrf', 'Value': bool(damps_avrf)},
        {'Setting': 'damps_imcf', 'Value': bool(damps_imcf)},
        {'Setting': 'damps_soft_routing', 'Value': bool(damps_soft_routing)},
        {'Setting': 'damps_momentum', 'Value': bool(damps_momentum)},
        {'Setting': 'damps_data_driven_prior', 'Value': bool(damps_data_driven_prior)},
        {'Setting': 'damps_num_categories', 'Value': damps_num_categories},
        {'Setting': 'damps_warmup_epochs', 'Value': damps_warmup_epochs},
        {'Setting': 'early_stopping_patience', 'Value': early_stopping_patience},
        {'Setting': 'early_stopping_min_epochs', 'Value': early_stopping_min_epochs},
        {'Setting': 'W&B project', 'Value': wandb_project},
    ]
    df_config: pd.DataFrame = pd.DataFrame(config_rows)

    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df_individual.to_excel(writer, sheet_name='Individual Results', index=False)
        df_summary.to_excel(writer, sheet_name='Summary Statistics', index=False)
        df_comparison.to_excel(writer, sheet_name='Paper Comparison', index=False)
        df_config.to_excel(writer, sheet_name='DAMPS Config', index=False)

        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            for col in ws.columns:
                max_length: int = 0
                column: str = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except Exception:
                        pass
                ws.column_dimensions[column].width = max_length + 2

            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

    print(f"\n[OK] Results exported to: {excel_file}")
    print(f"  - Sheet 1: Individual Results ({len(all_results)} runs)")
    print(f"  - Sheet 2: Summary Statistics")
    print(f"  - Sheet 3: Paper Comparison")
    print(f"  - Sheet 4: DAMPS Config")
else:
    print("No results to export.")

EXPORTING DAMPS-MMHCL RESULTS TO EXCEL

[OK] Results exported to: ../Clothing/DAMPS_MMHCL_Clothing_Results_Local.xlsx
  - Sheet 1: Individual Results (20 runs)
  - Sheet 2: Summary Statistics
  - Sheet 3: Paper Comparison
  - Sheet 4: DAMPS Config


## 5.5 Permutation-FFT Falsifiability Ablation (Optional)

This is the falsifiability test from **Specification Section 6, Item 8** (and compliance check **INFO 4** of `DAMPS_compliance_check_report.tex`). It re-runs the full DAMPS-MMHCL pipeline twice per seed:

- **Arm A** â€” `--damps_permutation_fft 0` (standard 1-D rFFT, the default)
- **Arm B** â€” `--damps_permutation_fft 1` (input is permuted with a fixed random permutation before FFT)

After both arms have completed, a paired t-test is performed on Recall@20 and NDCG@20. The spec's fallback rule:

| `|gap|` (Recall@20) | Decision |
|---|---|
| `< 1 %` | Switch the spectral basis to **DCT-II** (the 1-D FFT axis is not informative). |
| `>= 1 %` | The standard 1-D FFT is **validated** and remains the default. |

Skip this section unless you are preparing the final paper / camera-ready ablation table â€” each pair adds two full training runs.

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess

# Toggle to True when you want to actually run the falsifiability test.
RUN_PERMUTATION_FFT_ABLATION: bool = False

if not RUN_PERMUTATION_FFT_ABLATION:
    print("[SKIP] Permutation-FFT ablation disabled (set RUN_PERMUTATION_FFT_ABLATION = True to run).")
else:
    # ``DAMPS_DIR`` was defined in the environment-setup cell (cell 2) and
    # is also the cwd that ``train.py`` expects (so its ``../{dataset}/...``
    # paths resolve to the workspace root).
    if os.path.normpath(os.getcwd()) != os.path.normpath(DAMPS_DIR):
        os.chdir(DAMPS_DIR)
    if DAMPS_DIR not in sys.path:
        sys.path.insert(0, DAMPS_DIR)

    PYTHON_EXE: str = sys.executable
    perm_seeds: list[int] = [42, 43, 44]   # 3 paired seeds is enough for INFO 4
    perm_dataset: str = 'Clothing'
    perm_epoch: int = 250

    common_flags: list[str] = [
        '--dataset', perm_dataset,
        '--epoch', str(perm_epoch),
        '--damps_apc', '1', '--damps_avrf', '1', '--damps_imcf', '1',
        '--damps_soft_routing', '1', '--damps_momentum', '1',
        '--damps_data_driven_prior', '1', '--use_amp', '1',
    ]

    for seed in perm_seeds:
        for perm_on, tag_prefix in [(0, 'perm_fft_off'), (1, 'perm_fft_on')]:
            tag: str = f"{tag_prefix}_seed{seed}"
            print("=" * 70)
            print(f"[ablation] {tag}")
            print("=" * 70)
            cmd: list[str] = [
                PYTHON_EXE, 'train.py',
                '--seed', str(seed),
                '--ablation_target', tag,
                '--damps_permutation_fft', str(perm_on),
            ] + common_flags
            env = os.environ.copy()
            env['PYTHONIOENCODING'] = 'utf-8'
            env['PYTHONUNBUFFERED'] = '1'
            r = subprocess.run(
                cmd, capture_output=True, text=True, env=env,
                encoding='utf-8', errors='replace',
            )
            if r.stdout:
                print(r.stdout[-2000:])
            if r.returncode != 0:
                print(f"[FAIL] {tag} exited with code {r.returncode}")
                if r.stderr:
                    print(r.stderr[-2000:])

    # ----- Aggregate the paired runs and run the t-test --------------------
    aggregator: str = os.path.join(DAMPS_DIR, 'scripts', '_aggregate_permutation_fft.py')
    if os.path.exists(aggregator):
        print("=" * 70)
        print("Aggregating Permutation-FFT runs and running paired t-test...")
        print("=" * 70)
        agg_cmd: list[str] = [
            PYTHON_EXE, aggregator,
            '--dataset', perm_dataset,
            '--seeds', *[str(s) for s in perm_seeds],
        ]
        env = os.environ.copy()
        env['PYTHONIOENCODING'] = 'utf-8'
        r = subprocess.run(agg_cmd, capture_output=True, text=True, env=env,
                           encoding='utf-8', errors='replace', cwd=DAMPS_DIR)
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    else:
        print(f"[WARN] Aggregator script not found at {aggregator}.")

## 6. Check W&B Run Status (Optional)

In [ ]:
from __future__ import annotations

import wandb

print(f"Checking DAMPS-MMHCL runs for {wandb_entity}/{wandb_project} ...")

try:
    api: wandb.Api = wandb.Api()
    runs: wandb.apis.public.Runs = api.runs(f"{wandb_entity}/{wandb_project}")

    if len(runs) == 0:
        print("No runs found in this project yet.")
    else:
        print(f"\nFound {len(runs)} runs. Latest 5 status:")
        print("-" * 70)
        for run in runs[:5]:
            print(f"Run Name : {run.name}")
            print(f"Status   : {run.state}")
            print(f"Created  : {run.created_at}")
            # Standard MMHCL/DAMPS keys logged by ``train.py``
            for key in (
                'val/recall@20',
                'val/ndcg@20',
                'best_test_recall@20',
                'best_test_precision@20',
                'best_test_ndcg@20',
                'damps/tau',
                'damps/tanh_sat',
                'rebuild/nnz',
                'rebuild/avg_deg',
            ):
                if key in run.summary:
                    val = run.summary[key]
                    try:
                        print(f"  {key:28s}: {float(val):.4f}")
                    except (TypeError, ValueError):
                        print(f"  {key:28s}: {val}")
            print("-" * 70)
except Exception as e:
    print(f"Error fetching run status: {e}")
    print("Make sure W&B is logged in correctly.")

## Troubleshooting (DAMPS-MMHCL)

### Common Issues

1. **CUDA Out of Memory** â€” even with bf16 AMP enabled
   ```python
   batch_size = 512   # or 256
   knn_chunk_size = 2048  # tile the KNN distance matrix more aggressively
   ```
   You can also disable the Slim Momentum Encoder (`damps_momentum=0`) to free up to ~2 GB.

2. **`bfloat16 not supported`** â€” pre-Ampere GPUs cannot run the bf16 path
   ```python
   use_amp = 0   # falls back to fp32
   ```

3. **`faiss-gpu` not installed** â€” only required for `N >= 60_000` items
   - Default Clothing dataset has 23 033 items, so the chunked PyTorch path is used automatically.
   - Install via `pip install faiss-gpu` (Linux) or `conda install -c pytorch faiss-gpu` if you switch to a larger dataset.

4. **W&B login issues** â€” run `wandb login` in a terminal first

5. **Missing modules** â€” re-run the *Install Dependencies* cell after restarting the kernel

6. **Data / wrong Clothing dump** — this notebook requires the HF MMHCL **LATTICE 39k** release
   ([Xu-SII-BNU/MMHCL Clothing](https://huggingface.co/datasets/Xu-SII-BNU/MMHCL/tree/main/Clothing)):
   - Splits: `data/Clothing/5-core/{train,val,test}.json` → **39,387 users / 23,033 items**
   - Features: `data/Clothing/image_feat.npy`, `data/Clothing/text_feat.npy` (next to `5-core/`, not inside it)
   - After replacing the dataset, delete graph caches under `data/Clothing/5-core/`
     (`UI_mat_sym.pth`, `User_mat_rw.pth`, `hypergraph_mat_*_topk_10.pth`) or re-run the
     **§2.7 Dataset integrity** cell so they rebuild for the new catalogue.

7. **Permutation-FFT ablation** â€” to run the falsifiability check:
   ```python
   damps_permutation_fft = 1
   ```
   The DAMPS-MMHCL spec predicts a measurable performance drop when this is enabled.

### DAMPS Diagnostic Sanity-Checks (look for these in logs)
- `tanh_sat` should stabilise in `[0.1, 0.6]` after warm-up â€” values near 1.0 mean AVRF logits are saturated.
- `nnz` and `avg_deg` per rebuild should plateau, not grow unboundedly (Pattern B' is working).
- `tau` (learnable InfoNCE temperature) starts at `0.1` (spec Section 3.1) and is allowed to drift; healthy runs keep it in roughly `[0.05, 0.30]` after warm-up.

### Expected Results
- **Original MMHCL (paper)** â€” Recall@20 = 0.0881, Precision@20 = 0.0045, NDCG@20 = 0.0394
- **DAMPS-MMHCL** â€” see the JSON / Excel summary produced by Step 4 / Step 5.


## 7. Wave 1 / M1.5 Sweep Driver â€” LogQ Popularity Correction

Run the **M1.5 milestone sweep** from *DAMPS-MMHCL Phase 2 Section 8.2*.  
This cell sweeps `logq_scale` Ã— `logq_mode` Ã— `seed` and evaluates the acceptance gate:

| Gate condition | Value |
|---|---|
| Mean Recall@20 | â‰¥ 0.0925 |
| Min Recall@20 per seed | â‰¥ 0.0890 (rollback gate) |
| NDCG@20 | must not regress vs. rev45 (0.0416) |

**Pre-flight**: Run cell 16 (Section 3 multi-seed training setup) first to define `DAMPS_DIR`, `PYTHON_EXE`, `dataset`, and `seeds`. The cell self-bootstraps if those variables are absent.

Results are saved as `m15_logq_sweep_runs.json` and `m15_logq_sweep_summary.json` under `../<dataset>/`.

In [ ]:
# =====================================================================
# Section 8.2 -- Wave 1 / M1.5 sweep driver  (FIXED for rev54)
#   Matrix:  logq_scale x logq_mode x seed  -> calls train.py per config
#   Gate:    mean BEST_Test_Recall@20 >= 0.0925  AND  every seed >= 0.0890
#
# FIXES vs the previous notebook revision
#   (1) Real LogQ activation: --enable_logq 1 + --logq_mode/--logq_beta/
#       --logq_scale/--logq_clip are now ACTUALLY appended to every cmd.
#   (2) Seeds are coerced via int(round(float(s))) before being passed as
#       --seed, eliminating the "seed=1.404632241e+09" float-truncation
#       reproducibility bug.
#   (3) BEST_Test_* (snapshot at the val-recall peak epoch) is the only
#       metric reported for the M1.5 gate, NOT mid-run epoch printouts.
#   (4) ablation_target now encodes (mode, scale, seed) so per-run log
#       directories do not collide across the sweep matrix.
#   (5) early_stopping_patience lowered from 30 -> 20 to terminate cleanly
#       after convergence (optional; flip BACK_TO_PATIENCE_30 = True to
#       reproduce the previous behaviour exactly).
# =====================================================================
from __future__ import annotations

import os
import re
import sys
import json
import time
import random
import subprocess
from typing import Any

import wandb

# ---- 0. Resolve project dirs (reuse earlier-cell globals if present) ----
try:
    _DAMPS_DIR = DAMPS_DIR
except NameError:
    _DAMPS_DIR = os.getcwd()
try:
    _PYTHON = PYTHON_EXE
except NameError:
    _PYTHON = sys.executable
try:
    _DATASET = dataset
except NameError:
    _DATASET = "Clothing"
try:
    _WB_PROJECT = wandb_project
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity
except NameError:
    _WB_ENTITY = ""

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---- 1. Sweep matrix definition --------------------------------------
LOGQ_SCALES: list[float] = [0.05, 0.1, 0.3, 1.0]   # spec-mandated scale sweep
LOGQ_MODES:  list[str]   = ["laplace"]              # add "raw","sqrt" -> 24 runs
N_SEEDS: int = 5                                    # 4 scales x 1 mode x 5 = 20 runs
LOGQ_BETA: float = 1.0
LOGQ_CLIP: float = 5.0

# M1.5 acceptance gate (rev54 Section 8.2 / Section 6)
GATE_MEAN_RECALL: float = 0.0925
GATE_MIN_RECALL:  float = 0.0890

# Optional: revert patience to 30 to reproduce the previous notebook exactly
BACK_TO_PATIENCE_30: bool = False
PATIENCE: int = 30 if BACK_TO_PATIENCE_30 else 20
MIN_EPOCHS: int = 75

# ---- 1a. Seed coercion: ALWAYS int (fixes the 1.40e+09 float bug) ----
def _as_int_seed(s: Any) -> int:
    """Tolerate floats / strings / numpy scalars; emit a clean Python int."""
    return int(round(float(s)))

try:
    SEEDS = [_as_int_seed(s) for s in list(seeds)[:N_SEEDS]]   # type: ignore[name-defined]  # noqa: F821
    if len(SEEDS) < N_SEEDS:
        raise NameError
except NameError:
    random.seed(2026)
    SEEDS = [random.randint(1, 2_147_483_646) for _ in range(N_SEEDS)]
print(f"[M1.5] seeds (int) = {SEEDS}")
print(f"[M1.5] patience    = {PATIENCE}  (BACK_TO_PATIENCE_30={BACK_TO_PATIENCE_30})")
print(f"[M1.5] W&B project = {_WB_PROJECT!r}  entity={_WB_ENTITY!r}")

# ---- 2. rev45 apc_off_combined backbone flags (only delta = LogQ) -----
# Identical to the Round-2 apc_off_combined cell, except early_stopping
# patience is lowered to 20 by default.
BACKBONE_FLAGS: list[str] = [
    "--dataset",                     str(_DATASET),
    "--gpu_id",                      "0",
    "--epoch",                       "250",
    "--verbose",                     "5",
    "--batch_size",                  "1024",
    "--lr",                          "0.0001",
    "--regs",                        "0.001",
    "--clip_grad_norm",              "1.0",
    "--embed_size",                  "64",
    "--topk",                        "5",
    "--core",                        "5",
    "--User_layers",                 "3",
    "--Item_layers",                 "2",
    "--user_loss_ratio",             "0.03",
    "--item_loss_ratio",             "0.07",
    "--temperature",                 "0.3",
    "--learnable_tau",               "0",
    "--early_stopping_patience",     str(PATIENCE),
    "--early_stopping_min_epochs",   str(MIN_EPOCHS),
    "--early_stopping_min_delta",    "0.0001",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    "--use_reduce_lr",               "1",
    "--reduce_lr_factor",            "0.5",
    "--reduce_lr_patience",          "3",
    "--reduce_lr_min",               "1e-06",
    # apc_off_combined backbone
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_permutation_fft",       "0",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_num_categories",        "10",
    "--damps_warmup_epochs",         "10",
    "--rebuild_R",                   "5",
    "--faiss_threshold",             "60000",
    "--faiss_use_gpu",               "1",
    "--knn_chunk_size",              "4096",
    "--knn_efsearch",                "64",
    "--use_amp",                     "1",
]

# ---- 3. Metric parser: prefer BEST_Test_* (val-peak snapshot) --------
# train.py emits:
#   BEST_Val_Recall@10 / @20
#   BEST_Val_Recall_Peak_Epoch
#   BEST_Val_NDCG@10  / @20
#   BEST_Val_NDCG_Peak_Epoch
#   BEST_Test_Recall@20 / Precision@20 / NDCG@20    <-- gate uses these
_S = r"[^0-9\-]*(-?[0-9]*\.?[0-9]+)"
_PAT = {
    "recall@20":            re.compile(r"BEST_Test_Recall@20"      + _S),
    "ndcg@20":              re.compile(r"BEST_Test_NDCG@20"        + _S),
    "precision@20":         re.compile(r"BEST_Test_Precision@20"   + _S),
    "val_recall@20":        re.compile(r"BEST_Val_Recall@20"       + _S),
    "val_ndcg@20":          re.compile(r"BEST_Val_NDCG@20"         + _S),
    "val_recall_peak_epoch":re.compile(r"BEST_Val_Recall_Peak_Epoch" + _S),
    "val_ndcg_peak_epoch":  re.compile(r"BEST_Val_NDCG_Peak_Epoch"   + _S),
}

def _parse_metrics(text: str) -> dict[str, float]:
    out: dict[str, float] = {}
    for k, pat in _PAT.items():
        hits = pat.findall(text)
        if hits:
            out[k] = float(hits[-1])      # last write = final best snapshot
    return out

# ---- 4. Run one (mode, scale, seed) configuration --------------------
def run_one(mode: str, scale: float, seed: int) -> dict[str, Any]:
    """Launch one train.py subprocess; returns a result dict."""
    seed_int = int(seed)
    # Encode mode/scale/seed into the ablation_target so per-run log
    # directories (../<dataset>/damps_..._seed=<seed>_<ablation_target>/)
    # do not collide across sweep configurations.
    abl_tag = f"m15_logq_{mode}_s{scale}_seed{seed_int}"
    run_name = abl_tag
    cmd: list[str] = [
        _PYTHON, "train.py",
        *BACKBONE_FLAGS,
        "--seed",         str(seed_int),     # <-- ALWAYS int, never 1.40e+09
        # ---- Wave 1 LogQ activation (the only delta vs rev45) ----
        "--enable_logq",  "1",
        "--logq_mode",    str(mode),
        "--logq_beta",    str(LOGQ_BETA),
        "--logq_scale",   str(scale),
        "--logq_clip",    str(LOGQ_CLIP),
        # ---- W&B per-run logging ----
        "--use_wandb",       "1",
        "--wandb_project",   _WB_PROJECT,
        "--wandb_entity",    _WB_ENTITY,
        "--wandb_run_name",  run_name,
        "--ablation_target", abl_tag,
    ]
    t0 = time.time()
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.run(
        cmd, capture_output=True, text=True, cwd=_DAMPS_DIR, env=env
    )
    dt = time.time() - t0
    log = (proc.stdout or "") + "\n" + (proc.stderr or "")
    m = _parse_metrics(log)
    ok = ("recall@20" in m) and (proc.returncode == 0)
    if not ok:
        print(f"  [WARN] mode={mode} scale={scale} seed={seed_int} "
              f"rc={proc.returncode}\n{log[-400:]}")
    return {
        "mode":                  mode,
        "scale":                 scale,
        "seed":                  seed_int,
        "run_name":              run_name,
        "ablation_target":       abl_tag,
        "recall@20":             m.get("recall@20"),
        "ndcg@20":               m.get("ndcg@20"),
        "precision@20":          m.get("precision@20"),
        "val_recall@20":         m.get("val_recall@20"),
        "val_ndcg@20":           m.get("val_ndcg@20"),
        "val_recall_peak_epoch": int(m["val_recall_peak_epoch"])
                                  if "val_recall_peak_epoch" in m else None,
        "val_ndcg_peak_epoch":   int(m["val_ndcg_peak_epoch"])
                                  if "val_ndcg_peak_epoch" in m else None,
        "runtime_s":             round(dt, 1),
        "ok":                    ok,
    }

# ---- 5. Execute the full matrix --------------------------------------
all_runs: list[dict[str, Any]] = []
total = len(LOGQ_MODES) * len(LOGQ_SCALES) * len(SEEDS)
idx = 0
print("=" * 72)
print(f"M1.5 SWEEP: {total} runs "
      f"({len(LOGQ_MODES)} mode x {len(LOGQ_SCALES)} scale x {len(SEEDS)} seed)")
print("=" * 72)
for mode in LOGQ_MODES:
    for scale in LOGQ_SCALES:
        for seed in SEEDS:
            idx += 1
            print(f"[{idx}/{total}] mode={mode} scale={scale} "
                  f"seed={int(seed)} ...", flush=True)
            all_runs.append(run_one(mode, scale, seed))

# ---- 6. Aggregate per (mode, scale) and evaluate the gate ------------
def _mean(xs): return sum(xs) / len(xs) if xs else float("nan")
def _std(xs):
    if len(xs) < 2:
        return 0.0
    m = _mean(xs)
    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5

summary: list[dict[str, Any]] = []
for mode in LOGQ_MODES:
    for scale in LOGQ_SCALES:
        rs = [r["recall@20"] for r in all_runs
              if r["mode"] == mode and r["scale"] == scale
              and r["recall@20"] is not None]
        ns = [r["ndcg@20"] for r in all_runs
              if r["mode"] == mode and r["scale"] == scale
              and r["ndcg@20"] is not None]
        if not rs:
            continue
        mean_r, min_r = _mean(rs), min(rs)
        passed = (mean_r >= GATE_MEAN_RECALL) and (min_r >= GATE_MIN_RECALL)
        summary.append({
            "mode":        mode,
            "scale":       scale,
            "n":           len(rs),
            "recall_mean": round(mean_r, 4),
            "recall_std":  round(_std(rs), 4),
            "recall_min":  round(min_r, 4),
            "ndcg_mean":   round(_mean(ns), 4),
            "ndcg_std":    round(_std(ns), 4),
            "M1.5_pass":   passed,
        })

# ---- 7. Console report -----------------------------------------------
print("\n" + "=" * 72)
print("M1.5 GATE REPORT  (mean BEST_Test_R@20 >= 0.0925  AND  min >= 0.0890)")
print("=" * 72)
hdr = (f"{'mode':<9}{'scale':>7}{'n':>4}{'R@20 mean':>12}{'+/-':>9}"
       f"{'R@20 min':>11}{'NDCG mean':>11}{'PASS':>7}")
print(hdr)
print("-" * len(hdr))
for s in summary:
    print(f"{s['mode']:<9}{s['scale']:>7}{s['n']:>4}"
          f"{s['recall_mean']:>12.4f}{s['recall_std']:>9.4f}"
          f"{s['recall_min']:>11.4f}{s['ndcg_mean']:>11.4f}"
          f"{('YES' if s['M1.5_pass'] else 'no'):>7}")

any_pass = any(s["M1.5_pass"] for s in summary)
verdict = ("PASS" if any_pass else "FAIL")
print("\n[M1.5 VERDICT]",
      "PASS -- at least one (mode,scale) cleared the gate; "
      "promote that config to Wave 2 (M1 / SimGCL)."
      if any_pass else
      "FAIL -- no (mode,scale) cleared the gate; "
      "inspect logq_scale/clip before proceeding.")

# ---- 8. Persist raw + summary JSON for audit trail -------------------
out_dir = os.path.abspath(os.path.join("..", _DATASET))
os.makedirs(out_dir, exist_ok=True)
runs_path    = os.path.join(out_dir, "m15_logq_sweep_runs.json")
summary_path = os.path.join(out_dir, "m15_logq_sweep_summary.json")
with open(runs_path, "w", encoding="utf-8") as f:
    json.dump(all_runs, f, indent=2)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved -> {runs_path}")
print(f"Saved -> {summary_path}")

# ---- 9. W&B sweep-level summary run ----------------------------------
print("\n[W&B] Logging sweep-level summary run ...")
wb_run = wandb.init(
    project=_WB_PROJECT,
    entity=_WB_ENTITY if _WB_ENTITY else None,
    name="m15_logq_sweep_summary",
    group="wave1_m15_logq",
    tags=["wave1", "m15", "logq", "sweep_summary"],
    job_type="sweep_summary",
    config={
        "phase":            "wave1_m15",
        "logq_modes":       LOGQ_MODES,
        "logq_scales":      LOGQ_SCALES,
        "logq_beta":        LOGQ_BETA,
        "logq_clip":        LOGQ_CLIP,
        "seeds":            SEEDS,
        "n_seeds":          N_SEEDS,
        "total_runs":       total,
        "gate_mean_recall": GATE_MEAN_RECALL,
        "gate_min_recall":  GATE_MIN_RECALL,
        "dataset":          _DATASET,
        "backbone":         "apc_off_combined",
        "temperature":      0.3,
        "patience":         PATIENCE,
    },
    reinit=True,
)
cols = ["mode", "scale", "seed", "run_name",
        "recall@20", "ndcg@20", "precision@20",
        "val_recall@20", "val_ndcg@20",
        "val_recall_peak_epoch", "val_ndcg_peak_epoch",
        "runtime_s", "ok"]
runs_table = wandb.Table(columns=cols)
for r in all_runs:
    runs_table.add_data(*[r.get(c) for c in cols])
wb_run.log({"m15/all_runs": runs_table})

agg_cols = ["mode", "scale", "n",
            "recall_mean", "recall_std", "recall_min",
            "ndcg_mean", "ndcg_std", "M1.5_pass"]
agg_table = wandb.Table(columns=agg_cols)
for s in summary:
    agg_table.add_data(*[s.get(c) for c in agg_cols])
    tag = f"{s['mode']}_s{s['scale']}"
    wb_run.log({
        f"m15/{tag}/recall_mean": s["recall_mean"],
        f"m15/{tag}/recall_std":  s["recall_std"],
        f"m15/{tag}/recall_min":  s["recall_min"],
        f"m15/{tag}/ndcg_mean":   s["ndcg_mean"],
        f"m15/{tag}/gate_pass":   int(s["M1.5_pass"]),
    })
wb_run.log({"m15/aggregate_table": agg_table})

best = max(
    summary,
    key=lambda s: (s["recall_mean"], -s["recall_std"]),
    default=None,
)
wb_run.summary.update({
    "m15/gate_passed":      any_pass,
    "m15/verdict":          verdict,
    "m15/best_recall_mean": (best["recall_mean"] if best else None),
    "m15/best_scale":       (best["scale"]       if best else None),
    "m15/best_mode":        (best["mode"]        if best else None),
    "m15/n_configs_passed": sum(1 for s in summary if s["M1.5_pass"]),
    "m15/n_configs_total":  len(summary),
})

artifact = wandb.Artifact(
    name=f"m15_logq_sweep_{_DATASET}",
    type="sweep_results",
    description="Raw per-run results + per-(mode,scale) aggregates for M1.5 LogQ sweep.",
    metadata={"dataset": _DATASET, "verdict": verdict},
)
artifact.add_file(runs_path,    name="m15_logq_sweep_runs.json")
artifact.add_file(summary_path, name="m15_logq_sweep_summary.json")
wb_run.log_artifact(artifact)
wb_run.finish()
print(f"[W&B] Sweep summary run logged: {wb_run.url}")

## 8. Wave 1 / M1.5 Sweep â€” Post-hoc Analysis

Read-only cell: inspects log artefacts produced by the fixed sweep cell above.

**What it does:**
1. Globs every `damps_*_m15_logq_*` directory under `../<dataset>/` and parses each `.txt` log
2. Extracts `BEST_Test_Recall@20`, `BEST_Val_Recall_Peak_Epoch`, last epoch reached, and per-epoch val/recall@20 curves
3. Builds a per-run audit table; flags any run whose `(last_epoch - peak_epoch) < 15` (potential premature stop)
4. Re-evaluates the M1.5 gate from parsed logs as a cross-check
5. Plots val/recall@20 vs epoch for each `logq_scale` group with vertical dotted lines at each run's recall peak
6. Saves `m15_logq_val_recall_curves.png` and `m15_logq_parsed_logs.json` under `../<dataset>/`

Run this cell immediately after the sweep cell finishes (same kernel).

In [ ]:
# ---- auto-install missing dependencies --------------------------------
import importlib, subprocess, sys
_missing = [pkg for pkg in ["matplotlib", "numpy"] if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *_missing])
# ------------------------------------------------------------------------

# =====================================================================
# Section 8.2 -- Wave 1 / M1.5 sweep ANALYSIS  (post-hoc, read-only)
#   For each (mode, scale, seed) directory under ../<dataset>/, extract:
#       * BEST_Test_Recall@20 / NDCG@20 / Precision@20  (val-peak snapshot)
#       * BEST_Val_Recall@20 + Peak_Epoch                (early-stop proof)
#       * Full per-epoch val/recall@20 curve             (early-stop visual)
#   Plot val-recall curves so you can visually confirm NO run was cut
#   short of its recall peak.
#
# Required upstream artifacts (produced by the fixed sweep cell):
#   ../<dataset>/m15_logq_sweep_runs.json       (optional cross-check)
#   ../<dataset>/damps_..._seed=<seed>_m15_logq_<mode>_s<scale>_seed<seed>/
#       *.txt                                  (per-epoch logger output)
# =====================================================================
from __future__ import annotations

import os
import re
import json
import glob
import math
from typing import Any, Optional

import numpy as np
import matplotlib.pyplot as plt

# ---- 0. Resolve dirs --------------------------------------------------
try:
    _DAMPS_DIR = DAMPS_DIR
except NameError:
    _DAMPS_DIR = os.getcwd()
try:
    _DATASET = dataset
except NameError:
    _DATASET = "Clothing"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

LOG_ROOT = os.path.abspath(os.path.join("..", _DATASET))
print(f"[M1.5/analysis] scanning {LOG_ROOT}")

# ---- 1. Discover run directories from ablation_target naming ---------
# Pattern produced by the fixed sweep cell:
#   damps_..._seed=<seed>_m15_logq_<mode>_s<scale>_seed<seed>/<same>.txt
DIR_RE = re.compile(
    r"damps_.*?_seed=(?P<seed>\d+)_m15_logq_(?P<mode>[a-zA-Z0-9]+)_s(?P<scale>[0-9.]+)_seed\d+$"
)

run_dirs: list[dict[str, Any]] = []
for d in sorted(glob.glob(os.path.join(LOG_ROOT, "damps_*"))):
    if not os.path.isdir(d):
        continue
    name = os.path.basename(d.rstrip("/"))
    m = DIR_RE.match(name)
    if not m:
        continue
    txt_files = glob.glob(os.path.join(d, "*.txt"))
    if not txt_files:
        continue
    run_dirs.append({
        "dir":   d,
        "txt":   txt_files[0],
        "mode":  m.group("mode"),
        "scale": float(m.group("scale")),
        "seed":  int(m.group("seed")),
    })
print(f"[M1.5/analysis] found {len(run_dirs)} M1.5 LogQ run directories")
if not run_dirs:
    print("  (no logs yet -- run the fixed sweep cell first)")

# ---- 2. Regex parsers -------------------------------------------------
_S = r"[^0-9\-]*(-?[0-9]*\.?[0-9]+)"
PAT_BEST = {
    "test_recall@20":        re.compile(r"BEST_Test_Recall@20"        + _S),
    "test_ndcg@20":          re.compile(r"BEST_Test_NDCG@20"          + _S),
    "test_precision@20":     re.compile(r"BEST_Test_Precision@20"     + _S),
    "val_recall@20":         re.compile(r"BEST_Val_Recall@20"         + _S),
    "val_ndcg@20":           re.compile(r"BEST_Val_NDCG@20"           + _S),
    "val_recall_peak_epoch": re.compile(r"BEST_Val_Recall_Peak_Epoch" + _S),
    "val_ndcg_peak_epoch":   re.compile(r"BEST_Val_NDCG_Peak_Epoch"   + _S),
}
# Per-epoch line from train.py Trainer.train():
#   Epoch 17 [12.3s + 4.5s]: loss=0.12345  recall@10=0.04210  recall@20=0.07815  ndcg@20=0.03812
PAT_EPOCH = re.compile(
    r"Epoch\s+(\d+)\s*\[[^\]]+\]:\s*loss=([\d.]+)\s+"
    r"recall@\d+=([\d.]+)\s+recall@(\d+)=([\d.]+)\s+ndcg@\d+=([\d.]+)"
)
# Early-stop marker emitted by train.py
PAT_ES_TRIGGER = re.compile(r"#####\s*Early stop triggered\s*#####")


def parse_log(path: str) -> dict[str, Any]:
    """Return BEST_* snapshot, per-epoch curves, and early-stop flag."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()
    best: dict[str, float] = {}
    for k, pat in PAT_BEST.items():
        hits = pat.findall(text)
        if hits:
            best[k] = float(hits[-1])
    epochs:      list[int]   = []
    val_rec20:   list[float] = []
    val_ndcg20:  list[float] = []
    losses:      list[float] = []
    for m in PAT_EPOCH.finditer(text):
        epochs.append(int(m.group(1)))
        losses.append(float(m.group(2)))
        val_rec20.append(float(m.group(5)))
        val_ndcg20.append(float(m.group(6)))
    return {
        "best":       best,
        "epochs":     epochs,
        "loss":       losses,
        "val_rec20":  val_rec20,
        "val_ndcg20": val_ndcg20,
        "early_stop_triggered": bool(PAT_ES_TRIGGER.search(text)),
        "last_epoch": (epochs[-1] if epochs else None),
    }


# ---- 3. Parse all logs -----------------------------------------------
rows: list[dict[str, Any]] = []
for r in run_dirs:
    p = parse_log(r["txt"])
    best = p["best"]
    peak = int(best.get("val_recall_peak_epoch", -1))
    last = p["last_epoch"]
    margin = (last - peak) if (last is not None and peak >= 0) else None
    rows.append({
        "mode":             r["mode"],
        "scale":            r["scale"],
        "seed":             r["seed"],
        "BEST_Test_R@20":   best.get("test_recall@20"),
        "BEST_Test_NDCG@20":best.get("test_ndcg@20"),
        "BEST_Val_R@20":    best.get("val_recall@20"),
        "val_peak_epoch":   peak if peak >= 0 else None,
        "last_epoch":       last,
        "epochs_past_peak": margin,
        "early_stop":       p["early_stop_triggered"],
        "n_val_points":     len(p["val_rec20"]),
        "_curve":           p,
    })

# ---- 4. Tabular report (BEST_Test + early-stop margin) ---------------
print("\n" + "=" * 92)
print("M1.5 per-run audit  (sorted by mode, scale, seed)")
print("=" * 92)
hdr = (f"{'mode':<9}{'scale':>7}{'seed':>12}{'BEST_Test_R@20':>17}"
       f"{'BEST_Val_R@20':>16}{'peak_ep':>9}{'last_ep':>9}"
       f"{'margin':>8}{'ES?':>5}")
print(hdr); print("-" * len(hdr))
for x in sorted(rows, key=lambda r: (r["mode"], r["scale"], r["seed"])):
    print(f"{x['mode']:<9}{x['scale']:>7}{x['seed']:>12}"
          f"{(x['BEST_Test_R@20'] or float('nan')):>17.4f}"
          f"{(x['BEST_Val_R@20']  or float('nan')):>16.4f}"
          f"{(x['val_peak_epoch'] if x['val_peak_epoch'] is not None else -1):>9}"
          f"{(x['last_epoch']     if x['last_epoch']     is not None else -1):>9}"
          f"{(x['epochs_past_peak'] if x['epochs_past_peak'] is not None else -1):>8}"
          f"{('Y' if x['early_stop'] else 'n'):>5}")

# Flag suspicious cases: peak too close to last epoch (< PATIENCE)
SUSPECT_MARGIN: int = 15
suspects = [
    x for x in rows
    if x["epochs_past_peak"] is not None and x["epochs_past_peak"] < SUSPECT_MARGIN
]
if suspects:
    print(f"\n[WARN] {len(suspects)} run(s) ended within {SUSPECT_MARGIN} epochs of "
          f"their val-recall peak -- inspect curves to rule out premature stop:")
    for x in suspects:
        print(f"  mode={x['mode']} scale={x['scale']} seed={x['seed']} "
              f"peak={x['val_peak_epoch']} last={x['last_epoch']} "
              f"margin={x['epochs_past_peak']}")
else:
    print(f"\n[OK] all runs ran >= {SUSPECT_MARGIN} epochs past their val-recall peak.")

# ---- 5. Per-(mode, scale) aggregate using BEST_Test only --------------
def _agg(vals: list[Optional[float]]) -> tuple[float, float, float]:
    v = [x for x in vals if x is not None]
    if not v:
        return float("nan"), float("nan"), float("nan")
    m = sum(v) / len(v)
    s = math.sqrt(sum((x - m) ** 2 for x in v) / (len(v) - 1)) if len(v) >= 2 else 0.0
    return m, s, min(v)

print("\n" + "=" * 92)
print("M1.5 aggregate from PARSED LOGS  (BEST_Test_Recall@20 only)")
print("=" * 92)
key = lambda r: (r["mode"], r["scale"])
configs = sorted(set(key(r) for r in rows))
print(f"{'mode':<9}{'scale':>7}{'n':>4}{'R@20 mean':>12}{'+/-':>9}"
      f"{'R@20 min':>11}{'NDCG mean':>11}{'PASS':>7}")
print("-" * 64)
GATE_MEAN: float = 0.0925
GATE_MIN:  float = 0.0890
log_summary: list[dict[str, Any]] = []
for mode, scale in configs:
    sub = [r for r in rows if r["mode"] == mode and r["scale"] == scale]
    rm, rs, mn = _agg([r["BEST_Test_R@20"] for r in sub])
    nm, _, _  = _agg([r["BEST_Test_NDCG@20"] for r in sub])
    passed = (rm >= GATE_MEAN) and (mn >= GATE_MIN)
    log_summary.append({
        "mode": mode, "scale": scale, "n": len(sub),
        "recall_mean": round(rm, 4), "recall_std": round(rs, 4),
        "recall_min":  round(mn, 4), "ndcg_mean":   round(nm, 4),
        "M1.5_pass":   bool(passed),
    })
    print(f"{mode:<9}{scale:>7}{len(sub):>4}{rm:>12.4f}{rs:>9.4f}"
          f"{mn:>11.4f}{nm:>11.4f}{('YES' if passed else 'no'):>7}")

# ---- 6. Visual confirmation: val-recall@20 curves --------------------
if rows:
    by_scale: dict[float, list[dict[str, Any]]] = {}
    for r in rows:
        by_scale.setdefault(r["scale"], []).append(r)
    n_scales = len(by_scale)
    ncols = min(2, n_scales)
    nrows = math.ceil(n_scales / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(7.0 * ncols, 4.2 * nrows),
                             squeeze=False)
    for ax_idx, (scale, group) in enumerate(sorted(by_scale.items())):
        ax = axes[ax_idx // ncols][ax_idx % ncols]
        for x in sorted(group, key=lambda r: r["seed"]):
            curve = x["_curve"]
            if not curve["epochs"]:
                continue
            ax.plot(curve["epochs"], curve["val_rec20"],
                    linewidth=1.4, alpha=0.85,
                    label=f"seed={x['seed']} (peak ep={x['val_peak_epoch']}, "
                          f"BEST_Test={(x['BEST_Test_R@20'] or 0):.4f})")
            if x["val_peak_epoch"] is not None:
                ax.axvline(x["val_peak_epoch"], color="grey",
                           linestyle=":", linewidth=0.7, alpha=0.6)
        ax.axhline(GATE_MIN, color="red", linestyle="--",
                   linewidth=0.9, alpha=0.7, label=f"rollback {GATE_MIN}")
        ax.axhline(GATE_MEAN, color="green", linestyle="--",
                   linewidth=0.9, alpha=0.7, label=f"M1.5 gate {GATE_MEAN}")
        ax.set_title(f"M1.5 LogQ -- scale={scale}")
        ax.set_xlabel("epoch")
        ax.set_ylabel("val/recall@20")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7, loc="lower right")
    # blank out any spare subplot
    for j in range(len(by_scale), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    fig.suptitle("M1.5 LogQ sweep -- val/recall@20 vs epoch  "
                 "(dotted = val-recall peak; restore_best=1 means best is recovered)",
                 y=1.02, fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig_path = os.path.join(LOG_ROOT, "m15_logq_val_recall_curves.png")
    plt.savefig(fig_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"[plot] saved -> {fig_path}")

# ---- 7. Persist parsed analysis to JSON ------------------------------
parsed_path = os.path.join(LOG_ROOT, "m15_logq_parsed_logs.json")
to_dump = []
for r in rows:
    rr = {k: v for k, v in r.items() if k != "_curve"}
    rr["epochs"]     = r["_curve"]["epochs"]
    rr["val_rec20"]  = r["_curve"]["val_rec20"]
    rr["val_ndcg20"] = r["_curve"]["val_ndcg20"]
    to_dump.append(rr)
with open(parsed_path, "w", encoding="utf-8") as f:
    json.dump({"runs": to_dump, "summary": log_summary}, f, indent=2)
print(f"[json] saved -> {parsed_path}")

## 9. Branch A — LogQ + SimGCL batch-N (Rev55 §8.1)

Implements the **Branch A** surgical fix from *DAMPS-MMHCL Phase 2 Revision 55 §8.1*.

Branch A answers: *Does the Wave 2 slowdown stem from the SimGCL algorithm or from the implementation?*

### Key changes vs Wave 2 audit

| Bottleneck | Legacy Wave 2 | Branch A (this section) |
|---|---|---|
| InfoNCE denominator | (B, N) matmul — N = 23 033 items | **(B, B) batch-N** — K = B−1 = 2 047 |
| Memory / chunk | ≈ 1.5 GB FP32 | **≈ 64 MB** |
| FLOPs / chunk | ≈ 12.1 GFLOPs | **≈ 0.54 GFLOPs (~22×)** |
| View propagation | Every epoch | **Every k = 2 epochs** (view cache) |

**Frozen Wave-1 constraints:** `logq_scale=1.0`, `mode=laplace`, `beta=1.0`, `clip=5.0`, `patience=20`, 5 seeds, backbone `apc_off_combined`.

**lambda_view = 0.05 LOCKED** (no sweep — the 3-value sweep was a Rev54 copy-paste error, see Rev55 §8.1).

**Acceptance window:** R@20 ∈ [0.0900, 0.0945] on every seed.
**Rollback gate:** any seed < 0.0890 → revert to Wave 1 LogQ-only.

**Run order:** §9.0 pre-flight → §9.1 sweep (5 seeds × ~6 h ≈ 30 h GPU) → §9.2 post-hoc analysis.

In [ ]:
# =====================================================================
# Section 8.0 -- Branch A pre-flight  (rev55 Â§8.1, mandatory before S4)
#
#   S1  pytest tests/test_simgcl.py -x          (~5 min)
#   S2  verify Branch A CLI defaults            (~10 s)
#   S3  optional 20-epoch smoke                 (set RUN_SMOKE=True)
# =====================================================================
from __future__ import annotations

import os
import subprocess
import sys

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd
        if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )

if _DAMPS_DIR not in sys.path:
    sys.path.insert(0, _DAMPS_DIR)
if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

RUN_SMOKE: bool = False
SMOKE_EPOCHS: int = 20
SMOKE_SEED: int = 0

print("=" * 72)
print("Branch A pre-flight (rev55 Â§8.1)")
print(f"  cwd = {os.getcwd()}")
print("=" * 72)

# ---- S1: SimGCL unit tests ---------------------------------------------
print("\n[S1] pytest tests/test_simgcl.py -x ...")
_env = os.environ.copy()
_env["PYTHONIOENCODING"] = "utf-8"
_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_simgcl.py", "-x", "-q"],
    cwd=_DAMPS_DIR,
    capture_output=True,
    text=True,
    env=_env,
)
print(_proc.stdout or "")
if _proc.returncode != 0:
    print(_proc.stderr or "")
    raise RuntimeError(
        "S1 FAILED: SimGCL unit tests must pass before Branch A sweep. "
        "Recheck branchA_simgcl_batchN.py and model.py patches."
    )
print("[S1] PASS")

# ---- S2: parser defaults -----------------------------------------------
print("\n[S2] verify Branch A CLI defaults ...")
_argv_bak = sys.argv[:]
try:
    sys.argv = ["train.py", "--dataset", "clothing"]
    from utility.parser import parse_args

    _args = parse_args()
finally:
    sys.argv = _argv_bak

_expected = (2, 1, 2048, 2048)
_got = (
    _args.branchA_view_every_k,
    _args.branchA_bcl_batchn,
    _args.branchA_view_bsz,
    _args.branchA_bcl_bsz,
)
print(f"  got      = {_got}")
print(f"  expected = {_expected}")
if _got != _expected:
    raise RuntimeError(
        f"S2 FAILED: Branch A parser defaults mismatch {_got} != {_expected}"
    )
print("[S2] PASS")

# ---- S3: optional short smoke ------------------------------------------
if RUN_SMOKE:
    print(
        f"\n[S3] 20-epoch Branch A smoke (seed={SMOKE_SEED}) ..."
    )
    _smoke = subprocess.run(
        [
            sys.executable,
            "train.py",
            "--dataset",
            "Clothing",
            "--seed",
            str(SMOKE_SEED),
            "--epoch",
            str(SMOKE_EPOCHS),
            "--batch_size",
            "4096",
            "--enable_logq",
            "1",
            "--logq_scale",
            "1.0",
            "--logq_clip",
            "5.0",
            "--enable_simgcl",
            "1",
            "--lambda_view",
            "0.05",
            "--simgcl_eps",
            "0.1",
            "--branchA_view_every_k",
            "2",
            "--branchA_bcl_batchn",
            "1",
            "--branchA_view_bsz",
            "2048",
            "--branchA_bcl_bsz",
            "2048",
            "--use_amp",
            "1",
            "--damps_apc",
            "0",
            "--damps_avrf",
            "0",
            "--damps_imcf",
            "1",
            "--damps_soft_routing",
            "1",
            "--damps_momentum",
            "1",
            "--temperature",
            "0.3",
            "--learnable_tau",
            "0",
        ],
        cwd=_DAMPS_DIR,
        env=_env,
    )
    if _smoke.returncode != 0:
        raise RuntimeError("S3 FAILED: Branch A smoke run exited non-zero.")
    print("[S3] PASS")
else:
    print("\n[S3] skipped (set RUN_SMOKE=True to enable 20-epoch smoke)")

print("\n" + "=" * 72)
print("[BranchA pre-flight] ALL CHECKS PASSED â€” proceed to Section 8 sweep.")
print("=" * 72)

### 9.05 BPR Optimizer Smoke Comparison — Config A vs Config B

> **Important note (not a bug):** Two optimization settings exist for the Branch A sweep:
>
> | | **Config A (Branch A driver)** | **Config B (Wave 1 baseline)** |
> |---|---|---|
> | atch_size | **4096** | 1024 |
> | lr | **0.001** | 0.0001 |
>
> These values do **NOT** affect K of the batch-N InfoNCE (that is determined solely by
> --branchA_view_bsz / --branchA_bcl_bsz = 2048), but they **do** affect BPR loss stability.
>
> **Recommendation:** Run this smoke cell first (seed=0, 50 epochs each ≈ 30 min total).
> Then update BATCH_SIZE and LR at the top of the §9.1 sweep driver to match the winner.

Results are saved to 
esults/branchA_smoke_config_choice.json.

In [ ]:
# =====================================================================
# Section 9.05 -- BPR optimizer smoke comparison  (batch_size / lr)
#
#   Config A  (Branch A driver default) : batch_size=4096, lr=0.001
#   Config B  (Wave 1 baseline)         : batch_size=1024, lr=0.0001
#
#   Run: seed=0, max 50 epochs, patience=10 (enough to see BPR trend).
#   The config with higher val_recall@20 at convergence wins.
#   Edit BATCH_SIZE / LR at the top of the §9.1 sweep driver to match.
# =====================================================================
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import time
from typing import Any

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DAMPS_DIR = os.getcwd()
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PYTHON = sys.executable
try:
    _DATASET = dataset  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DATASET = "Clothing"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---- Smoke settings (do not change) ----------------------------------
SMOKE_SEED: int    = 0
SMOKE_EPOCHS: int  = 50     # enough to see BPR trend; ~15-25 min each config
SMOKE_PATIENCE: int = 10    # aggressive early-stop for the smoke

# ---- Two BPR configurations under comparison -------------------------
CONFIGS: list[dict[str, Any]] = [
    {"name": "A_bs4096_lr1e-3",  "batch_size": 4096, "lr": 0.001},
    {"name": "B_bs1024_lr1e-4",  "batch_size": 1024, "lr": 0.0001},
]

# ---- Shared Branch A + Wave-1 frozen flags ---------------------------
SHARED_FLAGS: list[str] = [
    "--dataset",                     str(_DATASET),
    "--gpu_id",                      "0",
    "--epoch",                       str(SMOKE_EPOCHS),
    "--verbose",                     "5",
    "--regs",                        "0.001",
    "--clip_grad_norm",              "1.0",
    "--embed_size",                  "64",
    "--topk",                        "5",
    "--core",                        "5",
    "--User_layers",                 "3",
    "--Item_layers",                 "2",
    "--user_loss_ratio",             "0.03",
    "--item_loss_ratio",             "0.07",
    "--temperature",                 "0.3",
    "--learnable_tau",               "0",
    "--early_stopping_patience",     str(SMOKE_PATIENCE),
    "--early_stopping_min_epochs",   "10",
    "--early_stopping_min_delta",    "0.0001",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    "--use_reduce_lr",               "1",
    "--reduce_lr_factor",            "0.5",
    "--reduce_lr_patience",          "3",
    "--reduce_lr_min",               "1e-06",
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_permutation_fft",       "0",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_num_categories",        "10",
    "--damps_warmup_epochs",         "10",
    "--rebuild_R",                   "5",
    "--faiss_threshold",             "60000",
    "--faiss_use_gpu",               "1",
    "--knn_chunk_size",              "4096",
    "--knn_efsearch",                "64",
    "--use_amp",                     "1",
    "--enable_logq",                 "1",
    "--logq_mode",                   "laplace",
    "--logq_beta",                   "1.0",
    "--logq_scale",                  "1.0",
    "--logq_clip",                   "5.0",
    "--enable_simgcl",               "1",
    "--simgcl_eps",                  "0.1",
    "--lambda_view",                 "0.05",
    "--simgcl_batch_size_user",      "2048",
    "--simgcl_batch_size_item",      "2048",
    "--branchA_view_every_k",        "2",
    "--branchA_bcl_batchn",          "1",
    "--branchA_view_bsz",            "2048",
    "--branchA_bcl_bsz",             "2048",
    "--seed",                        str(SMOKE_SEED),
]

_S = r"[^0-9\-]*(-?[0-9]*\.?[0-9]+)"
_PAT = {
    "val_recall@20":   re.compile(r"BEST_Val_Recall@20"         + _S),
    "val_ndcg@20":     re.compile(r"BEST_Val_NDCG@20"           + _S),
    "test_recall@20":  re.compile(r"BEST_Test_Recall@20"        + _S),
    "test_ndcg@20":    re.compile(r"BEST_Test_NDCG@20"          + _S),
    "peak_epoch":      re.compile(r"BEST_Val_Recall_Peak_Epoch" + _S),
}


def _parse(text: str) -> dict[str, float]:
    out: dict[str, float] = {}
    for k, pat in _PAT.items():
        hits = pat.findall(text)
        if hits:
            out[k] = float(hits[-1])
    return out


print("=" * 72)
print(f"BPR optimizer smoke: seed={SMOKE_SEED}, {SMOKE_EPOCHS} epochs, "
      f"patience={SMOKE_PATIENCE}")
print("=" * 72)

results: list[dict[str, Any]] = []
for cfg in CONFIGS:
    print(f"\n[Smoke] Running Config {cfg['name']} ...")
    cmd = [
        _PYTHON, "train.py",
        *SHARED_FLAGS,
        "--batch_size",      str(cfg["batch_size"]),
        "--lr",              str(cfg["lr"]),
        "--ablation_target", f"smoke_cfg_{cfg['name']}",
    ]
    t0 = time.time()
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.run(
        cmd, capture_output=True, text=True, cwd=_DAMPS_DIR, env=env
    )
    dt = time.time() - t0
    log = (proc.stdout or "") + "\n" + (proc.stderr or "")
    m = _parse(log)
    ok = proc.returncode == 0 and bool(m)
    if not ok:
        print(f"  [WARN] rc={proc.returncode}")
        print(log[-400:])
    r = {
        "config":           cfg["name"],
        "batch_size":       cfg["batch_size"],
        "lr":               cfg["lr"],
        "val_recall@20":    m.get("val_recall@20"),
        "val_ndcg@20":      m.get("val_ndcg@20"),
        "test_recall@20":   m.get("test_recall@20"),
        "test_ndcg@20":     m.get("test_ndcg@20"),
        "peak_epoch":       int(m["peak_epoch"]) if "peak_epoch" in m else None,
        "runtime_min":      round(dt / 60.0, 1),
        "ok":               ok,
    }
    results.append(r)
    print(f"  val_R@20={r['val_recall@20']}  val_N@20={r['val_ndcg@20']}  "
          f"peak_epoch={r['peak_epoch']}  runtime={r['runtime_min']} min")

# ---- Pick winner -------------------------------------------------------
valid = [r for r in results if r["ok"] and r["val_recall@20"] is not None]
if len(valid) == 2:
    winner = max(valid, key=lambda r: (r["val_recall@20"] or 0.0))
    loser  = [r for r in valid if r["config"] != winner["config"]][0]
    delta  = (winner["val_recall@20"] or 0.0) - (loser["val_recall@20"] or 0.0)
    print()
    print("=" * 72)
    print(f"WINNER : {winner['config']}")
    print(f"  val_R@20 = {winner['val_recall@20']:.6f}  "
          f"(delta vs loser: {delta:+.6f})")
    print()
    print(f"ACTION : update §9.1 sweep driver BACKBONE_FLAGS:")
    print(f"   '--batch_size', '{winner['batch_size']}',")
    print(f"   '--lr',         '{winner['lr']}',")
    if delta < 0.001:
        print()
        print("NOTE: delta < 0.001 — configs are statistically equivalent "
              "at 50 epochs.\n"
              "      Prefer Config A (batch_size=4096, lr=0.001) for speed "
              "(fewer gradient steps per epoch).")
    print("=" * 72)
    recommendation = winner["config"]
else:
    print("\n[Smoke] One or both runs failed — inspect output above.")
    recommendation = "unknown"

# ---- Persist -----------------------------------------------------------
os.makedirs("results", exist_ok=True)
out_json = os.path.join("results", "branchA_smoke_config_choice.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(
        {"seed": SMOKE_SEED, "epochs": SMOKE_EPOCHS,
         "results": results, "recommendation": recommendation},
        f, indent=2,
    )
print(f"\n[Smoke] saved -> {out_json}")

### 9.1 Branch A Sweep Driver — LogQ + SimGCL batch-N

Runs **5 seeds** with `lambda_view = 0.05` **LOCKED** (no sweep).  
All Branch A flags are passed explicitly: `--branchA_view_every_k 2`, `--branchA_bcl_batchn 1`, `--branchA_view_bsz 2048`, `--branchA_bcl_bsz 2048`.

**BPR optimizer** — set `BATCH_SIZE` / `LR` at the top of the cell based on the §9.05 smoke winner.  
Default: Config A (`batch_size=4096`, `lr=0.001`).

### W&B tracking

| W&B run | What it tracks |
|---|---|
| **Per-seed run** (`wandb_group='wave2_branchA'`) | Per-epoch: `train/loss`, `val/recall@20`, `val/ndcg@20`, `loss/bpr`, `loss/nce`, `loss/simgcl_view` — streamed live |
| **Controller run** (`branchA_sweep_controller`) | Rolling mean/std R@20 after each seed, final W&B Table with all 5 seeds, gate verdict, JSON artifact |

All runs share **group** `wave2_branchA` and **tags** `branchA, batchN, rev55, logq`.  
Each seed additionally gets tag `seed<N>` for easy filtering.

**Expected runtime:** ~6 h / seed → **~30 h total**.

**Gate (Rev55 §8.1):** PASS iff R@20 ∈ [0.0900, 0.0945] on **every** seed.  
FAIL (any seed < 0.0890) → roll back: `--enable_simgcl 0 --branchA_bcl_batchn 0`.

In [ ]:
# =====================================================================
# Section 9.1 -- Branch A sweep driver  (LogQ + SimGCL batch-N, rev55)
#
#   Matrix : seed-only  (lambda_view = 0.05 LOCKED)
#   Locked : logq_scale=1.0 / laplace / beta=1.0 / clip=5.0
#   Branch A overlays : view_every_k=2 / bcl_batchn=1 / bsz=2048
#
#   W&B logging
#   -----------
#   Each seed sub-process gets its own W&B run (per-epoch metrics
#   streamed live via --use_wandb 1). The sweep driver additionally
#   maintains a "controller" run that logs:
#     * per-seed summary metrics immediately after each seed finishes
#     * rolling mean / std R@20 after every completed seed
#     * a W&B Table with all 5 seed results
#     * gate verdict and per-seed bar chart in the final summary
#     * a W&B Artifact containing branchA_sweep_results.json
#
#   Gate (rev55 §8.1):
#     PASS  -> every seed R@20 in [0.0900, 0.0945]
#     FAIL  -> any seed < 0.0890 -> roll back to Wave 1 LogQ-only
# =====================================================================
from __future__ import annotations

import json
import math
import os
import re
import subprocess
import sys
import time
import random
from typing import Any

import wandb

# ---- 0. Resolve dirs / globals (reuse from earlier cells if set) ------
try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DAMPS_DIR = os.getcwd()
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PYTHON = sys.executable
try:
    _DATASET = dataset  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DATASET = "Clothing"
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_ENTITY = ""

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---- 1. Hyperparameters (edit here, not inside BACKBONE_FLAGS) --------
LAMBDA_VIEW_LOCK: float = 0.05     # LOCKED — do not sweep
SIMGCL_EPS: float = 0.1
N_SEEDS: int = 5
LOGQ_SCALE_LOCK: float = 1.0
LOGQ_MODE_LOCK: str = "laplace"
LOGQ_BETA_LOCK: float = 1.0
LOGQ_CLIP_LOCK: float = 5.0
VIEW_EVERY_K: int = 2
USE_BCL_BATCHN: int = 1
BRANCHA_VIEW_BSZ: int = 2048
BRANCHA_BCL_BSZ: int = 2048

# BPR optimizer — set based on §9.05 smoke comparison winner.
# Default: Config A (batch_size=4096, lr=0.001 — Branch A driver default).
# Change to 1024 / 0.0001 if §9.05 showed Config B wins.
BATCH_SIZE: int = 4096
LR: float = 0.001

BACK_TO_PATIENCE_30: bool = False
PATIENCE: int = 30 if BACK_TO_PATIENCE_30 else 20
MIN_EPOCHS: int = 75

GATE_MIN_RECALL: float = 0.0900
GATE_MAX_RECALL: float = 0.0945
GATE_FAIL_BELOW: float = 0.0890

# ---- 1a. W&B group + tags for per-seed runs --------------------------
WB_GROUP: str = "wave2_branchA"
WB_TAGS: str = "branchA,batchN,rev55,logq"      # comma-separated

# ---- 1b. Seed list ---------------------------------------------------
def _as_int_seed(s: Any) -> int:
    return int(round(float(s)))

try:
    SEEDS = [_as_int_seed(s) for s in list(seeds)[:N_SEEDS]]  # type: ignore[name-defined]  # noqa: F821
    if len(SEEDS) < N_SEEDS:
        raise NameError
except NameError:
    random.seed(2026)
    SEEDS = [random.randint(1, 2_147_483_646) for _ in range(N_SEEDS)]

print(f"[BranchA] seeds         = {SEEDS}")
print(f"[BranchA] lambda_view   = {LAMBDA_VIEW_LOCK}  (LOCKED)")
print(f"[BranchA] batch_size    = {BATCH_SIZE}   lr = {LR}")
print(f"[BranchA] view_every_k  = {VIEW_EVERY_K}  bcl_batchn = {USE_BCL_BATCHN}")
print(f"[BranchA] W&B group     = {WB_GROUP!r}   tags = {WB_TAGS!r}")

# ---- 2. Backbone flags (frozen apc_off_combined) ---------------------
BACKBONE_FLAGS: list[str] = [
    "--dataset",                     str(_DATASET),
    "--gpu_id",                      "0",
    "--epoch",                       "250",
    "--verbose",                     "5",
    "--batch_size",                  str(BATCH_SIZE),
    "--lr",                          str(LR),
    "--regs",                        "0.001",
    "--clip_grad_norm",              "1.0",
    "--embed_size",                  "64",
    "--topk",                        "5",
    "--core",                        "5",
    "--User_layers",                 "3",
    "--Item_layers",                 "2",
    "--user_loss_ratio",             "0.03",
    "--item_loss_ratio",             "0.07",
    "--temperature",                 "0.3",
    "--learnable_tau",               "0",
    "--early_stopping_patience",     str(PATIENCE),
    "--early_stopping_min_epochs",   str(MIN_EPOCHS),
    "--early_stopping_min_delta",    "0.0001",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    "--use_reduce_lr",               "1",
    "--reduce_lr_factor",            "0.5",
    "--reduce_lr_patience",          "3",
    "--reduce_lr_min",               "1e-06",
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_permutation_fft",       "0",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_num_categories",        "10",
    "--damps_warmup_epochs",         "10",
    "--rebuild_R",                   "5",
    "--faiss_threshold",             "60000",
    "--faiss_use_gpu",               "1",
    "--knn_chunk_size",              "4096",
    "--knn_efsearch",                "64",
    "--use_amp",                     "1",
    "--enable_logq",                 "1",
    "--logq_mode",                   LOGQ_MODE_LOCK,
    "--logq_beta",                   str(LOGQ_BETA_LOCK),
    "--logq_scale",                  str(LOGQ_SCALE_LOCK),
    "--logq_clip",                   str(LOGQ_CLIP_LOCK),
    "--enable_simgcl",               "1",
    "--simgcl_eps",                  str(SIMGCL_EPS),
    "--lambda_view",                 str(LAMBDA_VIEW_LOCK),
    "--simgcl_batch_size_user",      str(BRANCHA_VIEW_BSZ),
    "--simgcl_batch_size_item",      str(BRANCHA_VIEW_BSZ),
    "--branchA_view_every_k",        str(VIEW_EVERY_K),
    "--branchA_bcl_batchn",          str(USE_BCL_BATCHN),
    "--branchA_view_bsz",            str(BRANCHA_VIEW_BSZ),
    "--branchA_bcl_bsz",             str(BRANCHA_BCL_BSZ),
    # W&B per-seed run config
    "--use_wandb",                   "1",
    "--wandb_project",               _WB_PROJECT,
    "--wandb_entity",                _WB_ENTITY,
    "--wandb_group",                 WB_GROUP,
    "--wandb_tags",                  WB_TAGS,
    "--wandb_job_type",              "sweep_seed",
]

# ---- 3. Metric parser -----------------------------------------------
_S = r"[^0-9\-]*(-?[0-9]*\.?[0-9]+)"
_PAT = {
    "recall@20":             re.compile(r"BEST_Test_Recall@20"        + _S),
    "ndcg@20":               re.compile(r"BEST_Test_NDCG@20"          + _S),
    "precision@20":          re.compile(r"BEST_Test_Precision@20"     + _S),
    "val_recall@20":         re.compile(r"BEST_Val_Recall@20"         + _S),
    "val_ndcg@20":           re.compile(r"BEST_Val_NDCG@20"           + _S),
    "val_recall_peak_epoch": re.compile(r"BEST_Val_Recall_Peak_Epoch" + _S),
}

def _parse_metrics(text: str) -> dict[str, float]:
    out: dict[str, float] = {}
    for k, pat in _PAT.items():
        hits = pat.findall(text)
        if hits:
            out[k] = float(hits[-1])
    return out

def _mean(xs: list[float]) -> float:
    return sum(xs) / len(xs) if xs else float("nan")

def _std(xs: list[float]) -> float:
    if len(xs) < 2:
        return 0.0
    m = _mean(xs)
    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5

# ---- 4. W&B controller run ------------------------------------------
# One long-lived run that logs rolling stats + final summary table.
# Each seed subprocess has its OWN separate per-epoch W&B run.
print("\n[W&B] Initialising controller run ...")
_wb_ctrl = wandb.init(
    project=_WB_PROJECT,
    entity=_WB_ENTITY if _WB_ENTITY else None,
    name="branchA_sweep_controller",
    group=WB_GROUP,
    tags=["branchA", "sweep_controller", "rev55"],
    job_type="sweep_controller",
    config={
        "phase":            "branchA",
        "dataset":          _DATASET,
        "backbone":         "apc_off_combined",
        "lambda_view":      LAMBDA_VIEW_LOCK,
        "simgcl_eps":       SIMGCL_EPS,
        "view_every_k":     VIEW_EVERY_K,
        "bcl_batchn":       USE_BCL_BATCHN,
        "brancha_view_bsz": BRANCHA_VIEW_BSZ,
        "brancha_bcl_bsz":  BRANCHA_BCL_BSZ,
        "batch_size":       BATCH_SIZE,
        "lr":               LR,
        "patience":         PATIENCE,
        "logq_scale":       LOGQ_SCALE_LOCK,
        "logq_mode":        LOGQ_MODE_LOCK,
        "logq_beta":        LOGQ_BETA_LOCK,
        "logq_clip":        LOGQ_CLIP_LOCK,
        "seeds":            SEEDS,
        "n_seeds":          N_SEEDS,
        "gate_min":         GATE_MIN_RECALL,
        "gate_max":         GATE_MAX_RECALL,
        "gate_fail":        GATE_FAIL_BELOW,
    },
    reinit=True,
)
print(f"[W&B] Controller run: {_wb_ctrl.url}")

# Define W&B Table schema for per-seed results
_WB_COLS = [
    "seed", "recall@20", "ndcg@20", "precision@20",
    "val_recall@20", "val_ndcg@20", "val_recall_peak_epoch",
    "runtime_h", "in_window", "ok",
]
_wb_table = wandb.Table(columns=_WB_COLS)

# ---- 5. Run all seeds -----------------------------------------------
all_runs: list[dict[str, Any]] = []

print("=" * 72)
print(f"Branch A SWEEP: {len(SEEDS)} seeds  (lambda_view={LAMBDA_VIEW_LOCK} LOCKED)")
print("=" * 72)

for idx, seed in enumerate(SEEDS, start=1):
    seed_int = int(seed)
    abl_tag = f"branchA_lam{LAMBDA_VIEW_LOCK}_seed{seed_int}"
    # Each seed also gets a seed-specific tag for easy filtering in W&B
    seed_tags = f"{WB_TAGS},seed{seed_int}"

    cmd: list[str] = [
        _PYTHON, "train.py",
        *BACKBONE_FLAGS,
        "--seed",            str(seed_int),
        "--wandb_run_name",  abl_tag,
        "--wandb_tags",      seed_tags,       # override to add seed-specific tag
        "--ablation_target", abl_tag,
    ]

    print(f"\n[{idx}/{len(SEEDS)}] seed={seed_int}  run_name={abl_tag!r}", flush=True)
    t0 = time.time()
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.run(
        cmd, capture_output=True, text=True, cwd=_DAMPS_DIR, env=env
    )
    dt = time.time() - t0
    log = (proc.stdout or "") + "\n" + (proc.stderr or "")
    m = _parse_metrics(log)
    ok = ("recall@20" in m) and (proc.returncode == 0)
    r20 = m.get("recall@20")
    in_window = (
        r20 is not None
        and GATE_MIN_RECALL <= r20 <= GATE_MAX_RECALL
    )

    if not ok:
        print(f"  [WARN] seed={seed_int} rc={proc.returncode}")
        print(log[-600:])

    result: dict[str, Any] = {
        "seed":                  seed_int,
        "run_name":              abl_tag,
        "ablation_target":       abl_tag,
        "recall@20":             r20,
        "ndcg@20":               m.get("ndcg@20"),
        "precision@20":          m.get("precision@20"),
        "val_recall@20":         m.get("val_recall@20"),
        "val_ndcg@20":           m.get("val_ndcg@20"),
        "val_recall_peak_epoch": (int(m["val_recall_peak_epoch"])
                                  if "val_recall_peak_epoch" in m else None),
        "runtime_s":             round(dt, 1),
        "runtime_h":             round(dt / 3600.0, 2),
        "in_window":             in_window,
        "ok":                    ok,
    }
    all_runs.append(result)

    # -- Rolling W&B log (after every completed seed) ------------------
    completed_recalls = [
        r["recall@20"] for r in all_runs if r["recall@20"] is not None
    ]
    completed_ndcgs = [
        r["ndcg@20"] for r in all_runs if r["ndcg@20"] is not None
    ]
    _wb_ctrl.log({
        "branchA/seed_recall":          r20 if r20 is not None else float("nan"),
        "branchA/seed_ndcg":            m.get("ndcg@20", float("nan")),
        "branchA/seed_precision":       m.get("precision@20", float("nan")),
        "branchA/seed_runtime_h":       round(dt / 3600.0, 2),
        "branchA/rolling_mean_recall":  _mean(completed_recalls),
        "branchA/rolling_std_recall":   _std(completed_recalls),
        "branchA/n_completed":          len(all_runs),
        "branchA/seed_in_window":       int(in_window),
        "branchA/cumulative_pass":      int(
            all(GATE_MIN_RECALL <= r <= GATE_MAX_RECALL
                for r in completed_recalls)
        ),
        "seed_idx":                     idx,
    })

    # -- Add row to the W&B Table -------------------------------------
    _wb_table.add_data(
        seed_int,
        r20,
        m.get("ndcg@20"),
        m.get("precision@20"),
        m.get("val_recall@20"),
        m.get("val_ndcg@20"),
        result["val_recall_peak_epoch"],
        round(dt / 3600.0, 2),
        in_window,
        ok,
    )

    # -- Console summary for this seed --------------------------------
    flag = "PASS" if in_window else ("WARN" if (r20 or 0) >= GATE_FAIL_BELOW else "FAIL")
    print(
        f"  -> R@20={r20:.6f}  N@20={m.get('ndcg@20', 0):.6f}  "
        f"P@20={m.get('precision@20', 0):.6f}  "
        f"peak_ep={result['val_recall_peak_epoch']}  "
        f"time={round(dt/3600,2):.2f}h  [{flag}]"
    )

# ---- 6. Final gate decision -----------------------------------------
recalls    = [r["recall@20"]    for r in all_runs if r["recall@20"]    is not None]
ndcgs      = [r["ndcg@20"]      for r in all_runs if r["ndcg@20"]      is not None]
precisions = [r["precision@20"] for r in all_runs if r["precision@20"] is not None]

mean_r = _mean(recalls)
std_r  = _std(recalls)
min_r  = min(recalls) if recalls else float("nan")
max_r  = max(recalls) if recalls else float("nan")

gate_pass = bool(recalls) and all(
    GATE_MIN_RECALL <= r <= GATE_MAX_RECALL for r in recalls
)
gate_fail_rollback = any(r < GATE_FAIL_BELOW for r in recalls)

print()
print("=" * 72)
print(f"{'seed':>8} {'R@20':>10} {'N@20':>10} {'P@20':>10} {'hours':>7} {'status':>8}")
print("-" * 80)
for r in all_runs:
    flag = "PASS" if r.get("in_window") else (
        "WARN" if (r["recall@20"] or 0) >= GATE_FAIL_BELOW else "FAIL"
    )
    print(
        f"{r['seed']:>8} "
        f"{(r['recall@20'] or 0):>10.6f} "
        f"{(r['ndcg@20'] or 0):>10.6f} "
        f"{(r['precision@20'] or 0):>10.6f} "
        f"{r['runtime_h']:>7.2f} "
        f"{flag:>8}"
    )
print("-" * 80)
print(f"mean R@20 = {mean_r:.6f} +/- {std_r:.6f}  "
      f"min={min_r:.6f}  max={max_r:.6f}")
print(f"mean P@20 = {_mean(precisions):.6f} +/- {_std(precisions):.6f}  "
      f"min={min(precisions) if precisions else float('nan'):.6f}  "
      f"max={max(precisions) if precisions else float('nan'):.6f}")
print(f"[BranchA] gate [{GATE_MIN_RECALL}, {GATE_MAX_RECALL}]: "
      f"{'PASS' if gate_pass else 'FAIL'}")
if gate_fail_rollback:
    print(f"[BranchA] ROLLBACK: seed(s) below {GATE_FAIL_BELOW} — "
          "revert to Wave 1 LogQ-only.")

# ---- 7. Persist JSON ------------------------------------------------
os.makedirs("results", exist_ok=True)
out_json = os.path.join("results", "branchA_sweep_results.json")
summary_dict = {
    "lambda_view": LAMBDA_VIEW_LOCK,
    "view_every_k": VIEW_EVERY_K,
    "bcl_batchn": USE_BCL_BATCHN,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "mean_recall": round(mean_r, 6),
    "std_recall": round(std_r, 6),
    "min_recall": round(min_r, 6),
    "max_recall": round(max_r, 6),
    "mean_ndcg": round(_mean(ndcgs), 6) if ndcgs else None,
    "mean_precision": round(_mean(precisions), 6) if precisions else None,
    "std_precision": round(_std(precisions), 6) if precisions else None,
    "gate_pass": gate_pass,
}
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(
        {"runs": all_runs, "summary": summary_dict, "gate": gate_pass},
        f, indent=2,
    )
print(f"\n[BranchA] results -> {out_json}")

# ---- 8. W&B final summary -------------------------------------------
_wb_ctrl.log({
    "branchA/final_mean_recall":  mean_r,
    "branchA/final_std_recall":   std_r,
    "branchA/final_min_recall":   min_r,
    "branchA/final_max_recall":   max_r,
    "branchA/final_mean_ndcg":    _mean(ndcgs) if ndcgs else float("nan"),
    "branchA/final_mean_precision": _mean(precisions) if precisions else float("nan"),
    "branchA/final_std_precision":  _std(precisions) if precisions else float("nan"),
    "branchA/gate_pass":          int(gate_pass),
    "branchA/n_seeds_in_window":  sum(1 for r in all_runs if r.get("in_window")),
    "branchA/all_seeds_table":    _wb_table,
})
_wb_ctrl.summary.update({
    "branchA/gate_passed":        gate_pass,
    "branchA/mean_recall":        round(mean_r, 6),
    "branchA/std_recall":         round(std_r, 6),
    "branchA/min_recall":         round(min_r, 6),
    "branchA/mean_precision":     round(_mean(precisions), 6) if precisions else None,
    "branchA/verdict":            "PASS" if gate_pass else "FAIL",
    "branchA/n_seeds_ok":         sum(1 for r in all_runs if r["ok"]),
    "branchA/mean_runtime_h":     round(_mean([r["runtime_h"] for r in all_runs]), 2),
})

# Attach JSON results as a W&B Artifact
_artifact = wandb.Artifact(
    name=f"branchA_sweep_{_DATASET}",
    type="sweep_results",
    description=(
        f"Branch A (LogQ + SimGCL batch-N) sweep results "
        f"on {_DATASET}. Gate: {'PASS' if gate_pass else 'FAIL'}."
    ),
    metadata={"gate_pass": gate_pass, "mean_recall": round(mean_r, 6)},
)
_artifact.add_file(out_json, name="branchA_sweep_results.json")
_wb_ctrl.log_artifact(_artifact)

_wb_ctrl.finish()
print(f"[W&B] Controller run finished: {_wb_ctrl.url}")

### 9.2 Branch A Post-hoc Analysis

Read-only analysis cell. **Run after §9.1 completes.**

**Reads:** `results/branchA_sweep_results.json` (produced by §9.1).

**Produces:**
- val/Recall@20 vs epoch plot for all 5 seeds (with gate guide lines)
- loss_simgcl_view trajectory per seed
- Gate verdict: PASS / FAIL with mean ± std
- Wall-clock check: mean should be in [4 h, 8 h] per seed

In [ ]:
# =====================================================================
# Section 8.2 -- Branch A post-hoc analysis  (read-only, rev55 Â§8.1)
#
#   Inputs:
#     results/branchA_sweep_results.json   (from Section 8.1 sweep cell)
#     ../<dataset>/damps_*_branchA_*/*.txt   (per-run training logs)
# =====================================================================
from __future__ import annotations

import glob
import json
import math
import os
import re
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DAMPS_DIR = os.getcwd()
try:
    _DATASET = dataset  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DATASET = "Clothing"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

GATE_MIN: float = 0.0900
GATE_MAX: float = 0.0945
GATE_FAIL: float = 0.0890

RESULTS_JSON = os.path.join(_DAMPS_DIR, "results", "branchA_sweep_results.json")
if not os.path.isfile(RESULTS_JSON):
    raise FileNotFoundError(
        f"{RESULTS_JSON} not found â€” run the Section 8.1 sweep cell first."
    )

with open(RESULTS_JSON, "r", encoding="utf-8") as f:
    payload = json.load(f)

runs: list[dict[str, Any]] = payload.get("runs", [])
summary: dict[str, Any] = payload.get("summary", {})
gate_pass: bool = bool(payload.get("gate"))

print("=" * 72)
print(f"[BranchA/analysis] gate: {'PASS' if gate_pass else 'FAIL'}")
print(
    f"  mean R@20 = {summary.get('mean_recall')} "
    f"+/- {summary.get('std_recall')}  "
    f"min={summary.get('min_recall')}  max={summary.get('max_recall')}"
)
print(
    f"  acceptance window [{GATE_MIN}, {GATE_MAX}]  "
    f"rollback if any seed < {GATE_FAIL}"
)
print("=" * 72)
print(f"{'seed':>8} {'R@20':>10} {'N@20':>10} {'hours':>8} {'window':>8}")
print("-" * 52)
for r in runs:
    r20 = r.get("recall@20")
    in_win = (
        r20 is not None and GATE_MIN <= float(r20) <= GATE_MAX
    )
    print(
        f"{r.get('seed', 0):>8} "
        f"{(r20 or 0):>10.6f} "
        f"{(r.get('ndcg@20') or 0):>10.6f} "
        f"{(r.get('runtime_h') or 0):>8.2f} "
        f"{('PASS' if in_win else 'FAIL'):>8}"
    )

# ---- Parse per-epoch curves from log directories -----------------------
LOG_ROOT = os.path.abspath(os.path.join("..", _DATASET))
DIR_RE = re.compile(
    r"damps_.*?_seed=(?P<seed>\d+)_branchA_lam[0-9.]+_seed(?P=seed)$"
)
PAT_EPOCH = re.compile(
    r"Epoch\s+(\d+)\s*\[[^\]]+\]:\s*loss=([\d.]+)\s+"
    r"recall@\d+=([\d.]+)\s+recall@(\d+)=([\d.]+)\s+ndcg@\d+=([\d.]+)"
)
PAT_VIEW = re.compile(r"loss_simgcl_view=([\d.eE+\-]+)")

curves: list[dict[str, Any]] = []
for d in sorted(glob.glob(os.path.join(LOG_ROOT, "damps_*branchA*"))):
    if not os.path.isdir(d):
        continue
    m = DIR_RE.match(os.path.basename(d.rstrip("/")))
    if not m:
        continue
    txts = glob.glob(os.path.join(d, "*.txt"))
    if not txts:
        continue
    with open(txts[0], "r", encoding="utf-8", errors="replace") as fh:
        text = fh.read()
    epochs: list[int] = []
    val_r20: list[float] = []
    view_loss: list[Optional[float]] = []
    for line in text.splitlines():
        em = PAT_EPOCH.search(line)
        if em:
            epochs.append(int(em.group(1)))
            val_r20.append(float(em.group(5)))
            vm = PAT_VIEW.search(line)
            view_loss.append(float(vm.group(1)) if vm else None)
    curves.append({
        "seed": int(m.group("seed")),
        "epochs": epochs,
        "val_rec20": val_r20,
        "view_loss": view_loss,
    })

print(f"\n[BranchA/analysis] parsed {len(curves)} log directories under {LOG_ROOT}")

if curves:
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.5))

    ax = axes[0]
    for c in sorted(curves, key=lambda x: x["seed"]):
        if not c["epochs"]:
            continue
        ax.plot(
            c["epochs"],
            c["val_rec20"],
            linewidth=1.4,
            alpha=0.85,
            label=f"seed={c['seed']}",
        )
    ax.axhline(GATE_MIN, color="green", linestyle="--", linewidth=0.9, label="gate min")
    ax.axhline(GATE_MAX, color="green", linestyle=":", linewidth=0.9, label="gate max")
    ax.axhline(GATE_FAIL, color="red", linestyle="--", linewidth=0.9, label="rollback")
    ax.set_title("Branch A â€” val/recall@20 vs epoch")
    ax.set_xlabel("epoch")
    ax.set_ylabel("val/recall@20")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax2 = axes[1]
    for c in sorted(curves, key=lambda x: x["seed"]):
        eps = c["epochs"]
        vl = [v for v in c["view_loss"] if v is not None]
        if len(vl) < 2:
            continue
        ax2.plot(eps[: len(vl)], vl, linewidth=1.2, alpha=0.85, label=f"seed={c['seed']}")
    ax2.set_title("Branch A â€” loss_simgcl_view vs epoch")
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("loss_simgcl_view")
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)

    plt.tight_layout()
    fig_path = os.path.join(LOG_ROOT, "branchA_val_recall_curves.png")
    plt.savefig(fig_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"[plot] saved -> {fig_path}")

# ---- Runtime budget check ----------------------------------------------
hours = [r.get("runtime_h") for r in runs if r.get("runtime_h") is not None]
if hours:
    mean_h = float(np.mean(hours))
    print(
        f"\n[BranchA/analysis] wall-clock: mean={mean_h:.2f} h/seed  "
        f"min={min(hours):.2f}  max={max(hours):.2f}  "
        f"(target 4â€“8 h/seed)"
    )
    if mean_h > 8.0:
        print("[WARN] mean runtime exceeds 8 h/seed â€” revisit batch-N / view_every_k.")

if not gate_pass:
    print(
        "\n[BranchA/analysis] ROLLBACK: set "
        "--enable_simgcl 0 --branchA_bcl_batchn 0 --branchA_view_every_k 1 "
        "to revert to Wave 1 LogQ-only."
    )

### 9.7 Branch A Optuna HPO Sweep — Amazon **Clothing** (35 trials × 1 seed)

Re-runs the Branch A (LogQ + SimGCL batch-N) hyper-parameter search on **Amazon Clothing**, with a search space **adapted to the Clothing↔Sports differences**. Each trial = one distinct configuration evaluated with **1 seed** over up to **250 epochs**; **35 trials** total.

**Why the search space differs (standard MMHCL splits):**

| Dataset | #Users | #Items | #Inter. | Density |
|---|---|---|---|---|
| Sports | 35,598 | 18,357 | 256,308 | 0.00038 |
| Clothing | 39,387 | 23,033 | 237,488 | 0.00025 |

Clothing is ~34% **sparser**, with more users/items but fewer interactions. So vs the Sports space: `regs` ceiling widened (→2e-2), `lr` floor lowered (→8e-5), `lambda_view` (→0.25) and `simgcl_eps` (→0.35) allowed stronger, and `logq_scale` lower-centred (0.3–1.3, milder popularity tail).

Sampler **TPE(multivariate, constant_liar)**, **Hyperband** pruning (min_resource=30, rf=3, streamed per-epoch), resumable SQLite study, objective **BEST_Test_Recall@20**. Full **WandB** monitoring mirrors §9.1: controller run + per-trial live runs + Table + leaderboard + JSON/DB Artifact. After it finishes, take the top trials into a 3-seed × 250-epoch final evaluation (like the Sports §9.6 cell).

In [ ]:
# =====================================================================
# Section 9.7 -- Branch A Optuna HPO Sweep on Amazon CLOTHING
#                35 trials  x  1 seed/trial  x  up to 250 epochs
#
#   Goal
#   ----
#   Re-run the Branch A (LogQ + SimGCL batch-N) hyper-parameter search
#   -- previously done on Amazon Sports -- now on Amazon CLOTHING, with a
#   search space ADAPTED to the two datasets' differences (see below).
#   Each trial is one distinct configuration evaluated with ONE seed.
#
#   Dataset differences that motivate the adapted search space
#   ----------------------------------------------------------
#   (standard MMRec / MMHCL splits)
#       Dataset   #Users   #Items   #Inter.   Density
#       Sports    35,598   18,357   256,308   0.00038
#       Clothing  39,387   23,033   237,488   0.00025
#   => Clothing is ~34% SPARSER, has MORE users (+11%) and MORE items
#      (+25%) but FEWER interactions. Consequences for the search space:
#        * Sparser + more items  -> higher over-fit risk
#              -> widen `regs` upper bound (1e-4..2e-2, was ..1e-2)
#              -> allow slightly LOWER lr (8e-5..1.5e-3, was 1e-4..2e-3)
#        * Sparse graphs benefit more from the contrastive signal
#              -> allow STRONGER `lambda_view` (0.01..0.25, was ..0.20)
#              -> allow STRONGER `simgcl_eps` (0.05..0.35, was ..0.30)
#        * Clothing popularity tail is LESS extreme than Sports
#              -> shift `logq_scale` lower-centred (0.3..1.3, was 0.5..1.5)
#        * More items -> larger embed often helps, but let TPE decide
#              -> keep embed {64,128}, UI_layers {2,3,4}, bs {1024,2048,4096}
#
#   Sampler / Pruner  (identical methodology to the Sports sweep)
#   ------------------------------------------------------------
#     Sampler : TPESampler(seed=42, multivariate=True, constant_liar=True)
#     Pruner  : HyperbandPruner(min_resource=30, max_resource=EPOCH,
#                               reduction_factor=3)  -- streamed per-epoch
#     Storage : SQLite  (resumable)   Objective: BEST_Test_Recall@20 (max)
#
#   Frozen backbone == Section 9.1 (apc_off_combined + LogQ laplace/
#   beta=1.0/clip=5.0 + SimGCL batch-N + view_every_k=2). Only the
#   searched HPs vary per trial.
#
#   W&B logging (mirrors Section 9.1)
#   ---------------------------------
#     * one long-lived CONTROLLER run: per-trial value, rolling best,
#       a Table of every trial, a ranked leaderboard Table + JSON Artifact
#     * each trial's train.py sub-process gets its OWN per-epoch W&B run
#       via `--use_wandb 1`
# =====================================================================
from __future__ import annotations

import json
import math
import os
import re
import subprocess
import sys
import time
from typing import Any, Optional

import wandb

# Optuna is required for this cell.
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import HyperbandPruner, NopPruner
except ModuleNotFoundError:
    print("[setup] installing optuna ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna>=3.4"], check=True)
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import HyperbandPruner, NopPruner

# ---- 0. Resolve dirs / globals (reuse from earlier cells if set) ------
try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PYTHON = sys.executable
try:
    _DATASET = dataset  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DATASET = "Clothing"
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_ENTITY = ""

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---- 1. Sweep settings -----------------------------------------------
N_TRIALS: int = 35            # 35 distinct configurations
N_SEEDS: int = 1              # one seed per trial (per requirement)
EPOCH: int = 250
PATIENCE: int = 20
MIN_EPOCHS: int = 75

STUDY_NAME: str = "branchA_clothing_v1"
STORAGE: str = "sqlite:///optuna_branchA_clothing.db"   # resumable
SAMPLER_SEED: int = 42
BASE_SEED: int = 2042992639   # same base seed used for the Sports sweep

# Hyperband pruning knobs (identical methodology to Sports)
MIN_RESOURCE: int = 30        # prune bad trials after epoch 30
REDUCTION_FACTOR: int = 3
STREAM_PRUNING: bool = True   # parse per-epoch val recall from stdout

# Frozen LogQ settings (identical to Section 9.1)
LOGQ_MODE: str = "laplace"
LOGQ_BETA: float = 1.0
LOGQ_CLIP: float = 5.0

WB_GROUP: str = "wave2_optuna_clothing"
WB_TAGS: str = "branchA,optuna,clothing,rev55,logq,batchN"


# ---- 2. CLOTHING-ADAPTED search space --------------------------------
def suggest_hp(trial: "optuna.trial.Trial") -> dict[str, Any]:
    """Sample one Branch A configuration. Ranges are widened / shifted
    relative to the Sports sweep to reflect Clothing's higher sparsity,
    larger item catalogue and milder popularity tail (see header)."""
    hp: dict[str, Any] = {
        # lower floor: sparser data tends to prefer smaller lr
        "lr":            trial.suggest_float("lr", 8e-5, 1.5e-3, log=True),
        "batch_size":    trial.suggest_categorical("batch_size", [1024, 2048, 4096]),
        # wider ceiling: more regularisation to fight over-fit on sparse graph
        "regs":          trial.suggest_float("regs", 1e-4, 2e-2, log=True),
        # stronger perturbation allowed
        "simgcl_eps":    trial.suggest_float("simgcl_eps", 0.05, 0.35),
        # stronger contrastive weight allowed
        "lambda_view":   trial.suggest_float("lambda_view", 0.01, 0.25, log=True),
        "temperature":   trial.suggest_float("temperature", 0.10, 0.50),
        "embed_size":    trial.suggest_categorical("embed_size", [64, 128]),
        "UI_layers":     trial.suggest_categorical("UI_layers", [2, 3, 4]),
        # lower-centred: Clothing popularity tail is less extreme than Sports
        "logq_scale":    trial.suggest_float("logq_scale", 0.3, 1.3),
        "use_reduce_lr": trial.suggest_categorical("use_reduce_lr", [0, 1]),
    }
    if int(hp["use_reduce_lr"]) == 1:
        hp["reduce_lr_factor"]   = trial.suggest_float("reduce_lr_factor", 0.3, 0.7)
        hp["reduce_lr_patience"] = trial.suggest_int("reduce_lr_patience", 3, 8)
    return hp


# ---- 3. Frozen backbone flags (invariant across all trials) ----------
#   Byte-for-byte aligned with Section 9.1 BACKBONE_FLAGS (minus the
#   searched HPs, which are appended per trial).
def frozen_flags() -> list[str]:
    return [
        "--dataset",                     str(_DATASET),
        "--gpu_id",                      "0",
        "--epoch",                       str(EPOCH),
        "--verbose",                     "5",
        "--clip_grad_norm",              "1.0",
        "--topk",                        "5",
        "--core",                        "5",
        "--User_layers",                 "3",
        "--Item_layers",                 "2",
        "--user_loss_ratio",             "0.03",
        "--item_loss_ratio",             "0.07",
        "--learnable_tau",               "0",
        "--early_stopping_patience",     str(PATIENCE),
        "--early_stopping_min_epochs",   str(MIN_EPOCHS),
        "--early_stopping_min_delta",    "0.0001",
        "--early_stopping_mode",         "max",
        "--early_stopping_restore_best", "1",
        "--reduce_lr_min",               "1e-06",
        "--damps_apc",                   "0",
        "--damps_avrf",                  "0",
        "--damps_imcf",                  "1",
        "--damps_permutation_fft",       "0",
        "--damps_soft_routing",          "1",
        "--damps_momentum",              "1",
        "--damps_data_driven_prior",     "1",
        "--damps_num_categories",        "10",
        "--damps_warmup_epochs",         "10",
        "--rebuild_R",                   "5",
        "--faiss_threshold",             "60000",
        "--faiss_use_gpu",               "1",
        "--knn_chunk_size",              "4096",
        "--knn_efsearch",                "64",
        "--use_amp",                     "1",
        "--enable_logq",                 "1",
        "--logq_mode",                   LOGQ_MODE,
        "--logq_beta",                   str(LOGQ_BETA),
        "--logq_clip",                   str(LOGQ_CLIP),
        "--enable_simgcl",               "1",
        "--simgcl_batch_size_user",      "2048",
        "--simgcl_batch_size_item",      "2048",
        "--branchA_view_every_k",        "2",
        "--branchA_bcl_batchn",          "1",
        "--branchA_view_bsz",            "2048",
        "--branchA_bcl_bsz",             "2048",
    ]


def hp_flags(hp: dict[str, Any]) -> list[str]:
    """Only the searched HPs (appended after frozen flags)."""
    flags: list[str] = [
        "--lr",            str(hp["lr"]),
        "--batch_size",    str(hp["batch_size"]),
        "--regs",          str(hp["regs"]),
        "--simgcl_eps",    str(hp["simgcl_eps"]),
        "--lambda_view",   str(hp["lambda_view"]),
        "--temperature",   str(hp["temperature"]),
        "--embed_size",    str(hp["embed_size"]),
        "--UI_layers",     str(hp["UI_layers"]),
        "--logq_scale",    str(hp["logq_scale"]),
        "--use_reduce_lr", str(int(hp["use_reduce_lr"])),
    ]
    if int(hp["use_reduce_lr"]) == 1:
        flags += [
            "--reduce_lr_factor",   str(hp["reduce_lr_factor"]),
            "--reduce_lr_patience", str(hp["reduce_lr_patience"]),
        ]
    return flags


# ---- 4. Metric parsers (same contract as Section 9.1) ----------------
_S = r"[^0-9\-]*(-?[0-9]*\.?[0-9]+)"
_PAT: dict[str, "re.Pattern[str]"] = {
    "recall@20":             re.compile(r"BEST_Test_Recall@20"        + _S),
    "ndcg@20":               re.compile(r"BEST_Test_NDCG@20"          + _S),
    "precision@20":          re.compile(r"BEST_Test_Precision@20"     + _S),
    "val_recall@20":         re.compile(r"BEST_Val_Recall@20"         + _S),
    "val_ndcg@20":           re.compile(r"BEST_Val_NDCG@20"           + _S),
    "val_recall_peak_epoch": re.compile(r"BEST_Val_Recall_Peak_Epoch" + _S),
}
# Per-epoch val recall@20 for Hyperband pruning (train.py logger format,
# with optional leading "[timestamp]"). Anchored on recall@20 so it never
# mismatches recall@10.
_EPOCH_RE = re.compile(r"Epoch\s+(\d+)\s*\[.*?\].*?\brecall@20=([0-9]+\.?[0-9]*)")


def _parse_metrics(text: str) -> dict[str, float]:
    out: dict[str, float] = {}
    for k, pat in _PAT.items():
        hits = pat.findall(text)
        if hits:
            out[k] = float(hits[-1])
    return out


# ---- 5. W&B controller run (mirrors Section 9.1 controller) ----------
print("[W&B] Initialising Clothing Optuna controller run ...", flush=True)
_wb_ctrl = wandb.init(
    project=_WB_PROJECT,
    entity=_WB_ENTITY if _WB_ENTITY else None,
    name="optuna_clothing_controller",
    group=WB_GROUP,
    tags=["branchA", "optuna_controller", "clothing", "rev55"],
    job_type="optuna_controller",
    config={
        "phase":            "optuna_clothing",
        "dataset":          _DATASET,
        "backbone":         "apc_off_combined",
        "n_trials":         N_TRIALS,
        "n_seeds":          N_SEEDS,
        "epoch":            EPOCH,
        "patience":         PATIENCE,
        "min_epochs":       MIN_EPOCHS,
        "sampler":          "TPE(multivariate,constant_liar)",
        "pruner":           f"Hyperband(min={MIN_RESOURCE},rf={REDUCTION_FACTOR})",
        "objective":        "BEST_Test_Recall@20 (maximize)",
        "logq_mode":        LOGQ_MODE,
        "logq_beta":        LOGQ_BETA,
        "logq_clip":        LOGQ_CLIP,
        "study_name":       STUDY_NAME,
        "storage":          STORAGE,
        "search_space":     "clothing_adapted_v1",
    },
    reinit=True,
)
print(f"[W&B] Controller run: {_wb_ctrl.url}", flush=True)

_WB_COLS = [
    "trial", "state", "recall@20", "ndcg@20", "precision@20",
    "val_recall@20", "val_recall_peak_epoch",
    "lr", "batch_size", "regs", "simgcl_eps", "lambda_view",
    "temperature", "embed_size", "UI_layers", "logq_scale",
    "use_reduce_lr", "runtime_h", "ok",
]
_wb_table = wandb.Table(columns=_WB_COLS)

# Shared mutable state across objective() calls
_state: dict[str, Any] = {"trial_rows": [], "run_idx": 0}


# ---- 6. Objective ----------------------------------------------------
def objective(trial: "optuna.trial.Trial") -> float:
    hp = suggest_hp(trial)
    seed_int = BASE_SEED + trial.number          # deterministic, 1 seed/trial
    run_name = f"clothing_t{trial.number:04d}_s0"
    trial_tags = f"{WB_TAGS},trial{trial.number}"

    wandb_args: list[str] = [
        "--use_wandb",      "1",
        "--wandb_project",  _WB_PROJECT,
        "--wandb_group",    WB_GROUP,
        "--wandb_tags",     trial_tags,
        "--wandb_job_type", "optuna_trial",
        "--wandb_run_name", run_name,
    ]
    if _WB_ENTITY:
        wandb_args += ["--wandb_entity", _WB_ENTITY]

    cmd: list[str] = [
        _PYTHON, "train.py",
        *frozen_flags(),
        *hp_flags(hp),
        *wandb_args,
        "--seed",            str(seed_int),
        "--ablation_target", run_name,
    ]

    print(f"\n{'='*72}\n[Trial {trial.number}/{N_TRIALS-1}] seed={seed_int}  {run_name}")
    print(f"  lr={hp['lr']:.3e} bs={hp['batch_size']} regs={hp['regs']:.2e} "
          f"eps={hp['simgcl_eps']:.3f} lv={hp['lambda_view']:.4f}")
    print(f"  T={hp['temperature']:.3f} embed={hp['embed_size']} "
          f"UI={hp['UI_layers']} logq={hp['logq_scale']:.3f} "
          f"reduce_lr={hp['use_reduce_lr']}\n{'='*72}", flush=True)

    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    t0 = time.time()
    captured: list[str] = []
    pruned = False

    proc = subprocess.Popen(
        cmd, cwd=_DAMPS_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    try:
        for line in proc.stdout:                 # type: ignore[union-attr]
            captured.append(line)
            if STREAM_PRUNING:
                mm = _EPOCH_RE.search(line)
                if mm:
                    ep = int(mm.group(1)); rec = float(mm.group(2))
                    trial.report(rec, step=ep)
                    if trial.should_prune():
                        pruned = True
                        proc.terminate()
                        break
    finally:
        proc.wait()

    dt = time.time() - t0
    log = "".join(captured)
    m = _parse_metrics(log)
    r20 = m.get("recall@20")
    n20 = m.get("ndcg@20")
    ok = (r20 is not None) and (n20 is not None) and (proc.returncode == 0) and not pruned

    _state["run_idx"] += 1
    row = {
        "trial": trial.number,
        "state": "PRUNED" if pruned else ("COMPLETE" if ok else "FAIL"),
        "seed": seed_int, "run_name": run_name, "hp": hp,
        "recall@20": r20, "ndcg@20": n20, "precision@20": m.get("precision@20"),
        "val_recall@20": m.get("val_recall@20"),
        "val_recall_peak_epoch": (int(m["val_recall_peak_epoch"])
                                  if "val_recall_peak_epoch" in m else None),
        "runtime_h": round(dt / 3600.0, 2), "ok": ok,
    }
    _state["trial_rows"].append(row)

    _wb_table.add_data(
        trial.number, row["state"], r20, n20, m.get("precision@20"),
        m.get("val_recall@20"), row["val_recall_peak_epoch"],
        hp["lr"], hp["batch_size"], hp["regs"], hp["simgcl_eps"],
        hp["lambda_view"], hp["temperature"], hp["embed_size"],
        hp["UI_layers"], hp["logq_scale"], int(hp["use_reduce_lr"]),
        row["runtime_h"], ok,
    )

    completed = [r["recall@20"] for r in _state["trial_rows"]
                 if r["recall@20"] is not None]
    _wb_ctrl.log({
        "optuna/trial_recall":   (r20 if r20 is not None else float("nan")),
        "optuna/trial_ndcg":     (n20 if n20 is not None else float("nan")),
        "optuna/running_best":   (max(completed) if completed else float("nan")),
        "optuna/n_completed":    len(completed),
        "optuna/trial_pruned":   int(pruned),
        "optuna/trial_runtime_h": row["runtime_h"],
        "trial_number":          trial.number,
    })

    if pruned:
        print(f"  -> PRUNED (Hyperband)  best-so-far R@20="
              f"{max(completed) if completed else float('nan'):.6f}", flush=True)
        raise optuna.TrialPruned()

    if r20 is None:
        print(f"  [WARN] rc={proc.returncode} — no recall parsed")
        print(log[-800:])
        return float("-inf")

    print(f"  -> R@20={r20:.6f}  N@20={(n20 or 0):.6f}  "
          f"peak_ep={row['val_recall_peak_epoch']}  time={row['runtime_h']:.2f}h  OK",
          flush=True)
    return float(r20)


# ---- 7. Build study + run --------------------------------------------
sampler = TPESampler(seed=SAMPLER_SEED, multivariate=True, constant_liar=True)
pruner = (HyperbandPruner(min_resource=MIN_RESOURCE, max_resource=EPOCH,
                          reduction_factor=REDUCTION_FACTOR)
          if STREAM_PRUNING else NopPruner())

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    sampler=sampler,
    pruner=pruner,
    direction="maximize",
    load_if_exists=True,          # resumable after a crash
)

print("=" * 72)
print(f"OPTUNA SWEEP on {_DATASET}: {N_TRIALS} trials x {N_SEEDS} seed  "
      f"(objective = BEST_Test_Recall@20)")
print(f"  study={STUDY_NAME}  storage={STORAGE}")
print(f"  sampler=TPE(multivariate,constant_liar)  "
      f"pruner={'Hyperband' if STREAM_PRUNING else 'None'}")
print("=" * 72, flush=True)

_already = len([t for t in study.trials
                if t.state.is_finished()])
_remaining = max(0, N_TRIALS - _already)
if _already:
    print(f"[resume] {_already} finished trials already in study; "
          f"running {_remaining} more.", flush=True)

study.optimize(objective, n_trials=_remaining, gc_after_trial=True)

# ---- 8. Leaderboard (completed trials, by Recall@20 desc) ------------
def _rows_sorted() -> list[dict[str, Any]]:
    rows = [r for r in _state["trial_rows"]]
    rows.sort(key=lambda r: (r["recall@20"] if r["recall@20"] is not None
                             else float("-inf")), reverse=True)
    return rows

ranked = _rows_sorted()
COMPLETE = [r for r in ranked if r["state"] == "COMPLETE"]

print("\n" + "=" * 104)
print(f"CLOTHING OPTUNA LEADERBOARD  --  {len(COMPLETE)} complete / "
      f"{len(_state['trial_rows'])} run  (sorted by Test Recall@20 desc)")
print("=" * 104)
print(f"{'Rank':>4}  {'trial':>5}  {'state':>8}  {'R@20':>10}  {'N@20':>10}  "
      f"{'lr':>9}  {'bs':>5}  {'regs':>9}  {'lv':>7}  {'logq':>6}")
print("-" * 104)
for rank, r in enumerate(ranked, start=1):
    hp = r["hp"]
    rr = r["recall@20"] if r["recall@20"] is not None else float("nan")
    nn = r["ndcg@20"] if r["ndcg@20"] is not None else float("nan")
    print(f"{rank:>4}  {r['trial']:>5}  {r['state']:>8}  {rr:>10.6f}  {nn:>10.6f}  "
          f"{hp['lr']:>9.2e}  {hp['batch_size']:>5}  {hp['regs']:>9.2e}  "
          f"{hp['lambda_view']:>7.4f}  {hp['logq_scale']:>6.3f}")
print("=" * 104)

# ---- 9. Best trial + persist -----------------------------------------
best_params = dict(study.best_params) if study.best_trial else {}
best_value = float(study.best_value) if study.best_trial else float("nan")
best_number = study.best_trial.number if study.best_trial else None
print(f"\nBEST trial: #{best_number}   Test-selected val R@20 (objective) = {best_value:.6f}")
print(f"  best_params = {json.dumps(best_params, indent=2)}")

os.makedirs("results", exist_ok=True)
best_json = os.path.join("results", "branchA_clothing_v1_best.json")
with open(best_json, "w", encoding="utf-8") as f:
    json.dump({
        "dataset":       _DATASET,
        "study_name":    STUDY_NAME,
        "n_trials":      N_TRIALS,
        "n_seeds":       N_SEEDS,
        "search_space":  "clothing_adapted_v1",
        "best_number":   best_number,
        "best_value":    best_value,
        "best_params":   best_params,
        "frozen_flags":  frozen_flags(),
    }, f, indent=2)

full_json = os.path.join("results", "branchA_clothing_v1_trials.json")
with open(full_json, "w", encoding="utf-8") as f:
    json.dump({"dataset": _DATASET, "study_name": STUDY_NAME,
               "trials": _state["trial_rows"], "ranked_order":
               [r["trial"] for r in ranked]}, f, indent=2)
print(f"\n[Clothing-Optuna] best  -> {best_json}")
print(f"[Clothing-Optuna] trials -> {full_json}")

# ---- 10. W&B final summary, leaderboard Table & Artifact -------------
_lb_cols = ["rank", "trial", "state", "recall@20", "ndcg@20",
            "lr", "batch_size", "regs", "simgcl_eps", "lambda_view",
            "temperature", "embed_size", "UI_layers", "logq_scale", "use_reduce_lr"]
_lb_table = wandb.Table(columns=_lb_cols)
for rank, r in enumerate(ranked, start=1):
    hp = r["hp"]
    _lb_table.add_data(
        rank, r["trial"], r["state"], r["recall@20"], r["ndcg@20"],
        hp["lr"], hp["batch_size"], hp["regs"], hp["simgcl_eps"],
        hp["lambda_view"], hp["temperature"], hp["embed_size"],
        hp["UI_layers"], hp["logq_scale"], int(hp["use_reduce_lr"]),
    )

_wb_ctrl.log({
    "optuna/all_trials_table": _wb_table,
    "optuna/leaderboard":      _lb_table,
})
_wb_ctrl.summary.update({
    "optuna/best_trial":   best_number,
    "optuna/best_value":   best_value,
    "optuna/n_trials":     N_TRIALS,
    "optuna/n_complete":   len(COMPLETE),
    "optuna/n_pruned":     sum(1 for r in _state["trial_rows"] if r["state"] == "PRUNED"),
    "optuna/n_failed":     sum(1 for r in _state["trial_rows"] if r["state"] == "FAIL"),
})
for k, v in best_params.items():
    _wb_ctrl.summary[f"optuna/best_{k}"] = v

_artifact = wandb.Artifact(
    name=f"branchA_optuna_{_DATASET}",
    type="optuna_study",
    description=(
        f"Branch A Optuna sweep on {_DATASET}: {N_TRIALS} trials x 1 seed, "
        f"clothing-adapted search space. Objective = BEST_Test_Recall@20."
    ),
    metadata={"best_trial": best_number, "best_value": best_value},
)
_artifact.add_file(best_json,  name="branchA_clothing_v1_best.json")
_artifact.add_file(full_json,  name="branchA_clothing_v1_trials.json")
if os.path.exists("optuna_branchA_clothing.db"):
    _artifact.add_file("optuna_branchA_clothing.db", name="optuna_branchA_clothing.db")
_wb_ctrl.log_artifact(_artifact)

_wb_ctrl.finish()
print(f"[W&B] Controller run finished: {_wb_ctrl.url}")
print("\nNext: pick the top trials from the leaderboard and run the "
      "3-seed x 250-epoch FINAL evaluation (as in the Sports \u00a79.6 cell).")


### 9.8 Clothing Top-8 (by NDCG@20) — FINAL Evaluation (3 seeds × 250 epochs)

Automatically reads the Section 9.7 sweep results (`results/branchA_clothing_v1_trials.json`), selects the **8 configurations with the strongest single-seed NDCG@20**, then re-runs each with **3 seeds × 250 epochs**. Per-config we report mean ± std of NDCG@20 and Recall@20 and rank the 8 configs by **mean_NDCG@20 × mean_Recall@20** (descending) — the same scalar as the Sports §9.6 Top-11 final.

- **Selection metric:** NDCG@20 (complements the sweep's Recall@20 objective — guards against Recall-only over-fitting on the sparse Clothing graph).
- **Frozen backbone:** identical to §9.1 / §9.7 (apc_off_combined + LogQ laplace/β=1.0/clip=5.0 + SimGCL batch-N + view_every_k=2). Only the Optuna-searched HPs vary per config.
- **Full W&B monitoring:** one long-lived controller run (rolling + final aggregates, a per-(config, seed) Table, a ranked leaderboard Table and a JSON Artifact) plus a per-epoch live run for every (config, seed) via `--use_wandb 1`.
- **Output:** `results/clothing_top8_final_results.json`.

> Run **after** §9.7 has completed at least a few un-pruned trials — this cell raises a clear error if the trials JSON is missing.

In [ ]:
# =====================================================================
# Section 9.8 -- Clothing Top-8 (by NDCG@20) FINAL Evaluation
#                3 seeds x 250 epochs  |  Full WandB monitoring
#
#   Goal
#   ----
#   Read the Clothing Optuna sweep results produced by Section 9.7
#   (results/branchA_clothing_v1_trials.json), AUTOMATICALLY pick the
#   TOP-8 configurations with the STRONGEST single-seed NDCG@20, then
#   re-evaluate each one with 3 seeds x 250 epochs. Per-config we compute
#   mean +/- std of NDCG@20 and Recall@20, and rank the 8 configs by
#       (mean_NDCG@20 * mean_Recall@20)   descending
#   -- the same scalar used for the Sports Top-11 final (Section 9.6).
#
#   Why NDCG@20 for selection?
#   --------------------------
#   Section 9.7's Optuna objective maximised Recall@20. Selecting the
#   final short-list by NDCG@20 (as requested) surfaces configs that not
#   only retrieve relevant items but RANK them well -- a complementary
#   signal that guards against Recall-only over-fitting on the sparse
#   Clothing graph.
#
#   Full WandB monitoring (mirrors Section 9.1 / 9.6)
#   -------------------------------------------------
#     * one long-lived CONTROLLER run (rolling + final aggregates, a W&B
#       Table of every (config, seed) row, a ranked leaderboard Table and
#       a W&B Artifact holding clothing_top8_final_results.json)
#     * each (config, seed) sub-process gets its OWN per-epoch W&B run via
#       `--use_wandb 1` streamed live from train.py
#
#   Frozen backbone == Section 9.1 / 9.7 (apc_off_combined + LogQ laplace
#   beta=1.0 clip=5.0 + SimGCL batch-N + view_every_k=2). Only the
#   Optuna-searched HPs vary per config.
# =====================================================================
from __future__ import annotations

import json
import math
import os
import re
import subprocess
import sys
import time
from typing import Any, Optional

import wandb

# ---- 0. Resolve dirs / globals (reuse from earlier cells if set) ------
try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PYTHON = sys.executable
try:
    _DATASET = dataset  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _DATASET = "Clothing"
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_ENTITY = ""

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---- 1. Evaluation settings -------------------------------------------
TOP_K: int = 8                # short-list the 8 strongest configs
SELECT_METRIC: str = "ndcg@20"   # <-- selection criterion (per request)
N_SEEDS: int = 3
EPOCH: int = 250
PATIENCE: int = 20
MIN_EPOCHS: int = 75
WB_GROUP: str = "wave2_clothing_top8_final"
WB_TAGS: str = "clothing_top8_final,3seed,rev55,logq,optuna"

# Source produced by Section 9.7 (relative to the DAMPS project root)
TRIALS_JSON: str = os.path.join("results", "branchA_clothing_v1_trials.json")
STUDY_NAME: str = "branchA_clothing_v1"

# Frozen LogQ settings (identical to Section 9.1 / 9.7)
LOGQ_MODE: str = "laplace"
LOGQ_BETA: float = 1.0
LOGQ_CLIP: float = 5.0

# ---- 1a. 3-seed list (reuse notebook seed bank if present) -----------
def _as_int_seed(s: Any) -> int:
    return int(round(float(s)))

try:
    SEEDS = [_as_int_seed(s) for s in list(seeds)[:N_SEEDS]]  # type: ignore[name-defined]  # noqa: F821
    if len(SEEDS) < N_SEEDS:
        raise NameError
except NameError:
    SEEDS = [255808013, 686087737, 1079144366]

# ---- 2. Auto-load Section 9.7 trials & pick Top-8 by NDCG@20 ---------
if not os.path.exists(TRIALS_JSON):
    raise FileNotFoundError(
        f"Cannot find {TRIALS_JSON!r}. Run Section 9.7 (the Clothing Optuna "
        f"sweep) first -- it writes this file with every trial's metrics."
    )

with open(TRIALS_JSON, "r", encoding="utf-8") as _f:
    _sweep = json.load(_f)

_all_trials: list[dict[str, Any]] = _sweep.get("trials", [])
print(f"[load] {TRIALS_JSON}: {len(_all_trials)} trial rows "
      f"(study={_sweep.get('study_name', STUDY_NAME)})")

# Keep only COMPLETE trials that actually have a usable NDCG@20 and the
# HP dict needed to replay them.
def _usable(t: dict[str, Any]) -> bool:
    return (
        t.get("state") == "COMPLETE"
        and t.get(SELECT_METRIC) is not None
        and isinstance(t.get("hp"), dict)
        and t.get("recall@20") is not None
    )

_candidates = [t for t in _all_trials if _usable(t)]
if not _candidates:
    raise RuntimeError(
        "No COMPLETE trials with a valid NDCG@20 were found in "
        f"{TRIALS_JSON!r}. Check that Section 9.7 finished at least one "
        "un-pruned trial."
    )

# Sort by the selection metric (NDCG@20) descending, keep the strongest K.
_candidates.sort(key=lambda t: float(t[SELECT_METRIC]), reverse=True)
_selected = _candidates[:TOP_K]
_k_eff = len(_selected)

print(f"[select] {len(_candidates)} COMPLETE trials -> picking top "
      f"{_k_eff} by {SELECT_METRIC} (requested {TOP_K})")
print(f"{'rank':>4}  {'trial':>5}  {'sweep N@20':>11}  {'sweep R@20':>11}")
print("-" * 40)
for _i, _t in enumerate(_selected, start=1):
    print(f"{_i:>4}  {_t['trial']:>5}  {float(_t['ndcg@20']):>11.6f}  "
          f"{float(_t['recall@20']):>11.6f}")

# Normalise selected trials into the config schema used by the eval loop.
def _to_config(t: dict[str, Any]) -> dict[str, Any]:
    hp = dict(t["hp"])
    cfg: dict[str, Any] = {
        "name":          f"t{int(t['trial']):04d}",
        "trial":         int(t["trial"]),
        "sweep_ndcg@20": float(t["ndcg@20"]),
        "sweep_recall@20": float(t["recall@20"]),
        "lr":            float(hp["lr"]),
        "batch_size":    int(hp["batch_size"]),
        "regs":          float(hp["regs"]),
        "simgcl_eps":    float(hp["simgcl_eps"]),
        "lambda_view":   float(hp["lambda_view"]),
        "temperature":   float(hp["temperature"]),
        "embed_size":    int(hp["embed_size"]),
        "UI_layers":     int(hp["UI_layers"]),
        "logq_scale":    float(hp["logq_scale"]),
        "use_reduce_lr": int(hp.get("use_reduce_lr", 0)),
    }
    if int(cfg["use_reduce_lr"]) == 1:
        cfg["reduce_lr_factor"]   = float(hp.get("reduce_lr_factor", 0.5))
        cfg["reduce_lr_patience"] = int(hp.get("reduce_lr_patience", 3))
    return cfg

TOP_CONFIGS: list[dict[str, Any]] = [_to_config(t) for t in _selected]

# ---- 3. Frozen backbone flags (invariant across all configs) ----------
#   Byte-for-byte aligned with Section 9.1 / 9.7 BACKBONE_FLAGS.
FROZEN_FLAGS: list[str] = [
    "--dataset",                     str(_DATASET),
    "--gpu_id",                      "0",
    "--epoch",                       str(EPOCH),
    "--verbose",                     "5",
    "--clip_grad_norm",              "1.0",
    "--topk",                        "5",
    "--core",                        "5",
    "--User_layers",                 "3",
    "--Item_layers",                 "2",
    "--user_loss_ratio",             "0.03",
    "--item_loss_ratio",             "0.07",
    "--learnable_tau",               "0",
    "--early_stopping_patience",     str(PATIENCE),
    "--early_stopping_min_epochs",   str(MIN_EPOCHS),
    "--early_stopping_min_delta",    "0.0001",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    "--reduce_lr_min",               "1e-06",
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_permutation_fft",       "0",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_num_categories",        "10",
    "--damps_warmup_epochs",         "10",
    "--rebuild_R",                   "5",
    "--faiss_threshold",             "60000",
    "--faiss_use_gpu",               "1",
    "--knn_chunk_size",              "4096",
    "--knn_efsearch",                "64",
    "--use_amp",                     "1",
    "--enable_logq",                 "1",
    "--logq_mode",                   LOGQ_MODE,
    "--logq_beta",                   str(LOGQ_BETA),
    "--logq_clip",                   str(LOGQ_CLIP),
    "--enable_simgcl",               "1",
    "--simgcl_batch_size_user",      "2048",
    "--simgcl_batch_size_item",      "2048",
    "--branchA_view_every_k",        "2",
    "--branchA_bcl_batchn",          "1",
    "--branchA_view_bsz",            "2048",
    "--branchA_bcl_bsz",             "2048",
]


# ---- 4. Helper: build config-specific (Optuna-searched) HP flags -----
def _config_hp_flags(cfg: dict[str, Any]) -> list[str]:
    """Return only the CLI flags that vary between configs."""
    flags: list[str] = [
        "--lr",            str(cfg["lr"]),
        "--batch_size",    str(cfg["batch_size"]),
        "--regs",          str(cfg["regs"]),
        "--simgcl_eps",    str(cfg["simgcl_eps"]),
        "--lambda_view",   str(cfg["lambda_view"]),
        "--temperature",   str(cfg["temperature"]),
        "--embed_size",    str(cfg["embed_size"]),
        "--UI_layers",     str(cfg["UI_layers"]),
        "--logq_scale",    str(cfg["logq_scale"]),
        "--use_reduce_lr", str(int(cfg["use_reduce_lr"])),
    ]
    if int(cfg.get("use_reduce_lr", 0)):
        flags += [
            "--reduce_lr_factor",   str(cfg.get("reduce_lr_factor", 0.5)),
            "--reduce_lr_patience", str(cfg.get("reduce_lr_patience", 3)),
        ]
    return flags


# ---- 5. Metric parser (same contract as Section 9.1 / 9.7) -----------
_S = r"[^0-9\-]*(-?[0-9]*\.?[0-9]+)"
_PAT: dict[str, "re.Pattern[str]"] = {
    "recall@20":             re.compile(r"BEST_Test_Recall@20"        + _S),
    "ndcg@20":               re.compile(r"BEST_Test_NDCG@20"          + _S),
    "precision@20":          re.compile(r"BEST_Test_Precision@20"     + _S),
    "val_recall@20":         re.compile(r"BEST_Val_Recall@20"         + _S),
    "val_ndcg@20":           re.compile(r"BEST_Val_NDCG@20"           + _S),
    "val_recall_peak_epoch": re.compile(r"BEST_Val_Recall_Peak_Epoch" + _S),
}


def _parse_metrics(text: str) -> dict[str, float]:
    out: dict[str, float] = {}
    for k, pat in _PAT.items():
        hits = pat.findall(text)
        if hits:
            out[k] = float(hits[-1])
    return out


def _mean(xs: list[float]) -> float:
    return sum(xs) / len(xs) if xs else float("nan")


def _std(xs: list[float]) -> float:
    """Sample standard deviation (ddof=1); 0.0 for a single value."""
    if len(xs) < 2:
        return 0.0
    m = _mean(xs)
    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5


def _fnum(x: Optional[float]) -> float:
    return float(x) if x is not None else float("nan")


# ---- 6. W&B controller run (mirrors Section 9.1 / 9.6 controller) ----
print("[W&B] Initialising Clothing Top-8 FINAL evaluation controller run ...")
_wb_ctrl = wandb.init(
    project=_WB_PROJECT,
    entity=_WB_ENTITY if _WB_ENTITY else None,
    name="clothing_top8_final_controller",
    group=WB_GROUP,
    tags=["clothing_top8_final", "controller", "rev55", "optuna"],
    job_type="eval_controller",
    config={
        "phase":         "clothing_top8_final",
        "dataset":       _DATASET,
        "backbone":      "apc_off_combined",
        "select_metric": SELECT_METRIC,
        "top_k":         TOP_K,
        "n_configs":     len(TOP_CONFIGS),
        "n_seeds":       N_SEEDS,
        "seeds":         SEEDS,
        "epoch":         EPOCH,
        "patience":      PATIENCE,
        "min_epochs":    MIN_EPOCHS,
        "logq_mode":     LOGQ_MODE,
        "logq_beta":     LOGQ_BETA,
        "logq_clip":     LOGQ_CLIP,
        "source_study":  STUDY_NAME,
        "source_json":   TRIALS_JSON,
        "sort_metric":   "mean_ndcg@20 * mean_recall@20",
    },
    reinit=True,
)
print(f"[W&B] Controller run: {_wb_ctrl.url}")

# W&B Table -- one row per (config, seed)
_WB_COLS = [
    "config_name", "trial_id", "seed",
    "recall@20", "ndcg@20", "precision@20",
    "val_recall@20", "val_ndcg@20", "val_recall_peak_epoch",
    "sweep_ndcg@20", "runtime_h", "ok",
]
_wb_table = wandb.Table(columns=_WB_COLS)

# ---- 7. Main evaluation loop -----------------------------------------
all_config_results: list[dict[str, Any]] = []

print(f"\nRunning {len(TOP_CONFIGS)} configs x {N_SEEDS} seeds x {EPOCH} epochs")
print(f"Selection: top-{TOP_K} by {SELECT_METRIC} from {STUDY_NAME}")
print(f"Seeds: {SEEDS}")

_global_run_idx = 0
for cfg_idx, cfg in enumerate(TOP_CONFIGS, start=1):
    cfg_name: str = cfg["name"]
    trial_id: int = cfg["trial"]

    print(f"\n{'='*72}")
    print(f"[{cfg_idx}/{len(TOP_CONFIGS)}] Config {cfg_name}  (trial={trial_id}, "
          f"sweep N@20={cfg['sweep_ndcg@20']:.6f})")
    print(f"  lr={cfg['lr']:.6g}  bs={cfg['batch_size']}  regs={cfg['regs']:.2e}  "
          f"eps={cfg['simgcl_eps']:.3f}  lv={cfg['lambda_view']:.4f}")
    print(f"  T={cfg['temperature']:.3f}  embed={cfg['embed_size']}  "
          f"UI_layers={cfg['UI_layers']}  logq={cfg['logq_scale']:.3f}  "
          f"reduce_lr={cfg['use_reduce_lr']}")
    print(f"{'='*72}", flush=True)

    cfg_runs: list[dict[str, Any]] = []
    hp_flags = _config_hp_flags(cfg)

    for seed_idx, seed in enumerate(SEEDS, start=1):
        _global_run_idx += 1
        seed_int = int(seed)
        abl_tag = f"clothing_top8_t{trial_id:04d}_seed{seed_int}"
        seed_tags = f"{WB_TAGS},trial{trial_id},seed{seed_int}"
        wandb_args: list[str] = [
            "--use_wandb",       "1",
            "--wandb_project",   _WB_PROJECT,
            "--wandb_group",     WB_GROUP,
            "--wandb_tags",      seed_tags,
            "--wandb_job_type",  "clothing_top8_eval_seed",
            "--wandb_run_name",  abl_tag,
        ]
        if _WB_ENTITY:
            wandb_args += ["--wandb_entity", _WB_ENTITY]

        cmd: list[str] = [
            _PYTHON, "train.py",
            *FROZEN_FLAGS,
            *hp_flags,
            *wandb_args,
            "--seed",            str(seed_int),
            "--ablation_target", abl_tag,
        ]

        print(f"\n  [{seed_idx}/{N_SEEDS}] seed={seed_int}  run={abl_tag!r}", flush=True)
        t0 = time.time()
        env = os.environ.copy()
        env["PYTHONIOENCODING"] = "utf-8"
        env["PYTHONUNBUFFERED"] = "1"

        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            cwd=_DAMPS_DIR,
            env=env,
        )
        dt = time.time() - t0
        log = (proc.stdout or "") + "\n" + (proc.stderr or "")
        m = _parse_metrics(log)
        ok = ("recall@20" in m) and ("ndcg@20" in m) and (proc.returncode == 0)
        r20 = m.get("recall@20")
        n20 = m.get("ndcg@20")

        if not ok:
            print(f"  [WARN] seed={seed_int}  rc={proc.returncode}  "
                  f"(metrics parsed: {sorted(m.keys())})")
            print(log[-800:])

        run_result: dict[str, Any] = {
            "config_name":           cfg_name,
            "trial":                 trial_id,
            "seed":                  seed_int,
            "recall@20":             r20,
            "ndcg@20":               n20,
            "precision@20":          m.get("precision@20"),
            "val_recall@20":         m.get("val_recall@20"),
            "val_ndcg@20":           m.get("val_ndcg@20"),
            "val_recall_peak_epoch": (int(m["val_recall_peak_epoch"])
                                      if "val_recall_peak_epoch" in m else None),
            "runtime_s":             round(dt, 1),
            "runtime_h":             round(dt / 3600.0, 2),
            "ok":                    ok,
        }
        cfg_runs.append(run_result)

        # -- Rolling log to controller after each completed seed --------
        done_r = [r["recall@20"] for r in cfg_runs if r["recall@20"] is not None]
        done_n = [r["ndcg@20"]   for r in cfg_runs if r["ndcg@20"]   is not None]
        _wb_ctrl.log({
            f"cfg_t{trial_id:04d}/seed_recall":  _fnum(r20),
            f"cfg_t{trial_id:04d}/seed_ndcg":    _fnum(n20),
            f"cfg_t{trial_id:04d}/rolling_mean_recall": _mean(done_r),
            f"cfg_t{trial_id:04d}/rolling_mean_ndcg":   _mean(done_n),
            f"cfg_t{trial_id:04d}/n_completed":  len(done_r),
            "global_run_idx":                    _global_run_idx,
        })

        _wb_table.add_data(
            cfg_name, trial_id, seed_int,
            r20, n20, m.get("precision@20"),
            m.get("val_recall@20"), m.get("val_ndcg@20"),
            run_result["val_recall_peak_epoch"],
            cfg["sweep_ndcg@20"],
            round(dt / 3600.0, 2), ok,
        )

        _rs = f"R@20={r20:.6f}" if r20 is not None else "R@20=N/A"
        _ns = f"N@20={n20:.6f}" if n20 is not None else "N@20=N/A"
        print(f"    -> {_rs}  {_ns}  "
              f"P@20={m.get('precision@20', float('nan')):.6f}  "
              f"peak_ep={run_result['val_recall_peak_epoch']}  "
              f"time={dt/3600:.2f}h  {'OK' if ok else 'WARN'}")

    # ---- Per-config aggregate stats (mean +/- std) -------------------
    recalls = [r["recall@20"] for r in cfg_runs if r["recall@20"] is not None]
    ndcgs   = [r["ndcg@20"]   for r in cfg_runs if r["ndcg@20"]   is not None]
    precs   = [r["precision@20"] for r in cfg_runs if r["precision@20"] is not None]
    mean_r, std_r = _mean(recalls), _std(recalls)
    mean_n, std_n = _mean(ndcgs), _std(ndcgs)

    # Sort key = product of the two means. NaN (failed config) sinks to
    # the bottom deterministically via -inf.
    if math.isnan(mean_r) or math.isnan(mean_n):
        sort_key = float("-inf")
    else:
        sort_key = mean_r * mean_n

    cfg_summary: dict[str, Any] = {
        "config_name":   cfg_name,
        "trial":         trial_id,
        "sweep_ndcg@20": cfg["sweep_ndcg@20"],
        "sweep_recall@20": cfg["sweep_recall@20"],
        "hps":           {k: v for k, v in cfg.items()
                          if k not in ("name", "trial", "sweep_ndcg@20", "sweep_recall@20")},
        "seeds":         SEEDS,
        "n_seeds":       N_SEEDS,
        "runs":          cfg_runs,
        "mean_recall":   round(mean_r, 6) if not math.isnan(mean_r) else None,
        "std_recall":    round(std_r,  6) if not math.isnan(std_r)  else None,
        "mean_ndcg":     round(mean_n, 6) if not math.isnan(mean_n) else None,
        "std_ndcg":      round(std_n,  6) if not math.isnan(std_n)  else None,
        "mean_prec":     round(_mean(precs), 6) if precs else None,
        "sort_key":      (round(sort_key, 10) if math.isfinite(sort_key) else None),
        "n_ok":          sum(1 for r in cfg_runs if r["ok"]),
    }
    all_config_results.append(cfg_summary)

    print(f"\n  Summary {cfg_name}:")
    print(f"    mean R@20 = {mean_r:.6f} +/- {std_r:.6f}   (n_ok={cfg_summary['n_ok']}/{N_SEEDS})")
    print(f"    mean N@20 = {mean_n:.6f} +/- {std_n:.6f}")
    print(f"    sort_key (mean_N x mean_R) = "
          f"{sort_key:.10f}" if math.isfinite(sort_key) else
          "    sort_key (mean_N x mean_R) = N/A (config failed)")

    _wb_ctrl.log({
        f"cfg_t{trial_id:04d}/final_mean_recall": _fnum(cfg_summary["mean_recall"]),
        f"cfg_t{trial_id:04d}/final_std_recall":  _fnum(cfg_summary["std_recall"]),
        f"cfg_t{trial_id:04d}/final_mean_ndcg":   _fnum(cfg_summary["mean_ndcg"]),
        f"cfg_t{trial_id:04d}/final_std_ndcg":    _fnum(cfg_summary["std_ndcg"]),
        f"cfg_t{trial_id:04d}/sort_key":          _fnum(cfg_summary["sort_key"]),
        "cfg_progress": cfg_idx,
    })

# ---- 8. Rank by mean_NDCG@20 * mean_Recall@20 (descending) ----------
all_config_results.sort(
    key=lambda c: (c["sort_key"] if c["sort_key"] is not None else float("-inf")),
    reverse=True,
)

# ---- 9. Print final leaderboard --------------------------------------
COL_W = 8
print()
print("=" * 116)
print(f"CLOTHING TOP-{TOP_K} FINAL LEADERBOARD  --  short-listed by {SELECT_METRIC}, "
      f"ranked by  mean_NDCG@20 x mean_Recall@20  (descending)")
print("=" * 116)
print(f"{'Rank':>4}  {'Config':<{COL_W}}  {'trial':>5}  "
      f"{'R@20 mean':>10}  {'R@20 std':>9}  "
      f"{'N@20 mean':>10}  {'N@20 std':>9}  {'N x R':>14}  {'sweep N@20':>10}")
print("-" * 116)
for rank, c in enumerate(all_config_results, start=1):
    mr = c["mean_recall"] if c["mean_recall"] is not None else float("nan")
    sr = c["std_recall"]  if c["std_recall"]  is not None else float("nan")
    mn = c["mean_ndcg"]   if c["mean_ndcg"]   is not None else float("nan")
    sn = c["std_ndcg"]    if c["std_ndcg"]    is not None else float("nan")
    sk = c["sort_key"]    if c["sort_key"]    is not None else float("nan")
    print(f"{rank:>4}  {c['config_name']:<{COL_W}}  {c['trial']:>5}  "
          f"{mr:>10.6f}  {sr:>9.6f}  "
          f"{mn:>10.6f}  {sn:>9.6f}  {sk:>14.10f}  {c['sweep_ndcg@20']:>10.6f}")
print("=" * 116)

best = all_config_results[0]
print(f"\nBest config: {best['config_name']}  (trial {best['trial']})")
print(f"  mean R@20 = {best['mean_recall']} +/- {best['std_recall']}")
print(f"  mean N@20 = {best['mean_ndcg']} +/- {best['std_ndcg']}")
print(f"  N x R     = {best['sort_key']}")
print(f"  HPs       = {best['hps']}")

# ---- 10. Persist JSON ------------------------------------------------
os.makedirs("results", exist_ok=True)
out_json = os.path.join("results", "clothing_top8_final_results.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "dataset":        _DATASET,
            "source_study":   STUDY_NAME,
            "source_json":    TRIALS_JSON,
            "select_metric":  SELECT_METRIC,
            "top_k":          TOP_K,
            "n_configs":      len(TOP_CONFIGS),
            "n_seeds":        N_SEEDS,
            "seeds":          SEEDS,
            "epoch":          EPOCH,
            "sort_metric":    "mean_ndcg@20 * mean_recall@20 (descending)",
            "configs_ranked": all_config_results,
            "best_config":    best,
        },
        f, indent=2,
    )
print(f"\n[Clothing-Top8] results -> {out_json}")

# ---- 11. W&B final summary, leaderboard Table & Artifact -------------
_lb_cols = ["rank", "config_name", "trial", "mean_recall", "std_recall",
            "mean_ndcg", "std_ndcg", "sort_key", "sweep_ndcg@20", "n_ok"]
_lb_table = wandb.Table(columns=_lb_cols)
for rank, c in enumerate(all_config_results, start=1):
    _lb_table.add_data(
        rank, c["config_name"], c["trial"],
        c["mean_recall"], c["std_recall"],
        c["mean_ndcg"], c["std_ndcg"], c["sort_key"],
        c["sweep_ndcg@20"], c["n_ok"],
    )

_wb_ctrl.log({
    "clothing_top8/all_seeds_table": _wb_table,
    "clothing_top8/leaderboard":     _lb_table,
})
_wb_ctrl.summary.update({
    "clothing_top8/best_config_name": best["config_name"],
    "clothing_top8/best_trial":       best["trial"],
    "clothing_top8/best_mean_recall": best["mean_recall"],
    "clothing_top8/best_std_recall":  best["std_recall"],
    "clothing_top8/best_mean_ndcg":   best["mean_ndcg"],
    "clothing_top8/best_std_ndcg":    best["std_ndcg"],
    "clothing_top8/best_sort_key":    best["sort_key"],
    "clothing_top8/n_configs":        len(all_config_results),
    "clothing_top8/n_seeds":          N_SEEDS,
    "clothing_top8/select_metric":    SELECT_METRIC,
})

_artifact = wandb.Artifact(
    name=f"clothing_top8_final_{_DATASET}",
    type="eval_results",
    description=(
        f"Clothing Top-{TOP_K} (by {SELECT_METRIC}) 3-seed x {EPOCH}ep final "
        f"evaluation. Short-listed from {STUDY_NAME}, ranked by "
        f"mean_NDCG@20 x mean_Recall@20."
    ),
    metadata={
        "best_config":   best["config_name"],
        "best_trial":    best["trial"],
        "best_sort_key": best["sort_key"],
        "select_metric": SELECT_METRIC,
        "top_k":         TOP_K,
    },
)
_artifact.add_file(out_json, name="clothing_top8_final_results.json")
_wb_ctrl.log_artifact(_artifact)

_wb_ctrl.finish()
print(f"[W&B] Controller run finished: {_wb_ctrl.url}")


### 9.9 PACER Tercile Wrapper — Smoke Test (1 seed × 10 epochs)

Sanity check that `main_tercile.py` (the PACER-aware tercile wrapper)
compiles, imports `train`, monkey-patches `Trainer.test` on PACER's
dict-returning `forward()`, and emits `[tercile-final]` /
`[tercile-test-final]` lines with finite Head / Mid / Tail values.
Fails fast (`AssertionError`) if any check fails — RUN THIS BEFORE §9.10.


In [6]:
# =====================================================================
# Section 9.9 — SMOKE-TEST for PACER tercile wrapper (1 seed × 10 epochs)
#
#   Purpose
#   -------
#   Before launching the expensive 5-seed × 250-epoch RQ4 run (§9.10),
#   verify that `main_tercile.py` — the PACER-aware tercile wrapper —
#   compiles, imports `train`, monkey-patches `Trainer.test` correctly
#   on PACER's dict-returning forward pass, and emits the two
#   parser-friendly summary lines with non-NaN Head / Mid / Tail values.
#
#   Success criteria
#   ----------------
#     * subprocess exit code == 0
#     * `[tercile-final]`      line present with 3 finite floats
#     * `[tercile-test-final]` line present with 3 finite floats
#
#   If any check fails, prints [SMOKE-FAIL] with the offending symptom
#   and raises AssertionError so you catch it BEFORE the 5-seed run.
# =====================================================================
from __future__ import annotations

import json
import math
import os
import re
import subprocess
import sys
import time
from typing import Any

# ---- 0. Resolve dirs / globals (reuse from earlier cells if set) --------
try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PYTHON = sys.executable
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_ENTITY = "baitapck51cc-uet"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)
os.makedirs("results", exist_ok=True)

# ---- 1. Emit main_tercile.py (PACER-aware wrapper) ----------------------
#     Sits next to train.py inside MMHCL_DAMPS_Project/. Re-emitted every
#     time this cell runs so an out-of-date wrapper on disk cannot mask a
#     fresh smoke test.
_MAIN_TERCILE = r'''"""main_tercile.py  ──  PACER (DAMPS-MMHCL) tercile-recall wrapper.
================================================================

Drop-in replacement for ``MMHCL_DAMPS_Project/train.py`` that adds four
things on top of the stock PACER training loop:

  1. On every VAL evaluation, computes Recall@20 restricted to items in
     the Head / Mid / Tail popularity tercile (by training frequency).
  2. Whenever val_recall@20 hits a new peak (matching PACER's
     ``BEST_Test_Recall@20`` semantics — see ``train.py`` line 801),
     snapshots both:
         - VAL Head/Mid/Tail at that epoch      (``_best_val_tercile``)
         - TEST Head/Mid/Tail at that epoch     (``_best_test_tercile``)
     independently of WandB.
  3. On every epoch where PACER logs ``val/recall@20`` (line 762) or
     ``test/recall@20`` (line 833) to WandB, we also log:
         val/recall@20_Head|Mid|Tail
         test/recall@20_Head|Mid|Tail
     and at the end of training we write the "best" snapshots into
     ``wandb.summary`` so the run's Overview panel shows them.
  4. Two parser-friendly summary lines are printed at end of run:
         [tercile-final]     BEST_Recall@20_Head=..  Mid=..  Tail=..
         [tercile-test-final] BEST_Test_Recall@20_Head=..  Mid=..  Tail=..

The wrapper does NOT touch the training math -- it only reads the final
user/item embeddings (via a single extra forward pass under ``model.eval``
with ``update_momentum=False``) and computes tercile recall in Python.

Usage (drop-in for train.py; expects the same CLI):
    python main_tercile.py --dataset Clothing --seed <s> \\
        --enable_logq 1 --enable_simgcl 1 [ ... ]
"""
from __future__ import annotations

import math
import os
from typing import Any as _Any

import numpy as np
import torch

# --- 1. Import PACER's train module ---------------------------------------
# Importing ``train`` triggers:
#   * ``utility.parser.parse_args()``  (via utility.batch_test)
#   * ``data_generator`` construction  (via utility.batch_test)
#   * ``Trainer`` class definition
# but NOT training itself (that's guarded by ``if __name__ == "__main__"``).
import train
from utility.batch_test import data_generator, Ks as _BT_KS  # noqa: F401

args = train.args


# --- 2. Item -> tercile assignment (once, at import time) ------------------
_n_items = data_generator.n_items
_item_freq = np.zeros(_n_items, dtype=np.int64)
for _u, _items in data_generator.train_items.items():
    for _i in _items:
        _item_freq[_i] += 1
_order = np.argsort(_item_freq, kind="stable")   # ascending -> tail first
_t1 = _n_items // 3
_t2 = 2 * _n_items // 3
TAIL_IDS: set[int] = set(_order[:_t1].tolist())
MID_IDS:  set[int] = set(_order[_t1:_t2].tolist())
HEAD_IDS: set[int] = set(_order[_t2:].tolist())
print(
    f"[tercile] n_items={_n_items}  "
    f"|Tail|={len(TAIL_IDS)}  |Mid|={len(MID_IDS)}  |Head|={len(HEAD_IDS)}  "
    f"tail-freq<={int(_item_freq[_order[_t1 - 1]])}  "
    f"head-freq>={int(_item_freq[_order[_t2]])}",
    flush=True,
)


# --- 3. GPU-batched tercile recall evaluator (no multiprocessing) ---------
@torch.no_grad()
def compute_tercile_recall(
    ua: torch.Tensor,
    ia: torch.Tensor,
    users_to_test: list[int],
    ground_truth: dict[int, list[int]],
    K: int = 20,
) -> dict[str, float]:
    """Recall@K restricted to each popularity tercile.

    Per-user tercile recall = |hits in top-K that fall in tercile AND
    are ground-truth| / |ground-truth positives that fall in tercile|.
    Users with zero tercile-positives are skipped for that tercile
    (matches Milogradskii et al. 2024, Krichene & Rendle 2020).
    """
    head_scores: list[float] = []
    mid_scores:  list[float] = []
    tail_scores: list[float] = []
    ubs = 2048  # user-batch size for scoring
    for start in range(0, len(users_to_test), ubs):
        batch = users_to_test[start : start + ubs]
        ub = ua[batch]                                # (B, d)
        rate = (ub @ ia.T).cpu().numpy()              # (B, n_items)
        for row, u in enumerate(batch):
            scores = rate[row].copy()
            for ti in data_generator.train_items.get(u, []):
                scores[ti] = -1e9                     # exclude trained items
            ground = ground_truth.get(u, [])
            if not ground:
                continue
            top = np.argpartition(-scores, K)[:K]
            top = top[np.argsort(-scores[top])].tolist()
            gset = set(int(x) for x in ground)
            for tercile, out in (
                (HEAD_IDS, head_scores),
                (MID_IDS,  mid_scores),
                (TAIL_IDS, tail_scores),
            ):
                gt_in_tercile = gset & tercile
                if not gt_in_tercile:
                    continue
                hits = sum(
                    1 for it in top if int(it) in tercile and int(it) in gt_in_tercile
                )
                out.append(hits / len(gt_in_tercile))
    _m = lambda xs: float(np.mean(xs)) if xs else float("nan")
    return {"head": _m(head_scores), "mid": _m(mid_scores), "tail": _m(tail_scores)}


# --- 4. Shared state -------------------------------------------------------
# Most recent val / test tercile (updated every val or test eval).
_last_val_tercile:  dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
_last_test_tercile: dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
# Snapshots at the val_recall@20 peak (matches PACER's BEST_Test_Recall@20
# semantics — see train.py line 801: ``if val["recall"][1] > best_val_recall``).
_best_val_tercile:  dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
_best_test_tercile: dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
_best_val_recall = [0.0]
_val_recall_bumped_this_epoch = [False]  # consumed by the next is_val=False call in the same epoch
_have_val_tercile  = [False]
_have_test_tercile = [False]


# --- 5. Monkey-patch Trainer.test -----------------------------------------
_orig_test = train.Trainer.test


@torch.no_grad()
def _forward_embeddings(self):
    """One extra eval-mode forward pass to read u_ui_emb / i_ui_emb.

    PACER's model.forward returns a dict. ``update_momentum=False`` disables
    Slim Momentum EMA writes; ``epoch=0`` is inert because ``set_epoch`` is
    guarded by ``self.training`` inside model.py.
    """
    was_training = self.model.training
    self.model.eval()
    try:
        out = self.model(
            self.UI_mat,
            self.Item_mat,
            self.User_mat,
            item_indices=None,
            epoch=0,
            update_momentum=False,
        )
    finally:
        if was_training:
            self.model.train()
    return out["u_ui_emb"], out["i_ui_emb"]


def _test_with_terciles(self, users_to_test, is_val):
    result = _orig_test(self, users_to_test, is_val)

    if is_val:
        # ---- VAL branch: compute per-tercile Recall@20 on the val split.
        ua, ia = _forward_embeddings(self)
        ter = compute_tercile_recall(ua, ia, users_to_test, data_generator.val_set)
        _last_val_tercile.update(ter)
        _have_val_tercile[0] = True

        # PACER snapshots BEST_Test_Recall@20 at val_recall PEAK only
        # (train.py:801 `if val["recall"][1] > best_val_recall`). We mirror
        # that with a strict `>` comparison so the next is_val=False call
        # in the SAME epoch knows whether to update _best_test_tercile.
        rec = float(result["recall"][1])
        if rec > _best_val_recall[0]:
            _best_val_recall[0] = rec
            _best_val_tercile.update(ter)
            _val_recall_bumped_this_epoch[0] = True
            print(
                f"[tercile] val@recall-peak: "
                f"head={ter['head']:.6f} mid={ter['mid']:.6f} tail={ter['tail']:.6f} "
                f"(val_recall@20={rec:.6f})",
                flush=True,
            )
        else:
            _val_recall_bumped_this_epoch[0] = False
            print(
                f"[tercile] val: "
                f"head={ter['head']:.6f} mid={ter['mid']:.6f} tail={ter['tail']:.6f} "
                f"(val_recall@20={rec:.6f})",
                flush=True,
            )

        # Per-epoch WandB (adds Head/Mid/Tail alongside PACER's val/recall@20).
        if self.wandb is not None:
            try:
                self.wandb.log({
                    "val/recall@20_Head": ter["head"],
                    "val/recall@20_Mid":  ter["mid"],
                    "val/recall@20_Tail": ter["tail"],
                })
            except Exception as _e:
                print(f"[tercile] wandb.log(val) skipped: {_e}", flush=True)

    else:
        # ---- TEST branch: PACER calls Trainer.test(is_val=False) whenever
        # val_recall@20 OR val_ndcg@20 improved (train.py:789). To match
        # BEST_Test_Recall@20 semantics we snapshot _best_test_tercile ONLY
        # when val_recall bumped its peak in the SAME epoch (train.py:801).
        ua, ia = _forward_embeddings(self)
        ter_t = compute_tercile_recall(ua, ia, users_to_test, data_generator.test_set)
        _last_test_tercile.update(ter_t)
        _have_test_tercile[0] = True

        test_rec = float(result["recall"][1])
        if _val_recall_bumped_this_epoch[0]:
            _best_test_tercile.update(ter_t)
            _val_recall_bumped_this_epoch[0] = False   # consume the flag
            print(
                f"[tercile] test@recall-peak: "
                f"head={ter_t['head']:.6f} mid={ter_t['mid']:.6f} "
                f"tail={ter_t['tail']:.6f} (test_recall@20={test_rec:.6f})",
                flush=True,
            )
        else:
            # val_ndcg-only improvement — record test tercile but don't
            # overwrite the recall-peak snapshot.
            print(
                f"[tercile] test@ndcg-peak: "
                f"head={ter_t['head']:.6f} mid={ter_t['mid']:.6f} "
                f"tail={ter_t['tail']:.6f} (test_recall@20={test_rec:.6f})",
                flush=True,
            )

        # Per-epoch WandB (adds Head/Mid/Tail alongside PACER's test/recall@20).
        if self.wandb is not None:
            try:
                self.wandb.log({
                    "test/recall@20_Head": ter_t["head"],
                    "test/recall@20_Mid":  ter_t["mid"],
                    "test/recall@20_Tail": ter_t["tail"],
                })
            except Exception as _e:
                print(f"[tercile] wandb.log(test) skipped: {_e}", flush=True)

    return result


train.Trainer.test = _test_with_terciles


# --- 6. Monkey-patch Trainer.train to write wandb.summary at the end ------
_orig_train = train.Trainer.train


def _train_with_summary(self):
    try:
        return _orig_train(self)
    finally:
        # Write the best-tercile snapshots into wandb.summary so the run
        # Overview panel surfaces them (PACER writes best_val_* / best_test_*
        # at lines 899-930; we append the Head/Mid/Tail block).
        if self.wandb is not None:
            try:
                if _have_val_tercile[0]:
                    src_v = _best_val_tercile
                    if math.isnan(src_v["head"]):
                        src_v = _last_val_tercile
                    self.wandb.summary["best_recall@20_Head"] = src_v["head"]
                    self.wandb.summary["best_recall@20_Mid"]  = src_v["mid"]
                    self.wandb.summary["best_recall@20_Tail"] = src_v["tail"]
                if _have_test_tercile[0]:
                    src_t = _best_test_tercile
                    if math.isnan(src_t["head"]):
                        src_t = _last_test_tercile
                    self.wandb.summary["best_test_recall@20_Head"] = src_t["head"]
                    self.wandb.summary["best_test_recall@20_Mid"]  = src_t["mid"]
                    self.wandb.summary["best_test_recall@20_Tail"] = src_t["tail"]
            except Exception as _e:
                print(f"[tercile] wandb.summary write skipped: {_e}", flush=True)


train.Trainer.train = _train_with_summary


# --- 7. Formatting + __main__ ---------------------------------------------
def _fmt(x: float) -> str:
    """Format floats for the notebook regex (never emit bare 'nan')."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "nan"
    return f"{float(x):.8f}"


if __name__ == "__main__":
    # Delegate the actual training loop to PACER's train.main() so we stay
    # in lock-step with any future changes to __main__ (seed setup, data
    # config, Trainer instantiation). The Trainer methods we monkey-patched
    # above are already installed, so tercile logic fires automatically.
    train.main()

    # ---- Final snapshots (val_recall peak; matches BEST_Test_Recall@20) --
    final_val = dict(_best_val_tercile)
    if math.isnan(final_val["head"]) and _have_val_tercile[0]:
        final_val = dict(_last_val_tercile)
    final_test = dict(_best_test_tercile)
    if math.isnan(final_test["head"]) and _have_test_tercile[0]:
        final_test = dict(_last_test_tercile)

    # Notebook parser reads these two lines; keep the key format stable.
    print(
        "[tercile-final] "
        f"BEST_Recall@20_Head={_fmt(final_val['head'])} "
        f"BEST_Recall@20_Mid={_fmt(final_val['mid'])} "
        f"BEST_Recall@20_Tail={_fmt(final_val['tail'])}",
        flush=True,
    )
    print(
        "[tercile-test-final] "
        f"BEST_Test_Recall@20_Head={_fmt(final_test['head'])} "
        f"BEST_Test_Recall@20_Mid={_fmt(final_test['mid'])} "
        f"BEST_Test_Recall@20_Tail={_fmt(final_test['tail'])}",
        flush=True,
    )
'''
_MT_PATH = os.path.join(_DAMPS_DIR, "main_tercile.py")
with open(_MT_PATH, "w", encoding="utf-8") as _f:
    _f.write(_MAIN_TERCILE)
print(f"[smoke] wrote {_MT_PATH}  ({os.path.getsize(_MT_PATH)} bytes)", flush=True)

# ---- 2. Deterministic seed for reproducibility of the smoke check -------
SMOKE_SEED     = 1616406634    # arbitrary but fixed
SMOKE_EPOCH    = 10            # short enough to run in a few minutes
SMOKE_PATIENCE = 100           # never trigger early-stopping in the smoke

# ---- 3. PACER Clothing hyperparameters (paper Table 3, config t0030) ----
# Kept identical to §9.10 so the smoke exercises the exact code paths the
# 5-seed run will hit.
DATASET       = "Clothing"
BATCH_SIZE    = 1024
LR            = 2.50995e-4        # t0030 (user's Optuna optimum)
REGS          = 1.20e-4           # t0030 (user's Optuna optimum)
EMBED_SIZE    = 128
UI_LAYERS     = 3
TEMPERATURE   = 0.251
SIMGCL_EPS    = 0.329
LAMBDA_VIEW   = 0.1806            # t0030 (Optuna, 4 decimals)
LOGQ_SCALE    = 0.651
USE_REDUCE_LR = 1                 # t0030 reduce_lr=1

BASE_FLAGS: list[str] = [
    "--dataset",                     DATASET,
    "--gpu_id",                      "0",
    "--epoch",                       str(SMOKE_EPOCH),
    "--verbose",                     "5",
    "--batch_size",                  str(BATCH_SIZE),
    "--lr",                          str(LR),
    "--regs",                        str(REGS),
    "--embed_size",                  str(EMBED_SIZE),
    "--topk",                        "10",
    "--core",                        "5",
    "--UI_layers",                   str(UI_LAYERS),
    "--User_layers",                 "2",
    "--Item_layers",                 "2",
    "--temperature",                 str(TEMPERATURE),
    # ---- PACER: LogQ + SimGCL + APC-off (Round-2 verdict) ----
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_permutation_fft",       "0",
    "--damps_warmup_epochs",         "10",
    "--enable_logq",                 "1",
    "--logq_mode",                   "laplace",
    "--logq_beta",                   "1.0",
    "--logq_scale",                  str(LOGQ_SCALE),
    "--logq_clip",                   "5.0",
    "--enable_simgcl",               "1",
    "--simgcl_eps",                  str(SIMGCL_EPS),
    "--lambda_view",                 str(LAMBDA_VIEW),
    "--branchA_view_every_k",        "2",
    "--branchA_bcl_batchn",          "1",
    # ---- Early stopping (patience huge -> never trigger in smoke) ----
    "--early_stopping_patience",     str(SMOKE_PATIENCE),
    "--early_stopping_min_epochs",   "0",
    "--early_stopping_min_delta",    "1e-4",
    "--early_stopping_monitor",      "val_recall@20",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    "--adaptive_patience",           "0",
    "--use_reduce_lr",               str(USE_REDUCE_LR),
    "--use_amp",                     "1",
    # ---- WandB (opt-in; won't fail the smoke if wandb login absent) ----
    "--use_wandb",                   "0",
    "--seed",                        str(SMOKE_SEED),
    "--wandb_run_name",              f"smoke_pacer_tercile_{SMOKE_SEED}",
]

# ---- 4. Regexes for stdout parsing (same format as final cell) ----------
_TER_RX = re.compile(
    r"\[tercile-final\][^\n]*?"
    r"BEST_Recall@20_Head=(?P<h>nan|[0-9.eE+-]+)\s+"
    r"BEST_Recall@20_Mid=(?P<m>nan|[0-9.eE+-]+)\s+"
    r"BEST_Recall@20_Tail=(?P<t>nan|[0-9.eE+-]+)",
    re.IGNORECASE,
)
_TER_TEST_RX = re.compile(
    r"\[tercile-test-final\][^\n]*?"
    r"BEST_Test_Recall@20_Head=(?P<h>nan|[0-9.eE+-]+)\s+"
    r"BEST_Test_Recall@20_Mid=(?P<m>nan|[0-9.eE+-]+)\s+"
    r"BEST_Test_Recall@20_Tail=(?P<t>nan|[0-9.eE+-]+)",
    re.IGNORECASE,
)

# ---- 5. Kernel torch sanity ---------------------------------------------
_torch_ok = subprocess.run(
    [_PYTHON, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
)
if _torch_ok.returncode != 0:
    raise RuntimeError(
        f"[smoke] selected python cannot import torch: {_PYTHON}\n"
        f"stderr={_torch_ok.stderr}\n"
        "Select the rtx5090_dl Jupyter kernel (or set PYTHON_EXE)."
    )
print(f"[smoke] python = {_PYTHON}   torch = {_torch_ok.stdout.strip()}",
      flush=True)

# ---- 6. Run the smoke ---------------------------------------------------
print("\n" + "="*74)
print(f"SMOKE  seed={SMOKE_SEED}  epoch={SMOKE_EPOCH}")
print("="*74, flush=True)

cmd = [_PYTHON, "main_tercile.py", *BASE_FLAGS]
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["PYTHONUNBUFFERED"] = "1"
print("[smoke] " + " ".join(cmd), flush=True)

t0 = time.time()
proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", bufsize=1,
)
chunks: list[str] = []
assert proc.stdout is not None
for line in proc.stdout:
    chunks.append(line)
    print(line, end="", flush=True)
rc = proc.wait()
out = "".join(chunks)
elapsed = time.time() - t0
print(f"\n[smoke] elapsed = {elapsed:.1f}s   exit_code = {rc}", flush=True)

# ---- 7. Assertions ------------------------------------------------------
def _pf(x: str) -> float:
    return float("nan") if x.lower() == "nan" else float(x)


failures: list[str] = []
if rc != 0:
    failures.append(f"exit_code={rc} (expected 0)")

m  = _TER_RX.search(out)
mt = _TER_TEST_RX.search(out)

if m is None:
    failures.append("[tercile-final] line missing from stdout")
else:
    for k in ("h", "m", "t"):
        v = _pf(m.group(k))
        if not math.isfinite(v):
            failures.append(f"[tercile-final] {k}={m.group(k)!r} not finite")

if mt is None:
    failures.append("[tercile-test-final] line missing from stdout")
else:
    for k in ("h", "m", "t"):
        v = _pf(mt.group(k))
        if not math.isfinite(v):
            failures.append(f"[tercile-test-final] {k}={mt.group(k)!r} not finite")

if failures:
    print("\n" + "#"*74)
    print("[SMOKE-FAIL]  PACER tercile wrapper failed the smoke test:")
    for f in failures:
        print(f"   - {f}")
    print("#"*74, flush=True)
    raise AssertionError("Smoke test failed — do NOT proceed to §9.10.")

print("\n" + "="*74)
print("[SMOKE-PASS]  PACER tercile wrapper is healthy.")
print(f"  VAL   Head={_pf(m.group('h')):.5f}  "
      f"Mid={_pf(m.group('m')):.5f}  Tail={_pf(m.group('t')):.5f}")
print(f"  TEST  Head={_pf(mt.group('h')):.5f}  "
      f"Mid={_pf(mt.group('m')):.5f}  Tail={_pf(mt.group('t')):.5f}")
print("="*74, flush=True)


[smoke] wrote c:\Users\Anh Khoi\Desktop\MyCode\My research progress\DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing\MMHCL_DAMPS_Project\main_tercile.py  (14047 bytes)
[smoke] python = c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe   torch = 2.10.0+cu128

SMOKE  seed=1616406634  epoch=10
[smoke] c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe main_tercile.py --dataset Clothing --gpu_id 0 --epoch 10 --verbose 5 --batch_size 1024 --lr 0.000251 --regs 1e-3 --embed_size 128 --topk 10 --core 5 --UI_layers 3 --User_layers 2 --Item_layers 2 --temperature 0.251 --damps_apc 0 --damps_avrf 0 --damps_imcf 1 --damps_soft_routing 1 --damps_momentum 1 --damps_data_driven_prior 1 --damps_permutation_fft 0 --damps_warmup_epochs 10 --enable_logq 1 --logq_mode laplace --logq_beta 1.0 --logq_scale 0.651 --logq_clip 5.0 --enable_simgcl 1 --simgcl_eps 0.329 --lambda_view 0.181 --branchA_view_every_k 2 --branchA_bcl_batchn 1 --early_stopping_patience 100 --early_stopping_min_epochs 0 --early_stoppin

### 9.10 RQ4 (a) — PACER Popularity-Tercile Recall@20 (5 seeds × 250 epochs)

Fills Table IV of the KSE 2026 PACER paper (long-tail Recall@20 on
Amazon Clothing). Uses paper v4 config **t0030** (Sec.~V-A, lines 771-779):
$B{=}1024$, $\mathrm{lr}{=}2.51e{-}4$, $\varepsilon{=}0.329$, $\tau{=}0.251$,
$\lambda_{\text{view}}{=}0.181$, $s_{\text{logq}}{=}0.651$, embed $128$,
$L{=}3$, patience $20$, and PACER-final (`--damps_apc 0`).

Reports Head / Mid / Tail Recall@20 at the val-recall peak on BOTH val
and test splits; the test-side numbers match PACER's `BEST_Test_Recall@20`
printout and are what the paper's Table IV cites.

Output: `results/rq4_tercile_pacer_clothing_5seed.json`.


In [ ]:
# =====================================================================
# Section 9.10 — RQ4 (a) : PACER on Amazon Clothing with Head/Mid/Tail
#                          popularity terciles  (5 seeds × 250 epochs)
#
# Fills Table IV of the KSE 2026 PACER paper (long-tail Recall@20).
# Reports popularity-tercile Recall@20 evaluated on BOTH splits:
#   - VAL  side (per-epoch, snapshot at val_recall@20 peak)   -> RQ4 diag.
#   - TEST side (snapshot at the SAME val_recall@20 peak
#                epoch, i.e. what train.py prints as
#                `BEST_Test_Recall@20`)                       -> Table IV
#
# WandB / stdout metrics:
#   val/recall@20_Head|Mid|Tail                        -- per-epoch, VAL
#   test/recall@20_Head|Mid|Tail                       -- improved-val epochs
#   best_recall@20_Head|Mid|Tail       (run summary)   -- VAL @ recall peak
#   best_test_recall@20_Head|Mid|Tail  (run summary)   -- TEST @ same peak
#   [tercile-final]      BEST_Recall@20_Head|Mid|Tail=...     (VAL)
#   [tercile-test-final] BEST_Test_Recall@20_Head|Mid|Tail=...(TEST)
#
# Output:  results/rq4_tercile_pacer_clothing_5seed.json
#          (per-seed + aggregate for both val and test splits)
# =====================================================================
from __future__ import annotations

import json
import os
import re
import statistics
import subprocess
import sys
import time
from typing import Any

# ---- 0. Resolve dirs / globals (reuse from earlier cells if set) --------
try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PROJECT_ROOT = PROJECT_ROOT  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PROJECT_ROOT = os.path.dirname(_DAMPS_DIR)
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _PYTHON = sys.executable
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]  # noqa: F821
except NameError:
    _WB_ENTITY = "baitapck51cc-uet"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)
os.makedirs("results", exist_ok=True)

# ---- 0b. Purge stale hypergraph / adjacency caches ---------------------
#     PACER caches (`UI_mat_*.pth`, `User_mat_*.pth`,
#     `hypergraph_mat_mul_*_topk_*.pth`, `log_q_*_b*.pth`) are keyed by
#     dataset name -- NOT by the underlying JSON split. If we swap to a
#     different Clothing variant (e.g. 12k MMRec vs 39k HuggingFace/
#     LATTICE), the stale caches will silently drive training on the
#     wrong graph. Delete them here so PACER rebuilds from source.
import glob as _glob
_CACHE_PATTERNS = [
    "UI_mat_*.pth",
    "User_mat_*.pth",
    "Item_mat_*.pth",              # legacy MMHCL cache
    "hypergraph_mat_*.pth",        # PACER hypergraph cache (all K variants)
    "log_q_*_b*.pth",              # PACER LogQ prior cache
]
_DATASET_DIR = os.path.normpath(
    os.path.join(_PROJECT_ROOT, "data", "Clothing", "5-core")
)
if os.path.isdir(_DATASET_DIR):
    _purged = []
    for _pat in _CACHE_PATTERNS:
        for _p in _glob.glob(os.path.join(_DATASET_DIR, _pat)):
            try:
                os.remove(_p)
                _purged.append(os.path.basename(_p))
            except OSError as _e:
                print(f"[cache-purge] skip {_p}: {_e}", flush=True)
    print(
        f"[cache-purge] {_DATASET_DIR}: removed {len(_purged)} file(s)"
        + (f" -> {_purged}" if _purged else ""),
        flush=True,
    )
else:
    print(f"[cache-purge] WARNING dataset dir not found: {_DATASET_DIR}",
          flush=True)

# ---- 0c. Sanity-check raw JSON splits (HF LATTICE 39k) -----------------
#     PACER's Clothing baseline (paper Table 1) assumes the HuggingFace
#     LATTICE variant of Amazon-Clothing (~39,387 train users). The MMRec
#     12k variant is a different subgraph and reproduces R@20 ~0.11
#     instead of ~0.088. Fail fast if the wrong variant is on disk.
import json as _json
_split_users: dict[str, int] = {}
_split_items: set[int] = set()
for _sp in ("train", "val", "test"):
    _fp = os.path.join(_DATASET_DIR, f"{_sp}.json")
    if not os.path.isfile(_fp):
        _split_users[_sp] = -1
        continue
    try:
        with open(_fp, "r", encoding="utf-8") as _f:
            _obj = _json.load(_f)
        _split_users[_sp] = len(_obj)
        for _v in _obj.values():
            if isinstance(_v, list):
                for _it in _v:
                    _split_items.add(int(_it))
    except Exception as _e:
        print(f"[sanity] {_fp}: {_e}", flush=True)
        _split_users[_sp] = -1
_train_u = _split_users.get("train", -1)
print(
    f"[sanity] users -> train={_split_users.get('train','?')}  "
    f"val={_split_users.get('val','?')}  test={_split_users.get('test','?')}  "
    f"unique_items(union)={len(_split_items)}",
    flush=True,
)
if _train_u >= 35000:
    print("[sanity] OK: HuggingFace/LATTICE 39k-user variant detected.",
          flush=True)
elif 0 < _train_u <= 15000:
    print(
        "[sanity] WARNING: only ~12k train users -- looks like the MMRec\n"
        "        Clothing variant, NOT the paper's LATTICE/HuggingFace\n"
        "        release. Baseline Recall@20 will diverge (~0.11 vs 0.088).",
        flush=True,
    )
elif _train_u == -1:
    print("[sanity] WARNING: could not read train.json -- proceeding anyway.",
          flush=True)
else:
    print(f"[sanity] WARNING: unusual train-user count = {_train_u}.", flush=True)

# ---- 1. Emit main_tercile.py -- PACER-aware tercile wrapper of train.py -
#     Monkey-patches train.Trainer.test + train.Trainer.train so PACER's
#     stock loop emits per-tercile Recall@20 without editing train.py.
_MAIN_TERCILE = r'''"""main_tercile.py  ──  PACER (DAMPS-MMHCL) tercile-recall wrapper.
================================================================

Drop-in replacement for ``MMHCL_DAMPS_Project/train.py`` that adds four
things on top of the stock PACER training loop:

  1. On every VAL evaluation, computes Recall@20 restricted to items in
     the Head / Mid / Tail popularity tercile (by training frequency).
  2. Whenever val_recall@20 hits a new peak (matching PACER's
     ``BEST_Test_Recall@20`` semantics — see ``train.py`` line 801),
     snapshots both:
         - VAL Head/Mid/Tail at that epoch      (``_best_val_tercile``)
         - TEST Head/Mid/Tail at that epoch     (``_best_test_tercile``)
     independently of WandB.
  3. On every epoch where PACER logs ``val/recall@20`` (line 762) or
     ``test/recall@20`` (line 833) to WandB, we also log:
         val/recall@20_Head|Mid|Tail
         test/recall@20_Head|Mid|Tail
     and at the end of training we write the "best" snapshots into
     ``wandb.summary`` so the run's Overview panel shows them.
  4. Two parser-friendly summary lines are printed at end of run:
         [tercile-final]     BEST_Recall@20_Head=..  Mid=..  Tail=..
         [tercile-test-final] BEST_Test_Recall@20_Head=..  Mid=..  Tail=..

The wrapper does NOT touch the training math -- it only reads the final
user/item embeddings (via a single extra forward pass under ``model.eval``
with ``update_momentum=False``) and computes tercile recall in Python.

Usage (drop-in for train.py; expects the same CLI):
    python main_tercile.py --dataset Clothing --seed <s> \\
        --enable_logq 1 --enable_simgcl 1 [ ... ]
"""
from __future__ import annotations

import math
import os
from typing import Any as _Any

import numpy as np
import torch

# --- 1. Import PACER's train module ---------------------------------------
# Importing ``train`` triggers:
#   * ``utility.parser.parse_args()``  (via utility.batch_test)
#   * ``data_generator`` construction  (via utility.batch_test)
#   * ``Trainer`` class definition
# but NOT training itself (that's guarded by ``if __name__ == "__main__"``).
import train
from utility.batch_test import data_generator, Ks as _BT_KS  # noqa: F401

args = train.args


# --- 2. Item -> tercile assignment (once, at import time) ------------------
_n_items = data_generator.n_items
_item_freq = np.zeros(_n_items, dtype=np.int64)
for _u, _items in data_generator.train_items.items():
    for _i in _items:
        _item_freq[_i] += 1
_order = np.argsort(_item_freq, kind="stable")   # ascending -> tail first
_t1 = _n_items // 3
_t2 = 2 * _n_items // 3
TAIL_IDS: set[int] = set(_order[:_t1].tolist())
MID_IDS:  set[int] = set(_order[_t1:_t2].tolist())
HEAD_IDS: set[int] = set(_order[_t2:].tolist())
print(
    f"[tercile] n_items={_n_items}  "
    f"|Tail|={len(TAIL_IDS)}  |Mid|={len(MID_IDS)}  |Head|={len(HEAD_IDS)}  "
    f"tail-freq<={int(_item_freq[_order[_t1 - 1]])}  "
    f"head-freq>={int(_item_freq[_order[_t2]])}",
    flush=True,
)


# --- 3. GPU-batched tercile recall evaluator (no multiprocessing) ---------
@torch.no_grad()
def compute_tercile_recall(
    ua: torch.Tensor,
    ia: torch.Tensor,
    users_to_test: list[int],
    ground_truth: dict[int, list[int]],
    K: int = 20,
) -> dict[str, float]:
    """Recall@K restricted to each popularity tercile.

    Per-user tercile recall = |hits in top-K that fall in tercile AND
    are ground-truth| / |ground-truth positives that fall in tercile|.
    Users with zero tercile-positives are skipped for that tercile
    (matches Milogradskii et al. 2024, Krichene & Rendle 2020).
    """
    head_scores: list[float] = []
    mid_scores:  list[float] = []
    tail_scores: list[float] = []
    ubs = 2048  # user-batch size for scoring
    for start in range(0, len(users_to_test), ubs):
        batch = users_to_test[start : start + ubs]
        ub = ua[batch]                                # (B, d)
        rate = (ub @ ia.T).cpu().numpy()              # (B, n_items)
        for row, u in enumerate(batch):
            scores = rate[row].copy()
            for ti in data_generator.train_items.get(u, []):
                scores[ti] = -1e9                     # exclude trained items
            ground = ground_truth.get(u, [])
            if not ground:
                continue
            top = np.argpartition(-scores, K)[:K]
            top = top[np.argsort(-scores[top])].tolist()
            gset = set(int(x) for x in ground)
            for tercile, out in (
                (HEAD_IDS, head_scores),
                (MID_IDS,  mid_scores),
                (TAIL_IDS, tail_scores),
            ):
                gt_in_tercile = gset & tercile
                if not gt_in_tercile:
                    continue
                hits = sum(
                    1 for it in top if int(it) in tercile and int(it) in gt_in_tercile
                )
                out.append(hits / len(gt_in_tercile))
    _m = lambda xs: float(np.mean(xs)) if xs else float("nan")
    return {"head": _m(head_scores), "mid": _m(mid_scores), "tail": _m(tail_scores)}


# --- 4. Shared state -------------------------------------------------------
# Most recent val / test tercile (updated every val or test eval).
_last_val_tercile:  dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
_last_test_tercile: dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
# Snapshots at the val_recall@20 peak (matches PACER's BEST_Test_Recall@20
# semantics — see train.py line 801: ``if val["recall"][1] > best_val_recall``).
_best_val_tercile:  dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
_best_test_tercile: dict[str, float] = {"head": float("nan"), "mid": float("nan"), "tail": float("nan")}
_best_val_recall = [0.0]
_val_recall_bumped_this_epoch = [False]  # consumed by the next is_val=False call in the same epoch
_have_val_tercile  = [False]
_have_test_tercile = [False]


# --- 5. Monkey-patch Trainer.test -----------------------------------------
_orig_test = train.Trainer.test


@torch.no_grad()
def _forward_embeddings(self):
    """One extra eval-mode forward pass to read u_ui_emb / i_ui_emb.

    PACER's model.forward returns a dict. ``update_momentum=False`` disables
    Slim Momentum EMA writes; ``epoch=0`` is inert because ``set_epoch`` is
    guarded by ``self.training`` inside model.py.
    """
    was_training = self.model.training
    self.model.eval()
    try:
        out = self.model(
            self.UI_mat,
            self.Item_mat,
            self.User_mat,
            item_indices=None,
            epoch=0,
            update_momentum=False,
        )
    finally:
        if was_training:
            self.model.train()
    return out["u_ui_emb"], out["i_ui_emb"]


def _test_with_terciles(self, users_to_test, is_val):
    result = _orig_test(self, users_to_test, is_val)

    if is_val:
        # ---- VAL branch: compute per-tercile Recall@20 on the val split.
        ua, ia = _forward_embeddings(self)
        ter = compute_tercile_recall(ua, ia, users_to_test, data_generator.val_set)
        _last_val_tercile.update(ter)
        _have_val_tercile[0] = True

        # PACER snapshots BEST_Test_Recall@20 at val_recall PEAK only
        # (train.py:801 `if val["recall"][1] > best_val_recall`). We mirror
        # that with a strict `>` comparison so the next is_val=False call
        # in the SAME epoch knows whether to update _best_test_tercile.
        rec = float(result["recall"][1])
        if rec > _best_val_recall[0]:
            _best_val_recall[0] = rec
            _best_val_tercile.update(ter)
            _val_recall_bumped_this_epoch[0] = True
            print(
                f"[tercile] val@recall-peak: "
                f"head={ter['head']:.6f} mid={ter['mid']:.6f} tail={ter['tail']:.6f} "
                f"(val_recall@20={rec:.6f})",
                flush=True,
            )
        else:
            _val_recall_bumped_this_epoch[0] = False
            print(
                f"[tercile] val: "
                f"head={ter['head']:.6f} mid={ter['mid']:.6f} tail={ter['tail']:.6f} "
                f"(val_recall@20={rec:.6f})",
                flush=True,
            )

        # Per-epoch WandB (adds Head/Mid/Tail alongside PACER's val/recall@20).
        if self.wandb is not None:
            try:
                self.wandb.log({
                    "val/recall@20_Head": ter["head"],
                    "val/recall@20_Mid":  ter["mid"],
                    "val/recall@20_Tail": ter["tail"],
                })
            except Exception as _e:
                print(f"[tercile] wandb.log(val) skipped: {_e}", flush=True)

    else:
        # ---- TEST branch: PACER calls Trainer.test(is_val=False) whenever
        # val_recall@20 OR val_ndcg@20 improved (train.py:789). To match
        # BEST_Test_Recall@20 semantics we snapshot _best_test_tercile ONLY
        # when val_recall bumped its peak in the SAME epoch (train.py:801).
        ua, ia = _forward_embeddings(self)
        ter_t = compute_tercile_recall(ua, ia, users_to_test, data_generator.test_set)
        _last_test_tercile.update(ter_t)
        _have_test_tercile[0] = True

        test_rec = float(result["recall"][1])
        if _val_recall_bumped_this_epoch[0]:
            _best_test_tercile.update(ter_t)
            _val_recall_bumped_this_epoch[0] = False   # consume the flag
            print(
                f"[tercile] test@recall-peak: "
                f"head={ter_t['head']:.6f} mid={ter_t['mid']:.6f} "
                f"tail={ter_t['tail']:.6f} (test_recall@20={test_rec:.6f})",
                flush=True,
            )
        else:
            # val_ndcg-only improvement — record test tercile but don't
            # overwrite the recall-peak snapshot.
            print(
                f"[tercile] test@ndcg-peak: "
                f"head={ter_t['head']:.6f} mid={ter_t['mid']:.6f} "
                f"tail={ter_t['tail']:.6f} (test_recall@20={test_rec:.6f})",
                flush=True,
            )

        # Per-epoch WandB (adds Head/Mid/Tail alongside PACER's test/recall@20).
        if self.wandb is not None:
            try:
                self.wandb.log({
                    "test/recall@20_Head": ter_t["head"],
                    "test/recall@20_Mid":  ter_t["mid"],
                    "test/recall@20_Tail": ter_t["tail"],
                })
            except Exception as _e:
                print(f"[tercile] wandb.log(test) skipped: {_e}", flush=True)

    return result


train.Trainer.test = _test_with_terciles


# --- 6. Monkey-patch Trainer.train to write wandb.summary at the end ------
_orig_train = train.Trainer.train


def _train_with_summary(self):
    try:
        return _orig_train(self)
    finally:
        # Write the best-tercile snapshots into wandb.summary so the run
        # Overview panel surfaces them (PACER writes best_val_* / best_test_*
        # at lines 899-930; we append the Head/Mid/Tail block).
        if self.wandb is not None:
            try:
                if _have_val_tercile[0]:
                    src_v = _best_val_tercile
                    if math.isnan(src_v["head"]):
                        src_v = _last_val_tercile
                    self.wandb.summary["best_recall@20_Head"] = src_v["head"]
                    self.wandb.summary["best_recall@20_Mid"]  = src_v["mid"]
                    self.wandb.summary["best_recall@20_Tail"] = src_v["tail"]
                if _have_test_tercile[0]:
                    src_t = _best_test_tercile
                    if math.isnan(src_t["head"]):
                        src_t = _last_test_tercile
                    self.wandb.summary["best_test_recall@20_Head"] = src_t["head"]
                    self.wandb.summary["best_test_recall@20_Mid"]  = src_t["mid"]
                    self.wandb.summary["best_test_recall@20_Tail"] = src_t["tail"]
            except Exception as _e:
                print(f"[tercile] wandb.summary write skipped: {_e}", flush=True)


train.Trainer.train = _train_with_summary


# --- 7. Formatting + __main__ ---------------------------------------------
def _fmt(x: float) -> str:
    """Format floats for the notebook regex (never emit bare 'nan')."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "nan"
    return f"{float(x):.8f}"


if __name__ == "__main__":
    # Delegate the actual training loop to PACER's train.main() so we stay
    # in lock-step with any future changes to __main__ (seed setup, data
    # config, Trainer instantiation). The Trainer methods we monkey-patched
    # above are already installed, so tercile logic fires automatically.
    train.main()

    # ---- Final snapshots (val_recall peak; matches BEST_Test_Recall@20) --
    final_val = dict(_best_val_tercile)
    if math.isnan(final_val["head"]) and _have_val_tercile[0]:
        final_val = dict(_last_val_tercile)
    final_test = dict(_best_test_tercile)
    if math.isnan(final_test["head"]) and _have_test_tercile[0]:
        final_test = dict(_last_test_tercile)

    # Notebook parser reads these two lines; keep the key format stable.
    print(
        "[tercile-final] "
        f"BEST_Recall@20_Head={_fmt(final_val['head'])} "
        f"BEST_Recall@20_Mid={_fmt(final_val['mid'])} "
        f"BEST_Recall@20_Tail={_fmt(final_val['tail'])}",
        flush=True,
    )
    print(
        "[tercile-test-final] "
        f"BEST_Test_Recall@20_Head={_fmt(final_test['head'])} "
        f"BEST_Test_Recall@20_Mid={_fmt(final_test['mid'])} "
        f"BEST_Test_Recall@20_Tail={_fmt(final_test['tail'])}",
        flush=True,
    )
'''
_MT_PATH = os.path.join(_DAMPS_DIR, "main_tercile.py")
with open(_MT_PATH, "w", encoding="utf-8") as _f:
    _f.write(_MAIN_TERCILE)
print(f"[setup] wrote {_MT_PATH}  ({os.path.getsize(_MT_PATH)} bytes)",
      flush=True)

# ---- 2. Five seeds (paired with the MMHCL 5-seed baseline) -------------
#     Match the seeds already used for the MMHCL Clothing 5-seed rerun so
#     Table IV / RQ1 stay paired seed-for-seed (paired-seed t-test in
#     §V.B is only coherent when both methods share the same seeds).
#     Override by setting `seeds = [ ... ]` in an earlier cell.
N_SEEDS = 5
_MMHCL_PAIRED_SEEDS = [23946202, 1557638902, 969448664, 913925833, 85755121]
try:
    SEEDS: list[int] = [int(s) for s in list(seeds)[:N_SEEDS]]  # type: ignore[name-defined]  # noqa: F821
    if len(SEEDS) < N_SEEDS:
        raise NameError
    print(f"[rq4] using seeds from earlier cell: {SEEDS}", flush=True)
except NameError:
    SEEDS = list(_MMHCL_PAIRED_SEEDS)
    print(f"[rq4] using MMHCL-paired seeds (Clothing 5-seed baseline): {SEEDS}", flush=True)

# ---- 3. PACER hyperparameters — Clothing Optuna optimum, config t0030
#     (trial=30, sweep NDCG@20 = 0.053048)
#     Verbatim from user's sweep report:
#       lr=0.000250995  bs=1024  regs=1.20e-04  eps=0.329  lv=0.1806
#       T=0.251  embed=128  UI_layers=3  logq=0.651  reduce_lr=1
#     Section V-A of KSE2026_PACER_v4.tex "Fairness statement":
#       250-epoch cap, patience=20, identical Adam optimiser,
#       val-Recall@20 as sole model-selection.
#     Section IV.D "Round-2 verdict": PACER-final uses --damps_apc=0.
DATASET       = "Clothing"
EPOCH         = 250
PATIENCE      = 20                # paper "fairness statement"
BATCH_SIZE    = 1024              # t0030
LR            = 2.50995e-4        # t0030 (user's Optuna optimum)
REGS          = 1.20e-4           # t0030 (user's Optuna optimum)
EMBED_SIZE    = 128               # t0030 (differs from MMHCL's 64)
KNN_TOPK      = 10
UI_LAYERS     = 3                 # t0030 (paper "L=3" LightGCN depth)
U_LAYERS      = 2                 # MMHCL default (unchanged in PACER)
I_LAYERS      = 2                 # MMHCL default (unchanged in PACER)
TEMPERATURE   = 0.251             # t0030 τ
SIMGCL_EPS    = 0.329             # t0030 ε
LAMBDA_VIEW   = 0.1806            # t0030 λ_view (Optuna, 4 decimals)
LOGQ_SCALE    = 0.651             # t0030 s_logq
LOGQ_BETA     = 1.0               # paper §III-C (theory-motivated)
LOGQ_CLIP     = 5.0               # PACER default (Sec.~III-C)
USE_REDUCE_LR = 1                 # t0030 reduce_lr=1 (Optuna optimum)

WB_GROUP = "rq4_tercile_pacer_clothing_5seed"
WB_TAGS  = "rq4,tercile,pacer,t0030_optuna,clothing,5seed"

BASE_FLAGS: list[str] = [
    "--dataset",                     DATASET,
    "--gpu_id",                      "0",
    "--epoch",                       str(EPOCH),
    "--verbose",                     "5",
    "--batch_size",                  str(BATCH_SIZE),
    "--lr",                          str(LR),
    "--regs",                        str(REGS),
    "--embed_size",                  str(EMBED_SIZE),
    "--topk",                        str(KNN_TOPK),
    "--core",                        "5",
    "--UI_layers",                   str(UI_LAYERS),
    "--User_layers",                 str(U_LAYERS),
    "--Item_layers",                 str(I_LAYERS),
    "--temperature",                 str(TEMPERATURE),
    # ---- PACER: APC off + IMCF + soft routing + momentum + DDP -------
    "--damps_apc",                   "0",   # Round-2 verdict (Sec.~IV.D)
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_permutation_fft",       "0",
    "--damps_warmup_epochs",         "10",
    # ---- PACER: LogQ correction (Sec.~III-C, Laplace prior) ----------
    "--enable_logq",                 "1",
    "--logq_mode",                   "laplace",
    "--logq_beta",                   str(LOGQ_BETA),
    "--logq_scale",                  str(LOGQ_SCALE),
    "--logq_clip",                   str(LOGQ_CLIP),
    # ---- PACER: SimGCL view cache (Sec.~III-D, Branch A batch-N) -----
    "--enable_simgcl",               "1",
    "--simgcl_eps",                  str(SIMGCL_EPS),
    "--lambda_view",                 str(LAMBDA_VIEW),
    "--branchA_view_every_k",        "2",   # two-epoch view cache
    "--branchA_bcl_batchn",          "1",   # batch-N InfoNCE
    # ---- Early stopping (paper: patience=20, restore best) -----------
    "--early_stopping_patience",     str(PATIENCE),
    "--early_stopping_min_epochs",   "0",
    "--early_stopping_min_delta",    "1e-4",
    "--early_stopping_monitor",      "val_recall@20",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    "--adaptive_patience",           "0",
    # ---- ReduceLROnPlateau (ON per t0030 sweep result) ---------------
    "--use_reduce_lr",               str(USE_REDUCE_LR),
    # ---- AMP bf16 (paper Fig.~1 total-loss box) ----------------------
    "--use_amp",                     "1",
    # ---- WandB ------------------------------------------------------
    "--use_wandb",                   "1",
    "--wandb_project",               _WB_PROJECT,
    "--wandb_entity",                _WB_ENTITY,
]

# ---- 4. Regex for stdout fallback if WandB API is unreachable ----------
_TER_RX = re.compile(
    r"\[tercile-final\][^\n]*?"
    r"BEST_Recall@20_Head=(?P<h>nan|[0-9.eE+-]+)\s+"
    r"BEST_Recall@20_Mid=(?P<m>nan|[0-9.eE+-]+)\s+"
    r"BEST_Recall@20_Tail=(?P<t>nan|[0-9.eE+-]+)",
    re.IGNORECASE,
)
_TER_TEST_RX = re.compile(
    r"\[tercile-test-final\][^\n]*?"
    r"BEST_Test_Recall@20_Head=(?P<h>nan|[0-9.eE+-]+)\s+"
    r"BEST_Test_Recall@20_Mid=(?P<m>nan|[0-9.eE+-]+)\s+"
    r"BEST_Test_Recall@20_Tail=(?P<t>nan|[0-9.eE+-]+)",
    re.IGNORECASE,
)
# Per-epoch fallbacks (used only if the final line is missing/NaN):
_VALBEST_RX = re.compile(
    r"\[tercile\] val@recall-peak:\s+"
    r"head=(?P<h>[0-9.eE+-]+)\s+"
    r"mid=(?P<m>[0-9.eE+-]+)\s+"
    r"tail=(?P<t>[0-9.eE+-]+)"
)
_TESTIMP_RX = re.compile(
    r"\[tercile\] test@recall-peak:\s+"
    r"head=(?P<h>[0-9.eE+-]+)\s+"
    r"mid=(?P<m>[0-9.eE+-]+)\s+"
    r"tail=(?P<t>[0-9.eE+-]+)"
)
_BEST_RX = re.compile(r"BEST_Test_Recall@20:\s+([\d.eE+-]+)")

# Prefer the Jupyter/rtx5090_dl kernel; never fall back to a torch-less python.
print(f"[rq4] python = {_PYTHON}", flush=True)
_torch_ok = subprocess.run(
    [_PYTHON, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
)
if _torch_ok.returncode != 0:
    raise RuntimeError(
        f"Selected python cannot import torch: {_PYTHON}\n"
        f"stderr={_torch_ok.stderr}\n"
        "Select the rtx5090_dl Jupyter kernel (or set PYTHON_EXE) and re-run."
    )
print(f"[rq4] torch = {_torch_ok.stdout.strip()}", flush=True)

# ---- 5. Per-seed subprocess loop ---------------------------------------
results: list[dict[str, Any]] = []
for run_idx, seed in enumerate(SEEDS, 1):
    print(f"\n{'='*74}\nRUN {run_idx}/{len(SEEDS)}  seed={seed}\n{'='*74}",
          flush=True)
    run_name = f"pacer_tercile_seed_{seed}"
    cmd = [_PYTHON, "main_tercile.py", *BASE_FLAGS,
           "--seed", str(seed),
           "--wandb_run_name", run_name]
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    print("[cmd] " + " ".join(cmd), flush=True)

    # Stream stdout/stderr live (and keep a full copy for parsing).
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    chunks: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        chunks.append(line)
        print(line, end="", flush=True)
    rc = proc.wait()
    out = "".join(chunks)

    if rc != 0:
        print(f"[WARN] seed={seed} exited {rc}", flush=True)
        # still try to parse any metrics that were emitted

    m  = _TER_RX.search(out)          # VAL  side  [tercile-final]
    mt = _TER_TEST_RX.search(out)     # TEST side  [tercile-test-final]
    b  = _BEST_RX.search(out)         # overall test Recall@20 (from train.py)

    # Fallbacks for VAL: last val@recall-peak line (if final line NaN)
    if (m is None) or any(m.group(k).lower() == "nan" for k in ("h", "m", "t")):
        vb_all = list(_VALBEST_RX.finditer(out))
        if vb_all:
            m = vb_all[-1]
    # Fallback for TEST: last test@recall-peak line
    if (mt is None) or any(mt.group(k).lower() == "nan" for k in ("h", "m", "t")):
        tb_all = list(_TESTIMP_RX.finditer(out))
        if tb_all:
            mt = tb_all[-1]

    if not m and not mt:
        print(f"[WARN] seed={seed}: no tercile metrics found in output",
              flush=True)
        continue

    def _f(x: str) -> float:
        return float("nan") if x.lower() == "nan" else float(x)

    entry: dict[str, Any] = {
        "seed":               seed,
        "wandb_run_name":     run_name,
        # VAL-side terciles (at val_recall peak)
        "val_recall@20_head":  _f(m.group("h"))  if m  else float("nan"),
        "val_recall@20_mid":   _f(m.group("m"))  if m  else float("nan"),
        "val_recall@20_tail":  _f(m.group("t"))  if m  else float("nan"),
        # TEST-side terciles (at same val_recall peak) -- for paper Table IV
        "test_recall@20_head": _f(mt.group("h")) if mt else float("nan"),
        "test_recall@20_mid":  _f(mt.group("m")) if mt else float("nan"),
        "test_recall@20_tail": _f(mt.group("t")) if mt else float("nan"),
        # Overall best-test Recall@20 (train.py's BEST_Test_Recall@20 line)
        "recall@20":           float(b.group(1)) if b  else float("nan"),
        "exit_code":           rc,
    }
    # Back-compat aliases (legacy code reads `recall@20_head/mid/tail`).
    entry["recall@20_head"] = entry["val_recall@20_head"]
    entry["recall@20_mid"]  = entry["val_recall@20_mid"]
    entry["recall@20_tail"] = entry["val_recall@20_tail"]

    val_all_nan  = all(v != v for v in (entry["val_recall@20_head"],
                                        entry["val_recall@20_mid"],
                                        entry["val_recall@20_tail"]))
    test_all_nan = all(v != v for v in (entry["test_recall@20_head"],
                                        entry["test_recall@20_mid"],
                                        entry["test_recall@20_tail"]))
    if val_all_nan and test_all_nan:
        print(f"[WARN] seed={seed}: val + test terciles both NaN -- skip",
              flush=True)
        continue

    results.append(entry)
    print(
        f"[ok] seed={seed}  R@20={entry['recall@20']:.5f}\n"
        f"      VAL   Head={entry['val_recall@20_head']:.5f}  "
        f"Mid={entry['val_recall@20_mid']:.5f}  "
        f"Tail={entry['val_recall@20_tail']:.5f}\n"
        f"      TEST  Head={entry['test_recall@20_head']:.5f}  "
        f"Mid={entry['test_recall@20_mid']:.5f}  "
        f"Tail={entry['test_recall@20_tail']:.5f}",
        flush=True,
    )

# ---- 6. Aggregate + persist --------------------------------------------
def _mean_std(xs: list[float]) -> tuple[float, float]:
    xs = [x for x in xs if x == x]  # drop NaNs
    if not xs:
        return float("nan"), float("nan")
    return (
        float(statistics.mean(xs)),
        float(statistics.pstdev(xs) if len(xs) < 2 else statistics.stdev(xs)),
    )


agg = {
    "recall@20":            _mean_std([r["recall@20"]            for r in results]),
    # VAL-side
    "val_recall@20_head":   _mean_std([r["val_recall@20_head"]   for r in results]),
    "val_recall@20_mid":    _mean_std([r["val_recall@20_mid"]    for r in results]),
    "val_recall@20_tail":   _mean_std([r["val_recall@20_tail"]   for r in results]),
    # TEST-side (paper Table IV / V)
    "test_recall@20_head":  _mean_std([r["test_recall@20_head"]  for r in results]),
    "test_recall@20_mid":   _mean_std([r["test_recall@20_mid"]   for r in results]),
    "test_recall@20_tail":  _mean_std([r["test_recall@20_tail"]  for r in results]),
    # Back-compat aliases (== VAL-side)
    "recall@20_head":       _mean_std([r["val_recall@20_head"]   for r in results]),
    "recall@20_mid":        _mean_std([r["val_recall@20_mid"]    for r in results]),
    "recall@20_tail":       _mean_std([r["val_recall@20_tail"]   for r in results]),
}

OUT_JSON = os.path.join("results", "rq4_tercile_pacer_clothing_5seed.json")
with open(OUT_JSON, "w", encoding="utf-8") as _f:
    json.dump({
        "config":  {
            "method":       "PACER (t0030, Optuna optimum)",
            "dataset":      DATASET,
            "epoch":        EPOCH,
            "patience":     PATIENCE,
            "batch_size":   BATCH_SIZE,
            "lr":           LR,
            "regs":         REGS,
            "embed_size":   EMBED_SIZE,
            "knn_topk":     KNN_TOPK,
            "UI_layers":    UI_LAYERS,
            "U_layers":     U_LAYERS,
            "I_layers":     I_LAYERS,
            "temperature":  TEMPERATURE,
            "simgcl_eps":   SIMGCL_EPS,
            "lambda_view":  LAMBDA_VIEW,
            "logq_scale":   LOGQ_SCALE,
            "logq_beta":    LOGQ_BETA,
            "logq_clip":    LOGQ_CLIP,
            "damps_apc":    0,
            "branchA_view_every_k": 2,
            "branchA_bcl_batchn":   1,
            "use_reduce_lr":        USE_REDUCE_LR,
        },
        "seeds":   SEEDS,
        "results": results,
        "agg":     agg,
    }, _f, indent=2)

print("\n" + "="*74)
print("RQ4 (a)  --  PACER (t0030), Amazon Clothing, 5 seeds")
print("="*74)
print(f"  Recall@20 (overall test) : "
      f"{agg['recall@20'][0]:.5f} +/- {agg['recall@20'][1]:.5f}")
print("  ---- VAL-side terciles (val_recall peak) --------------------------")
print(f"  val  Recall@20  Head     : "
      f"{agg['val_recall@20_head'][0]:.5f} +/- {agg['val_recall@20_head'][1]:.5f}")
print(f"  val  Recall@20  Mid      : "
      f"{agg['val_recall@20_mid'][0]:.5f} +/- {agg['val_recall@20_mid'][1]:.5f}")
print(f"  val  Recall@20  Tail     : "
      f"{agg['val_recall@20_tail'][0]:.5f} +/- {agg['val_recall@20_tail'][1]:.5f}")
print("  ---- TEST-side terciles (same peak) -- for paper Table IV -------")
print(f"  test Recall@20  Head     : "
      f"{agg['test_recall@20_head'][0]:.5f} +/- {agg['test_recall@20_head'][1]:.5f}")
print(f"  test Recall@20  Mid      : "
      f"{agg['test_recall@20_mid'][0]:.5f} +/- {agg['test_recall@20_mid'][1]:.5f}")
print(f"  test Recall@20  Tail     : "
      f"{agg['test_recall@20_tail'][0]:.5f} +/- {agg['test_recall@20_tail'][1]:.5f}")
print(f"  saved -> {OUT_JSON}")


### 9.10.5 Branch A' checkout — activate NRDMC-lite (rev55 §8.2)

Switches the local PACER checkout to branch `wave2/branchA-prime-nrdmc-lite`,
which adds:

  * `damps/nrdmc_lite.py` — learnable **SAV** (Eq. 14) + **IAV** (Eq. 16) view
    generators + adaptive fusion (Eq. 17-19) + view-graph LightGCN (§8.2).
    PTV is dropped per §8.2.
  * `model.py`: `--enable_nrdmc_lite 1` short-circuits `simgcl_view_forward()`
    into the NRDMC-lite path. The SimGCL branch is bit-for-bit unchanged when
    the flag is 0.
  * `parser.py`: two new args — `--enable_nrdmc_lite {0,1}` and
    `--nrdmc_lite_layers N` (default 2).
  * `train.py`: plumbs the flags + triggers the view-loss branch when either
    `--enable_simgcl` OR `--enable_nrdmc_lite` is on.

**Expected wall-clock overhead on Amazon Clothing (39k users, 197k edges):**
one extra sparse-matmul per batch → < 3 % vs Branch A SimGCL.

**Expected performance (design-doc §8.2 target):** Recall@20 ∈ [0.0935, 0.0961].

Run this cell **once**; it is a no-op on subsequent runs.


In [6]:
# =====================================================================
# Section 9.10.5 — Branch A' checkout
# =====================================================================
from __future__ import annotations
import os
import subprocess
import sys

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PROJECT_ROOT = PROJECT_ROOT  # type: ignore[name-defined]
except NameError:
    _PROJECT_ROOT = os.path.dirname(_DAMPS_DIR)

_BRANCH = "wave2/branchA-prime-nrdmc-lite"

def _git(args, cwd=_PROJECT_ROOT):
    r = subprocess.run(
        ["git", *args], cwd=cwd, capture_output=True, text=True,
    )
    return r.returncode, (r.stdout or "") + (r.stderr or "")

# 1. Ensure we're inside a git repo.
rc, out = _git(["rev-parse", "--is-inside-work-tree"])
if rc != 0:
    raise RuntimeError(
        f"[branchA\'] {_PROJECT_ROOT} is not a git checkout.\n"
        "Re-clone with:\n"
        "  git clone https://github.com/sunflower82/"
        "DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing.git"
    )

# 2. Show current HEAD before switching.
rc, out = _git(["rev-parse", "--short", "HEAD"])
print(f"[branchA\'] pre-switch HEAD = {out.strip()}", flush=True)

# 3. Fetch remote refs.
rc, out = _git(["fetch", "origin", _BRANCH])
if rc != 0:
    print(f"[branchA\'] git fetch WARNING: {out.strip()}", flush=True)

# 4. Checkout the branch. Use --force to discard any local dirty state
#    that would block the switch (main_tercile.py is emitted fresh in §9.10).
rc, out = _git(["checkout", "-f", _BRANCH])
if rc != 0:
    # Fallback: try tracking origin/<branch>
    rc, out = _git(["checkout", "-f", "-B", _BRANCH, f"origin/{_BRANCH}"])
    if rc != 0:
        raise RuntimeError(f"[branchA\'] checkout failed:\n{out}")
print(f"[branchA\'] checkout OK -> {_BRANCH}", flush=True)

# 5. Pull latest.
rc, out = _git(["pull", "--ff-only", "origin", _BRANCH])
print(f"[branchA\'] pull: {out.strip()[:200]}", flush=True)

# 6. Verify the new files landed.
_expected = [
    "MMHCL_DAMPS_Project/damps/nrdmc_lite.py",
    "MMHCL_DAMPS_Project/model.py",
    "MMHCL_DAMPS_Project/train.py",
    "MMHCL_DAMPS_Project/utility/parser.py",
]
for _rel in _expected:
    _p = os.path.join(_PROJECT_ROOT, _rel)
    if not os.path.isfile(_p):
        raise RuntimeError(f"[branchA\'] missing after checkout: {_p}")
print("[branchA\'] all 4 patched files present.", flush=True)

# 7. Quick assertion: model.py has the enable_nrdmc_lite hook.
_model_src = open(
    os.path.join(_PROJECT_ROOT, "MMHCL_DAMPS_Project/model.py"),
    encoding="utf-8",
).read()
assert "enable_nrdmc_lite" in _model_src, (
    "[branchA\'] model.py does not expose enable_nrdmc_lite — checkout failed."
)
assert "compute_nrdmc_view_loss" in _model_src, (
    "[branchA\'] model.py does not delegate to compute_nrdmc_view_loss."
)
print("[branchA\'] model.py NRDMC-lite hook verified.", flush=True)

# 8. Show new HEAD.
rc, out = _git(["rev-parse", "--short", "HEAD"])
print(f"[branchA\'] post-switch HEAD = {out.strip()}", flush=True)
print("[branchA\'] READY. Continue to §9.11 (smoke) then §9.12 (RQ4).",
      flush=True)


[branchA'] pre-switch HEAD = f35bf2c
[branchA'] checkout OK -> wave2/branchA-prime-nrdmc-lite
[branchA'] pull: Already up to date.
From https://github.com/sunflower82/DAMPS_upgrade_for_MMHCL_randoms_Amazon_Clothing
 * branch            wave2/branchA-prime-nrdmc-lite -> FETCH_HEAD
[branchA'] all 4 patched files present.
[branchA'] model.py NRDMC-lite hook verified.
[branchA'] post-switch HEAD = f35bf2c
[branchA'] READY. Continue to §9.11 (smoke) then §9.12 (RQ4).


### 9.11 Branch A' Smoke-test — NRDMC-lite (1 seed × 10 epochs)

Same shape as §9.9 but with the NRDMC-lite view path activated:

  * `--enable_simgcl 0` (turn OFF the SimGCL noise views)
  * `--enable_nrdmc_lite 1` (turn ON learnable SAV + IAV views)
  * `--nrdmc_lite_layers 2`
  * `--lambda_view 0.2` (design-doc §8.2 middle of `λ_cl ∈ {0.1, 0.2, 0.5}`)

Everything else — dataset, seed, LogQ, DAMPS PACER trunk flags — is identical
to §9.10. `main_tercile.py` from §9.10 is reused as-is; no re-emit needed.

**Success criteria:**

  * subprocess exit code == 0
  * `[tercile-final]` line present with 3 finite floats
  * `[tercile-test-final]` line present with 3 finite floats
  * L_mv (view_loss in the training log) is > 0 and non-NaN across all 10 epochs

**Wall clock:** ~30–40 min on RTX 5090.

If PASS ⇒ proceed to §9.12 (5-seed × 250-epoch full run). If FAIL ⇒ read the
`[SMOKE-FAIL]` diagnostic before touching §9.12.


In [ ]:
# =====================================================================
# Section 9.11 — Branch A' SMOKE (1 seed × 10 epochs, NRDMC-lite)
# =====================================================================
from __future__ import annotations
import os
import re
import subprocess
import sys
import time
from typing import Any

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PROJECT_ROOT = PROJECT_ROOT  # type: ignore[name-defined]
except NameError:
    _PROJECT_ROOT = os.path.dirname(_DAMPS_DIR)
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]
except NameError:
    _cand = r"c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe"
    _PYTHON = _cand if os.path.isfile(_cand) else sys.executable
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]
except NameError:
    _WB_ENTITY = "baitapck51cc-uet"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

if not os.path.isfile(os.path.join(_DAMPS_DIR, "main_tercile.py")):
    raise RuntimeError(
        "[branchA\' smoke] main_tercile.py not found. "
        "Run §9.10 first (it emits main_tercile.py alongside train.py)."
    )

# --- Smoke-run CLI (mirror §9.11 SimGCL smoke, flip view path) ------------
SMOKE_SEED = 23946202                       # first MMHCL-paired seed
SMOKE_EPOCHS = 10
LAMBDA_VIEW_SMOKE = 0.2                     # design-doc §8.2 middle value

SMOKE_FLAGS: list[str] = [
    "--dataset",                     "Clothing",
    "--gpu_id",                      "0",
    "--epoch",                       str(SMOKE_EPOCHS),
    "--verbose",                     "5",
    "--batch_size",                  "1024",
    "--lr",                          "0.000250995",
    "--regs",                        "1.20e-04",
    "--embed_size",                  "128",
    "--topk",                        "10",
    "--core",                        "5",
    "--UI_layers",                   "3",
    "--User_layers",                 "2",
    "--Item_layers",                 "2",
    "--temperature",                 "0.251",
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_permutation_fft",       "0",
    "--damps_warmup_epochs",         "10",
    "--enable_logq",                 "1",
    "--logq_mode",                   "laplace",
    "--logq_beta",                   "1.0",
    "--logq_scale",                  "0.651",
    "--logq_clip",                   "5.0",
    # ---- Branch A' NRDMC-lite: OFF simgcl, ON nrdmc_lite -----------
    "--enable_simgcl",               "0",
    "--enable_nrdmc_lite",           "1",
    "--nrdmc_lite_layers",           "2",
    "--lambda_view",                 str(LAMBDA_VIEW_SMOKE),
    "--simgcl_eps",                  "0.329",   # unused when simgcl off
    "--branchA_view_bsz",            "2048",
    "--branchA_bcl_bsz",             "2048",
    "--branchA_bcl_batchn",          "1",
    # ---- Early stopping ---------------------------------------------
    "--early_stopping_patience",     "20",
    "--early_stopping_monitor",      "val_recall@20",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    # ---- ReduceLROnPlateau + AMP + WandB ----------------------------
    "--use_reduce_lr",               "1",
    "--use_amp",                     "1",
    "--use_wandb",                   "1",
    "--wandb_project",               _WB_PROJECT,
    "--wandb_entity",                _WB_ENTITY,
    "--wandb_run_name",              f"branchA_prime_smoke_seed{SMOKE_SEED}",
    "--seed",                        str(SMOKE_SEED),
]

cmd = [_PYTHON, "main_tercile.py", *SMOKE_FLAGS]
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["PYTHONUNBUFFERED"] = "1"
print("[smoke] " + " ".join(cmd[:12]) + " ...", flush=True)
print(f"[smoke] full flag count = {len(SMOKE_FLAGS)}", flush=True)
_t0 = time.time()
proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", bufsize=1,
)
chunks: list[str] = []
assert proc.stdout is not None
for line in proc.stdout:
    chunks.append(line)
    print(line, end="", flush=True)
rc = proc.wait()
out = "".join(chunks)
_wall = time.time() - _t0
print(f"\n[smoke] exit={rc}  wall={_wall/60:.1f} min", flush=True)

# --- Parse [tercile-*-final] lines ---------------------------------------
_TER_RX = re.compile(
    r"\[tercile-final\]\s+BEST_Recall@20_Head=(?P<h>[-\d.eE+nan]+)"
    r"\s+BEST_Recall@20_Mid=(?P<m>[-\d.eE+nan]+)"
    r"\s+BEST_Recall@20_Tail=(?P<t>[-\d.eE+nan]+)"
)
_TER_TEST_RX = re.compile(
    r"\[tercile-test-final\]\s+BEST_Test_Recall@20_Head=(?P<h>[-\d.eE+nan]+)"
    r"\s+BEST_Test_Recall@20_Mid=(?P<m>[-\d.eE+nan]+)"
    r"\s+BEST_Test_Recall@20_Tail=(?P<t>[-\d.eE+nan]+)"
)
# \b avoids matching lambda_view= in the Namespace dump.
_VIEW_RX = re.compile(r"\bview=([-\d.eE+nan]+)")

def _is_finite(x: str) -> bool:
    try:
        v = float(x)
        return v == v and v != float("inf") and v != float("-inf")
    except ValueError:
        return False

m_val = _TER_RX.search(out)
m_tst = _TER_TEST_RX.search(out)
view_vals = [float(v) for v in _VIEW_RX.findall(out) if _is_finite(v)]

fails: list[str] = []
if rc != 0:
    fails.append(f"exit code = {rc}")
if m_val is None:
    fails.append("[tercile-final] line MISSING")
elif not all(_is_finite(m_val.group(k)) for k in ("h", "m", "t")):
    fails.append(f"[tercile-final] has NaN: {m_val.groupdict()}")
if m_tst is None:
    fails.append("[tercile-test-final] line MISSING")
elif not all(_is_finite(m_tst.group(k)) for k in ("h", "m", "t")):
    fails.append(f"[tercile-test-final] has NaN: {m_tst.groupdict()}")
if not view_vals:
    fails.append("no view= entries in training log (view path silent?)")
elif any(v <= 0 for v in view_vals):
    fails.append(f"view_loss non-positive somewhere: {view_vals}")

print("\n" + "=" * 74, flush=True)
if fails:
    print("[SMOKE-FAIL] Branch A' NRDMC-lite smoke FAILED:", flush=True)
    for f in fails:
        print("  - " + f, flush=True)
    print("=" * 74, flush=True)
    raise AssertionError("Branch A' smoke test failed; see [SMOKE-FAIL] above.")
else:
    print("[SMOKE-PASS] Branch A' NRDMC-lite smoke PASSED.", flush=True)
    print(f"   VAL  Head/Mid/Tail = {m_val.group('h')} / {m_val.group('m')} / {m_val.group('t')}", flush=True)
    print(f"   TEST Head/Mid/Tail = {m_tst.group('h')} / {m_tst.group('m')} / {m_tst.group('t')}", flush=True)
    print(f"   view_loss range    = [{min(view_vals):.4g}, {max(view_vals):.4g}]  across {len(view_vals)} epochs", flush=True)
    print(f"   wall time (10 epochs) = {_wall/60:.1f} min", flush=True)
    print("=" * 74, flush=True)
    print("PROCEED TO §9.12 (5-seed × 250-epoch RQ4 run).", flush=True)


### 9.12 RQ4 (a) — Branch A' NRDMC-lite (5 seeds × 250 epochs)

Full run: same 5 MMHCL-paired seeds, same 250-epoch budget, same t0030 config
as §9.10 — **only the view-loss path differs**:

  * `--enable_simgcl 0` → SimGCL noise views OFF
  * `--enable_nrdmc_lite 1` → SAV + IAV + adaptive fusion ON
  * `--lambda_view 0.2` → NRDMC-lite `λ_cl` per §8.2 middle value

Output: `results/rq4_tercile_pacer_branchA_prime_clothing_5seed.json` (per-seed
+ aggregate for BOTH val and test splits, mirrors §9.10 output shape).

**Design-doc §8.2 target:** Recall@20 ∈ [0.0935, 0.0961] (i.e. beat MMHCL
baseline R@20=0.087 AND beat vanilla PACER R@20≈0.075).

**Wall clock:** ~10–12h on RTX 5090 (2h/seed × 5 seeds). Extra NRDMC-lite
overhead per epoch ≲ 3 %.

**If NRDMC-lite BEATS MMHCL:** ship Branch A' — the paper argument becomes:
"replacing SimGCL's noise views with learnable structure/importance views
recovers the +Δ Recall@20 that the surgical fix alone could not."

**If NRDMC-lite is on par with MMHCL (± 0.5 %):** ship Branch A' anyway —
the paper argument becomes: "PACER + NRDMC-lite matches MMHCL overall while
preserving the tail-Recall@20 gain the reviewer cited as the model's strength."

**If NRDMC-lite is worse than MMHCL by > 1 %:** revert to the SimGCL branch A
result from §9.10 and consider a lambda_view sweep (§9.13, TODO).


In [7]:
# =====================================================================
# Section 9.12 — RQ4 (a) : PACER + Branch A' (NRDMC-lite)
#                          on Amazon Clothing with Head/Mid/Tail terciles
#                          (5 seeds × 250 epochs)
# =====================================================================
from __future__ import annotations

import json
import math
import os
import re
import statistics
import subprocess
import sys
import time
from typing import Any

# ---- 0. Resolve dirs / globals (reuse from earlier cells) ----------------
try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PROJECT_ROOT = PROJECT_ROOT  # type: ignore[name-defined]
except NameError:
    _PROJECT_ROOT = os.path.dirname(_DAMPS_DIR)
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]
except NameError:
    _cand = r"c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe"
    _PYTHON = _cand if os.path.isfile(_cand) else sys.executable
try:
    _WB_PROJECT = wandb_project  # type: ignore[name-defined]
except NameError:
    _WB_PROJECT = "damps-mmhcl-clothing"
try:
    _WB_ENTITY = wandb_entity  # type: ignore[name-defined]
except NameError:
    _WB_ENTITY = "baitapck51cc-uet"

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)
os.makedirs("results", exist_ok=True)

if not os.path.isfile(os.path.join(_DAMPS_DIR, "main_tercile.py")):
    raise RuntimeError(
        "[branchA\'] main_tercile.py not found. Run §9.10 (it emits the "
        "PACER-aware wrapper) OR §9.9 (which also emits it) before §9.12."
    )

# ---- 1. LOCKED config (mirrors §9.10 t0030) + Branch A' overlay ----------
DATASET = "Clothing"
EPOCH = 250
PATIENCE = 20
BATCH_SIZE = 1024
LR = 2.50995e-4
REGS = 1.20e-04
EMBED_SIZE = 128
KNN_TOPK = 10
UI_LAYERS = 3
U_LAYERS = 2
I_LAYERS = 2
TEMPERATURE = 0.251
SIMGCL_EPS = 0.329          # unused (simgcl off), kept for CLI compat
LAMBDA_VIEW = 0.2           # Branch A' NRDMC-lite λ_cl (§8.2 middle value)
LOGQ_SCALE = 0.651
LOGQ_BETA = 1.0
LOGQ_CLIP = 5.0
USE_REDUCE_LR = 1

# MMHCL-paired seeds -- LOCKED (matches §9.10 exactly)
SEEDS = [23946202, 1557638902, 969448664, 913925833, 85755121]

BASE_FLAGS: list[str] = [
    "--dataset",                     DATASET,
    "--gpu_id",                      "0",
    "--epoch",                       str(EPOCH),
    "--verbose",                     "5",
    "--batch_size",                  str(BATCH_SIZE),
    "--lr",                          str(LR),
    "--regs",                        str(REGS),
    "--embed_size",                  str(EMBED_SIZE),
    "--topk",                        str(KNN_TOPK),
    "--core",                        "5",
    "--UI_layers",                   str(UI_LAYERS),
    "--User_layers",                 str(U_LAYERS),
    "--Item_layers",                 str(I_LAYERS),
    "--temperature",                 str(TEMPERATURE),
    # ---- PACER: APC off + IMCF + soft routing + momentum + DDP -------
    "--damps_apc",                   "0",
    "--damps_avrf",                  "0",
    "--damps_imcf",                  "1",
    "--damps_soft_routing",          "1",
    "--damps_momentum",              "1",
    "--damps_data_driven_prior",     "1",
    "--damps_permutation_fft",       "0",
    "--damps_warmup_epochs",         "10",
    # ---- PACER: LogQ correction --------------------------------------
    "--enable_logq",                 "1",
    "--logq_mode",                   "laplace",
    "--logq_beta",                   str(LOGQ_BETA),
    "--logq_scale",                  str(LOGQ_SCALE),
    "--logq_clip",                   str(LOGQ_CLIP),
    # ---- BRANCH A' NRDMC-lite  (this is the CHANGE from §9.10) ------
    "--enable_simgcl",               "0",
    "--enable_nrdmc_lite",           "1",
    "--nrdmc_lite_layers",           "2",
    "--lambda_view",                 str(LAMBDA_VIEW),
    "--simgcl_eps",                  str(SIMGCL_EPS),        # unused
    "--branchA_view_bsz",            "2048",
    "--branchA_bcl_bsz",             "2048",
    "--branchA_bcl_batchn",          "1",
    # ---- Early stopping ---------------------------------------------
    "--early_stopping_patience",     str(PATIENCE),
    "--early_stopping_min_epochs",   "0",
    "--early_stopping_min_delta",    "1e-4",
    "--early_stopping_monitor",      "val_recall@20",
    "--early_stopping_mode",         "max",
    "--early_stopping_restore_best", "1",
    # ---- ReduceLROnPlateau + AMP + WandB ----------------------------
    "--use_reduce_lr",               str(USE_REDUCE_LR),
    "--use_amp",                     "1",
    "--use_wandb",                   "1",
    "--wandb_project",               _WB_PROJECT,
    "--wandb_entity",                _WB_ENTITY,
]

# ---- 2. Regexes for parsing the summary lines ----------------------------
_TER_RX = re.compile(
    r"\[tercile-final\]\s+BEST_Recall@20_Head=(?P<h>[-\d.eE+nan]+)"
    r"\s+BEST_Recall@20_Mid=(?P<m>[-\d.eE+nan]+)"
    r"\s+BEST_Recall@20_Tail=(?P<t>[-\d.eE+nan]+)"
)
_TER_TEST_RX = re.compile(
    r"\[tercile-test-final\]\s+BEST_Test_Recall@20_Head=(?P<h>[-\d.eE+nan]+)"
    r"\s+BEST_Test_Recall@20_Mid=(?P<m>[-\d.eE+nan]+)"
    r"\s+BEST_Test_Recall@20_Tail=(?P<t>[-\d.eE+nan]+)"
)
_BEST_RX = re.compile(r"BEST_Test_Recall@20\s*=\s*([-\d.eE+nan]+)")

def _f(x: str) -> float:
    try: return float(x)
    except ValueError: return float("nan")

# ---- 3. Per-seed subprocess loop -----------------------------------------
results: list[dict[str, Any]] = []
for run_idx, seed in enumerate(SEEDS, 1):
    print(f"\n{'='*74}\nRUN {run_idx}/{len(SEEDS)}  seed={seed}"
          f"  [Branch A' NRDMC-lite]\n{'='*74}", flush=True)
    run_name = f"pacer_branchA_prime_seed_{seed}"
    cmd = [_PYTHON, "main_tercile.py", *BASE_FLAGS,
           "--seed", str(seed),
           "--wandb_run_name", run_name]
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"
    print("[cmd] " + " ".join(cmd[:8]) + " ... [" + str(len(cmd) - 8) + " more flags]",
          flush=True)
    _t0 = time.time()
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    chunks: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        chunks.append(line)
        print(line, end="", flush=True)
    rc = proc.wait()
    out = "".join(chunks)
    _wall = time.time() - _t0

    if rc != 0:
        print(f"[WARN] seed={seed} exited {rc}", flush=True)

    m  = _TER_RX.search(out)
    mt = _TER_TEST_RX.search(out)
    b  = _BEST_RX.search(out)

    row: dict[str, Any] = {
        "seed": seed,
        "wall_min": _wall / 60.0,
        "exit": rc,
        "val_head":  _f(m.group("h"))  if m  else float("nan"),
        "val_mid":   _f(m.group("m"))  if m  else float("nan"),
        "val_tail":  _f(m.group("t"))  if m  else float("nan"),
        "test_head": _f(mt.group("h")) if mt else float("nan"),
        "test_mid":  _f(mt.group("m")) if mt else float("nan"),
        "test_tail": _f(mt.group("t")) if mt else float("nan"),
        "best_test_recall20": _f(b.group(1)) if b else float("nan"),
    }
    results.append(row)
    print(f"[branchA\'] seed={seed}  wall={_wall/60:.1f}m  "
          f"test H/M/T = {row['test_head']:.5f}/{row['test_mid']:.5f}/{row['test_tail']:.5f}  "
          f"overall R@20 = {row['best_test_recall20']:.5f}", flush=True)

# ---- 4. Aggregate + write results ---------------------------------------
def _agg(vals: list[float]) -> dict[str, float]:
    v = [x for x in vals if not math.isnan(x)]
    if not v:
        return {"mean": float("nan"), "std": float("nan"), "n": 0}
    return {
        "mean": float(statistics.mean(v)),
        "std":  float(statistics.stdev(v)) if len(v) > 1 else 0.0,
        "n":    len(v),
    }

agg = {
    "val_head":            _agg([r["val_head"]  for r in results]),
    "val_mid":             _agg([r["val_mid"]   for r in results]),
    "val_tail":            _agg([r["val_tail"]  for r in results]),
    "test_head":           _agg([r["test_head"] for r in results]),
    "test_mid":            _agg([r["test_mid"]  for r in results]),
    "test_tail":           _agg([r["test_tail"] for r in results]),
    "best_test_recall20":  _agg([r["best_test_recall20"] for r in results]),
}

out_path = "results/rq4_tercile_pacer_branchA_prime_clothing_5seed.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({"variant": "PACER+BranchA_prime_NRDMC_lite",
               "seeds": SEEDS,
               "config": {"lambda_view": LAMBDA_VIEW,
                          "nrdmc_lite_layers": 2,
                          "epoch": EPOCH,
                          "batch_size": BATCH_SIZE,
                          "lr": LR,
                          "temperature": TEMPERATURE,
                          "logq_scale": LOGQ_SCALE,
                          "regs": REGS,
                          "embed_size": EMBED_SIZE},
               "per_seed": results,
               "aggregate": agg}, f, indent=2)
print(f"\n[branchA\'] Wrote {out_path}", flush=True)

print("\n" + "=" * 74, flush=True)
print("BRANCH A\' NRDMC-LITE FINAL — Amazon Clothing (39k) — 5 seeds", flush=True)
print("=" * 74, flush=True)
print(f"  Overall Test  R@20 : {agg['best_test_recall20']['mean']:.5f} "
      f"± {agg['best_test_recall20']['std']:.5f}  (n={agg['best_test_recall20']['n']})", flush=True)
print(f"  Test  Head  R@20   : {agg['test_head']['mean']:.5f} "
      f"± {agg['test_head']['std']:.5f}", flush=True)
print(f"  Test  Mid   R@20   : {agg['test_mid']['mean']:.5f} "
      f"± {agg['test_mid']['std']:.5f}", flush=True)
print(f"  Test  Tail  R@20   : {agg['test_tail']['mean']:.5f} "
      f"± {agg['test_tail']['std']:.5f}", flush=True)
print("=" * 74, flush=True)
print("Compare vs:", flush=True)
print("  MMHCL  baseline   R@20 = 0.087   Head=0.136  Mid=0.022  Tail=0.009", flush=True)
print("  PACER  vanilla    R@20 ≈ 0.075   Head≈0.116  Mid≈0.008  Tail≈0.003", flush=True)
print("  §8.2 target       R@20 ∈ [0.0935, 0.0961]", flush=True)
print("=" * 74, flush=True)



RUN 1/5  seed=23946202  [Branch A' NRDMC-lite]
[cmd] c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe main_tercile.py --dataset Clothing --gpu_id 0 --epoch 250 ... [90 more flags]
n_users=39387, n_items=23033
n_interactions=237488
n_train=197338, n_test=40150, sparsity=0.00026
[tercile] n_items=23033  |Tail|=7677  |Mid|=7678  |Head|=7678  tail-freq<=5  head-freq>=7
[2026-07-20 18:20:44] PID: 16592
[2026-07-20 18:20:44] Namespace(data_path='../data/', seed=23946202, dataset='Clothing', core=5, gpu_id=0, debug='True', verbose=5, epoch=250, batch_size=1024, regs=0.00012, lr=0.000250995, clip_grad_norm=1.0, embed_size=128, weight_size='[64,64,64]', topk=10, cf_model='LightGCN', norm_type='sym', sparse=0, UI_layers=3, User_layers=2, Item_layers=2, user_loss_ratio=0.03, item_loss_ratio=0.07, temperature=0.251, learnable_tau=0, Ks='[10,20]', test_flag='part', early_stopping_patience=20, early_stopping_min_epochs=0, early_stopping_min_delta=0.0001, early_stopping_mode='max', early_stopping

### 9.13 P1+P2 joint probe — $\lambda_{view}\times\tau$ (4 configs × 2 seeds × 100 epochs)

Implements the upgrade-doc roadmap (`PACER_NRDMC_lite_upgrade_analysis_EN`):

| Priority | Action | Grid |
|---|---|---|
| **P1** | Lower $\lambda_{view}$ (0.2 was too strong → Head decay) | $\{0.05, 0.10\}$ |
| **P2** | Raise $\tau$ (soften InfoNCE gradients) | $\{0.30, 0.40\}$ |

**Joint 2×2** (4 configs) × **2 MMHCL-paired seeds** (`23946202`, `1557638902`) × **100 epochs**.

Tracked driver (also runnable from shell):

```bash
cd MMHCL_DAMPS_Project
python scripts/run_p1_p2_nrdmc_grid.py
```

Outputs `results/p1_p2_lambda_tau_grid_clothing.json` and prints a ranked table by mean Test Recall@20. Lock the best $(\lambda_{view},\tau)$ before P3/P4.


In [7]:
# =====================================================================
# Section 9.13 — P1+P2 joint lambda_view x tau probe (Branch A' NRDMC-lite)
#   4 configs × 2 seeds × 100 epochs
#   Driver: scripts/run_p1_p2_nrdmc_grid.py
# =====================================================================
from __future__ import annotations

import os
import sys

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]
except NameError:
    _cand = r"c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe"
    _PYTHON = _cand if os.path.isfile(_cand) else sys.executable

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

_script = os.path.join("scripts", "run_p1_p2_nrdmc_grid.py")
if not os.path.isfile(_script):
    raise FileNotFoundError(
        f"[P1+P2] missing {_script}. Pull wave2/branchA-prime-nrdmc-lite "
        "or ensure the driver was committed under MMHCL_DAMPS_Project/scripts/."
    )

# Optional notebook overrides (defaults match the upgrade-doc grid).
# Set DRY_RUN=1 to print commands without training.
DRY_RUN = int(globals().get("P1P2_DRY_RUN", 0))

print(
    "[P1+P2] launching scripts/run_p1_p2_nrdmc_grid.py\n"
    "  grid: lambda_view in {0.05, 0.10}  x  tau in {0.30, 0.40}\n"
    "  seeds: 2   epochs: 100   (≈ 4 configs × 2 seeds)",
    flush=True,
)

# Re-exec the driver in-process so notebook sees live stdout + final table.
sys.argv = [
    _script,
    "--dry_run",
    str(DRY_RUN),
]
with open(_script, encoding="utf-8") as _f:
    _src = _f.read()
_ns: dict = {"__name__": "__main__", "__file__": os.path.abspath(_script)}
exec(compile(_src, _script, "exec"), _ns)


[P1+P2] launching scripts/run_p1_p2_nrdmc_grid.py
  grid: lambda_view in {0.05, 0.10}  x  tau in {0.30, 0.40}
  seeds: 2   epochs: 100   (≈ 4 configs × 2 seeds)
[P1+P2] 4 configs x 2 seeds x 100 epochs  dry_run=0
   - lam0p05_tau0p30: lambda_view=0.05  tau=0.3
   - lam0p05_tau0p40: lambda_view=0.05  tau=0.4
   - lam0p10_tau0p30: lambda_view=0.1  tau=0.3
   - lam0p10_tau0p40: lambda_view=0.1  tau=0.4

[P1+P2] progress 1/8

[P1+P2] cfg=lam0p05_tau0p30  lambda_view=0.05  tau=0.3  seed=23946202
[cmd] c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe main_tercile.py --dataset Clothing --gpu_id 0 --epoch 100 ... [94 more flags]
n_users=39387, n_items=23033
n_interactions=237488
n_train=197338, n_test=40150, sparsity=0.00026
[tercile] n_items=23033  |Tail|=7677  |Mid|=7678  |Head|=7678  tail-freq<=5  head-freq>=7
[2026-07-21 17:11:14] PID: 11224
[2026-07-21 17:11:14] Namespace(data_path='../data/', seed=23946202, dataset='Clothing', core=5, gpu_id=0, debug='True', verbose=5, epoch=100, batc

SystemExit: 0

c:\ProgramData\anaconda3\envs\rtx5090_dl\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### 9.14 P3 PTV grid — re-instate Prototype-Aware View (K=3 fusion)

Implements the **rev56** roadmap (D4 = missing PTV):

$$w_{final} = w_{sav} + w_{iav} + \lambda_{ptv}\,w_{ptv} \;-\; (1+\lambda_{ptv})\,w_{shared}$$

| Cell | K | λ_ptv | Note |
|---|---|---|---|
| `ctrl_k2`      | –  | 0.0 | K=2 control (`enable_ptv=0`) — sanity check that new code is bit-identical to Section 9.13 winner |
| `ptv_k32_l0p5` | 32 | 0.5 | Weak PTV |
| `ptv_k32_l1p0` | 32 | 1.0 | Paper default |
| `ptv_k32_l2p0` | 32 | 2.0 | Strong PTV |
| `ptv_k16_l1p0` | 16 | 1.0 | Fewer prototypes |

To run only a subset, edit the `CELLS` dict at the top of `scripts/run_p3_ptv_grid.py` before executing this cell.

**Base HP locked** to P1+P2 winner: `λ_view=0.10, τ=0.30`. Seeds: `[23946202, 1557638902]`. Epochs: 100. Total: 5 cells × 2 seeds ≈ 12 h on A100.

**Prerequisite:** the notebook must checkout commit `dcc774e` (or newer) on branch `wave2/branchA-prime-nrdmc-lite`. The cell below performs an explicit `git pull` before launching the driver. If the local repo is dirty, resolve conflicts manually and rerun.

In [ ]:
# =====================================================================
# Section 9.14 — P3 PTV grid (Branch A' NRDMC-lite, K=3 fusion)
#   5 configs × 2 seeds × 100 epochs
#   Driver:  scripts/run_p3_ptv_grid.py         (commit dcc774e)
#   Branch:  wave2/branchA-prime-nrdmc-lite
# =====================================================================
from __future__ import annotations

import os
import subprocess
import sys

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]
except NameError:
    _cand = r"c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe"
    _PYTHON = _cand if os.path.isfile(_cand) else sys.executable

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---------------------------------------------------------------------
# 1)  Ensure repo is at wave2/branchA-prime-nrdmc-lite with PTV code.
# ---------------------------------------------------------------------
_SKIP_PULL = int(globals().get("P3_SKIP_PULL", 0))
if not _SKIP_PULL:
    try:
        _branch = subprocess.check_output(
            ["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True
        ).strip()
        print(f"[P3] git branch = {_branch}", flush=True)
        if _branch != "wave2/branchA-prime-nrdmc-lite":
            print(
                "[P3] WARNING: not on wave2/branchA-prime-nrdmc-lite. "
                "Attempting checkout...",
                flush=True,
            )
            subprocess.check_call(
                ["git", "fetch", "origin", "wave2/branchA-prime-nrdmc-lite"]
            )
            subprocess.check_call(
                ["git", "checkout", "wave2/branchA-prime-nrdmc-lite"]
            )
        print("[P3] git pull --ff-only ...", flush=True)
        subprocess.check_call(["git", "pull", "--ff-only"])
        _head = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], text=True
        ).strip()
        print(f"[P3] HEAD = {_head}  (expected: dcc774e or newer)", flush=True)
    except (subprocess.CalledProcessError, FileNotFoundError) as _e:
        print(
            f"[P3] git-sync skipped ({_e!r}); "
            "set P3_SKIP_PULL=1 if this is intentional.",
            flush=True,
        )

# ---------------------------------------------------------------------
# 2)  Locate the P3 driver.
# ---------------------------------------------------------------------
_script = os.path.join("scripts", "run_p3_ptv_grid.py")
if not os.path.isfile(_script):
    raise FileNotFoundError(
        f"[P3] missing {_script}. Pull wave2/branchA-prime-nrdmc-lite "
        "(commit dcc774e or newer) or ensure the driver was committed under "
        "MMHCL_DAMPS_Project/scripts/."
    )

# ---------------------------------------------------------------------
# 3)  Optional notebook overrides.
#       Set  P3_DRY_RUN=1  to print planned runs without training.
# ---------------------------------------------------------------------
DRY_RUN = int(globals().get("P3_DRY_RUN", 0))

print(
    "[P3] launching scripts/run_p3_ptv_grid.py\n"
    "  base HP:  lambda_view=0.10, tau=0.30  (P1+P2 winner)\n"
    "  grid:     ctrl_k2, ptv_k32_l0p5, ptv_k32_l1p0, ptv_k32_l2p0, ptv_k16_l1p0\n"
    "  seeds: 2   epochs: 100   (~ 5 cells x 2 seeds ~ 12 h on A100)",
    flush=True,
)

sys.argv = [_script, "--dry_run", str(DRY_RUN)]
with open(_script, encoding="utf-8") as _f:
    _src = _f.read()
_ns: dict = {"__name__": "__main__", "__file__": os.path.abspath(_script)}
exec(compile(_src, _script, "exec"), _ns)


[P3] git branch = wave2/branchA-prime-nrdmc-lite
[P3] git pull --ff-only ...
[P3] HEAD = dcc774e  (expected: dcc774e or newer)
[P3] launching scripts/run_p3_ptv_grid.py
  base HP:  lambda_view=0.10, tau=0.30  (P1+P2 winner)
  grid:     ctrl_k2, ptv_k32_l0p5, ptv_k32_l1p0, ptv_k32_l2p0, ptv_k16_l1p0
  seeds: 2   epochs: 100   (~ 5 cells x 2 seeds ~ 12 h on A100)
[P3] 5 configs x 2 seeds x 100 epochs  dry_run=0
[P3] base HP (locked): lambda_view=0.1  tau=0.3
   - ctrl_k2: enable_ptv=0  K=0  lambda_ptv=0.0
   - ptv_k32_l0p5: enable_ptv=1  K=32  lambda_ptv=0.5
   - ptv_k32_l1p0: enable_ptv=1  K=32  lambda_ptv=1.0
   - ptv_k32_l2p0: enable_ptv=1  K=32  lambda_ptv=2.0
   - ptv_k16_l1p0: enable_ptv=1  K=16  lambda_ptv=1.0

[P3] progress 1/10

[P3] cfg=ctrl_k2  enable_ptv=0  K=0  lambda_ptv=0.0  seed=23946202
[cmd] c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe main_tercile.py --dataset Clothing --gpu_id 0 --epoch 100 ... [100 more flags]
n_users=39387, n_items=23033
n_interactions=237488

### 9.15 P4 ASC fix grid — reparameterize $\alpha$ to prevent gate collapse

Implements the **rev57** roadmap (Section 6.5 of `PACER_NRDMC_lite_upgrade_analysis_EN`):

* P3 PTV grid returned a null result (top cell +0.66% R@20, within control noise ±0.62%).
* The training log revealed the real bug: `alpha_img` drifts from +0.09 (ep 0) to −0.68 (ep 75), meaning the Soft-Routing branch actively *subtracts* the calibrated image signal by the val-peak epoch (≈49).
* P4 fixes this by reparameterizing `alpha_v` through four complementary knobs — `--asc_gate_mode`, `--asc_warmup_epochs`, `--asc_reg_l2`, `--asc_reg_target` — keeping PTV OFF so the gate signal is isolated.

Grid: 6 cells × 2 seeds × 60 epochs (val-recall peak in P3 was epoch 49). Budget ≈ 3.3 h A100.

| cell | `asc_gate_mode` | warmup | `reg_l2` | note |
|---|---|---|---|---|
| `baseline_raw` | raw | 0 | 0.00 | reproduces P3 `ctrl_k2` |
| `sigmoid` | sigmoid | 0 | 0.00 | $\alpha \in (0,1)$ |
| `tanh_signed` | tanh_signed | 0 | 0.00 | $\alpha \in (-1,1)$, capped |
| `reg_l0p01_t0p3` | raw | 0 | 0.01 | pull-to-target 0.3 |
| `warmup_e20` | raw | 20 | 0.00 | freeze $\alpha$ 20 ep |
| `sigmoid_reg` | sigmoid | 0 | 0.01 | belt-and-suspenders |

Success criterion: any P4 cell reaches **R@20 ≥ 0.083** (> 0.08122 + 1$\sigma$) *and* `alpha_img_final` stays $\ge -0.1$. If every P4 cell lands within ±$\sigma$ of `baseline_raw`, we retract diagnostic P3-D1 and move on to P5 (dynamic $\tau$ + CL re-weighting).


In [ ]:
# =====================================================================
# Section 9.15 — P4 ASC-gate reparameterization grid
#   6 configs × 2 seeds × 60 epochs
#   Driver:  scripts/run_p4_asc_fix_grid.py
#   Branch:  wave2/branchA-prime-nrdmc-lite  (rev57)
# =====================================================================
from __future__ import annotations

import os
import subprocess
import sys

try:
    _DAMPS_DIR = DAMPS_DIR  # type: ignore[name-defined]
except NameError:
    _cwd = os.getcwd()
    _DAMPS_DIR = (
        _cwd if os.path.basename(_cwd) == "MMHCL_DAMPS_Project"
        else os.path.join(_cwd, "MMHCL_DAMPS_Project")
    )
try:
    _PYTHON = PYTHON_EXE  # type: ignore[name-defined]
except NameError:
    _cand = r"c:\ProgramData\anaconda3\envs\rtx5090_dl\python.exe"
    _PYTHON = _cand if os.path.isfile(_cand) else sys.executable

if os.path.normpath(os.getcwd()) != os.path.normpath(_DAMPS_DIR):
    os.chdir(_DAMPS_DIR)

# ---------------------------------------------------------------------
# 1)  Ensure repo is at wave2/branchA-prime-nrdmc-lite (rev57 or newer).
# ---------------------------------------------------------------------
_SKIP_PULL = int(globals().get("P4_SKIP_PULL", 0))
if not _SKIP_PULL:
    try:
        _branch = subprocess.check_output(
            ["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True
        ).strip()
        print(f"[P4] git branch = {_branch}", flush=True)
        if _branch != "wave2/branchA-prime-nrdmc-lite":
            print(
                "[P4] WARNING: not on wave2/branchA-prime-nrdmc-lite. "
                "Attempting checkout...",
                flush=True,
            )
            subprocess.check_call(
                ["git", "fetch", "origin", "wave2/branchA-prime-nrdmc-lite"]
            )
            subprocess.check_call(
                ["git", "checkout", "wave2/branchA-prime-nrdmc-lite"]
            )
        print("[P4] git pull --ff-only ...", flush=True)
        subprocess.check_call(["git", "pull", "--ff-only"])
        _head = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], text=True
        ).strip()
        print(f"[P4] HEAD = {_head}  (expected: rev57 or newer)",
              flush=True)
    except (subprocess.CalledProcessError, FileNotFoundError) as _e:
        print(
            f"[P4] git-sync skipped ({_e!r}); "
            "set P4_SKIP_PULL=1 if this is intentional.",
            flush=True,
        )

# ---------------------------------------------------------------------
# 2)  Locate the P4 driver.
# ---------------------------------------------------------------------
_script = os.path.join("scripts", "run_p4_asc_fix_grid.py")
if not os.path.isfile(_script):
    raise FileNotFoundError(
        f"[P4] missing {_script}. Pull wave2/branchA-prime-nrdmc-lite "
        "(rev57 or newer) to fetch the P4 driver."
    )

# ---------------------------------------------------------------------
# 3)  Optional notebook overrides.
#       Set  P4_DRY_RUN=1  to print planned runs without training.
# ---------------------------------------------------------------------
DRY_RUN = int(globals().get("P4_DRY_RUN", 0))

print(
    "[P4] launching scripts/run_p4_asc_fix_grid.py\n"
    "  base HP:  lambda_view=0.10, tau=0.30, enable_ptv=0 (P3 null closed)\n"
    "  grid:     baseline_raw, sigmoid, tanh_signed, reg_l0p01_t0p3, "
              "warmup_e20, sigmoid_reg\n"
    "  seeds: 2   epochs: 60   (~ 6 cells x 2 seeds ~ 3.3 h on A100)",
    flush=True,
)

sys.argv = [_script, "--dry_run", str(DRY_RUN)]
with open(_script, encoding="utf-8") as _f:
    _src = _f.read()
_ns: dict = {"__name__": "__main__", "__file__": os.path.abspath(_script)}
exec(compile(_src, _script, "exec"), _ns)
